https://tonikph-my.sharepoint.com/:p:/g/personal/bbanik_tonikbank_com/IQAvsct_VJYKRoYRQISh8jy1AWHNdHcuDqk9bIfwQjpiW6M?e=wvYx82

# Import Packages

In [1]:
# %% [markdown]
# # Jupyter Notebook Loading Header
#
# This is a custom loading header for Jupyter Notebooks in Visual Studio Code.
# It includes common imports and settings to get you started quickly.
# %% [markdown]
## Import Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from google.cloud import bigquery
from google.cloud import storage
import os
import tempfile
import time
from datetime import datetime
import uuid
import joblib
import uuid
from sklearn.metrics import roc_auc_score
from datetime import datetime, timedelta, date
import gcsfs
import duckdb as dd
import pickle
import joblib
from typing import Union
import io
import re
path = r'C:\Users\Dwaipayan\AppData\Roaming\gcloud\legacy_credentials\dchakroborti@tonikbank.com\adc.json'
os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = path
client = bigquery.Client(project='prj-prod-dataplatform')
os.environ["GOOGLE_CLOUD_PROJECT"] = "prj-prod-dataplatform"
# %% [markdown]
## Configure Settings
# Set options or configurations as needed
pd.set_option('display.max_columns', None)
pd.set_option("Display.max_rows", 100)
sns.set_style('whitegrid')

# CONFIGURATION

In [2]:
# GCP credentials — use ADC JSON or set GOOGLE_APPLICATION_CREDENTIALS env var
CREDENTIALS_PATH = r"C:\Users\Dwaipayan\AppData\Roaming\gcloud\legacy_credentials\dchakroborti@tonikbank.com\adc.json"
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = CREDENTIALS_PATH

os.environ["GOOGLE_CLOUD_PROJECT"] = "prj-prod-dataplatform"

PROJECT_ID = "prj-prod-dataplatform"
GS_BUCKET = "prod-asia-southeast1-tonik-aiml-workspace"

CLOUDPATH = "DC/Tendo_Model_Monitoring_Data"
CURRENT_DATE = datetime.now().strftime("%Y%m%d")

# Output folder — Power BI reads CSVs from here
OUTPUT_DIR = r"D:\OneDrive - Tonik Financial Pte Ltd\MyStuff\Data Engineering\Tendo_Model_Monitoring_Dashboards_Requirement\Tendo_Monitoring_data"


# Functions

In [3]:
def define_cat_features(columns, cat_vars):
    return list(set(cat_vars).intersection(columns))

def generate_bucket_url(filename, bucket_name):

    return f"gs://{bucket_name}/{filename}"


def save_to_gcs(data, filename, bucket_name):
    """
    Save data to Google Cloud Storage bucket.

    :param data: The data to save. Can be a string, bytes, or a file-like object.
    :param filename: The name of the file to save in the bucket.
    :param bucket_name: The name of the GCS bucket. Defaults to 'ABC'.
    :return: The public URL of the saved file.
    """
    # Create a client
    client = storage.Client()

    # Get the bucket
    bucket = client.bucket(bucket_name)

    # Create a blob (file) in the bucket
    blob = bucket.blob(filename)

    # Determine the content type and upload accordingly
    if isinstance(data, str):
        blob.upload_from_string(data)
    elif isinstance(data, bytes):
        blob.upload_from_string(data, content_type='application/octet-stream')
    elif hasattr(data, 'read'):  # File-like object
        blob.upload_from_file(data)
    else:
        raise ValueError("Unsupported data type. Please provide a string, bytes, or file-like object.")


    print(f"File {filename} uploaded to {bucket_name}.")
    return blob.public_url


def load_artifact_from_gcs(filename, bucket_name):
    client = storage.Client()
    bucket = client.bucket(bucket_name)
    blob = bucket.blob(filename)

    # Download model
    artifact_bytes = blob.download_as_bytes()
    artifact = pickle.loads(artifact_bytes)
    print(f"Model loaded from gs://{bucket_name}/{filename}")

    return artifact


def load_from_gcs(filename, bucket_name, output_type='bytes'):
    """
    Load data from Google Cloud Storage bucket with flexible output formats.

    :param filename: The name of the file to load from the bucket.
    :param bucket_name: The name of the GCS bucket.
    :param output_type: The desired output format. Can be 'bytes', 'string', 'pickle', or 'file'.
                       'bytes': Returns raw bytes
                       'string': Returns decoded string
                       'pickle': Returns unpickled Python object
                       'file': Returns a file-like object for streaming
    :return: The loaded data in the specified format.
    """
    import pickle
    import io

    # Create a client
    client = storage.Client()

    # Get the bucket and blob
    bucket = client.bucket(bucket_name)
    blob = bucket.blob(filename)

    # Handle different output types
    if output_type == 'bytes':
        return blob.download_as_bytes()

    elif output_type == 'string':
        return blob.download_as_string().decode('utf-8')

    elif output_type == 'pickle':
        pickled_data = blob.download_as_bytes()
        return pickle.loads(pickled_data)

    elif output_type == 'file':
        file_obj = io.BytesIO()
        blob.download_to_file(file_obj)
        file_obj.seek(0)  # Reset file pointer to beginning
        return file_obj

    else:
        raise ValueError("Unsupported output_type. Must be one of: 'bytes', 'string', 'pickle', 'file'")

def load_artifacts_logreg(exp_number):

    model_filename = f'Oleh/tendo/experiments/{exp_number}/model.pkl'
    model = load_artifact_from_gcs(model_filename, GS_BUCKET)

    feature_filename = f'Oleh/tendo/experiments/{exp_number}/features.pkl'
    features = load_artifact_from_gcs(feature_filename, GS_BUCKET)

    scaler_filename = f'Oleh/tendo/experiments/{exp_number}/scaler.pkl'
    scaler = load_artifact_from_gcs(scaler_filename, GS_BUCKET)

    return model, features, scaler

def save_artifact_to_gcs(artifact, filename, bucket_name):
    """Saves the Cox model to Google Cloud Storage."""
    client = storage.Client()
    bucket = client.bucket(bucket_name)
    blob = bucket.blob(filename)

    # Serialize artifact
    artifact_bytes = pickle.dumps(artifact)

    # Upload to GCS
    blob.upload_from_string(artifact_bytes)
    print(f"Artifact saved to gs://{bucket_name}/{filename}")


import pickle
import io
from google.cloud import storage
def save_pickle_to_gcs(data, bucket_name, destination_blob_name):
    """
    Save any Python object as a pickle file to Google Cloud Storage
    
    Args:
        data: The Python object to pickle (DataFrame, dict, list, etc.)
        bucket_name: Name of the GCS bucket
        destination_blob_name: Path/filename in the bucket
    """
    # Initialize the GCS client
    client = storage.Client()
    bucket = client.bucket(bucket_name)
    blob = bucket.blob(destination_blob_name)
    
    # Serialize the data to pickle format in memory
    pickle_buffer = io.BytesIO()
    pickle.dump(data, pickle_buffer)
    pickle_buffer.seek(0)
    
    # Upload the pickle data to GCS
    blob.upload_from_file(pickle_buffer, content_type='application/octet-stream')
    print(f"Pickle file uploaded to gs://{bucket_name}/{destination_blob_name}")

In [4]:
import pandas as pd
from google.cloud import bigquery
from sklearn.metrics import roc_auc_score
from typing import Dict

def calculate_gini_for_table(
    data: pd.DataFrame,
    date_column: str,
    score_column: str,
    target_column: str,
    data_periods_dict: Dict,
    weights_col: str = None
):
    """
    Calculate Gini coefficient for different time periods.

    Args:
        project_id: BigQuery project ID
        table_name: Full table name (dataset.table)
        date_column: Name of the date column
        score_column: Name of the score column
        target_column: Name of the target column
        target_maturity_column: Name of the target maturity column
        data_periods_dict: Dictionary with periods, e.g.:
            {'Train': {'start': '2024-01-01', 'end': '2025-01-31'},
             'Test': {'start': '2025-02-01', 'end': '2025-12-31'}}

    Returns:
        pandas.DataFrame: Table with Gini coefficients for each period
    """
    dt = data[data[target_column].notna()].copy()

    # Convert date column to datetime and extract just the date part
    dt[date_column] = pd.to_datetime(dt[date_column]).dt.date

    # Initialize results
    gini_results = []

    print("Gini Coefficient Results:")
    print("=" * 50)

    # Calculate Gini for each period
    for period_name, period_info in data_periods_dict.items():
        start_date = pd.to_datetime(period_info['start']).date()
        end_date = pd.to_datetime(period_info['end']).date()

        # Filter data for the current period
        period_mask = (dt[date_column] >= start_date) & (dt[date_column] <= end_date)
        period_data = dt[period_mask].copy()

        sample_size = period_data['ee_customer_id'].nunique()
        bad_rate = 100*(1 - period_data[['ee_customer_id',target_column]].drop_duplicates()[target_column].sum() / sample_size)

        if len(period_data) == 0:
            print(f"{period_name}: No data available for period {start_date} to {end_date}")
            gini_results.append({
                'Period': period_name,
                'Start_Date': start_date,
                'End_Date': end_date,
                'Sample_Size': 0,
                'Bad Rate': np.nan,
                'Gini_Coefficient': None
            })
            continue

        # Check if we have both classes (0 and 1) in target
        unique_targets = period_data[target_column].unique()
        if len(unique_targets) < 2:
            print(f"{period_name}: Only one class present in target variable. Cannot calculate Gini.")
            gini_results.append({
                'Period': period_name,
                'Start_Date': start_date,
                'End_Date': end_date,
                'Sample_Size': sample_size,
                'Bad Rate': bad_rate,
                'Gini_Coefficient': None
            })
            continue

        # Calculate Gini coefficient
        try:
            if weights_col:
                auc = roc_auc_score(period_data[target_column], period_data[score_column], sample_weight=period_data[weights_col])
                gini = 2 * auc - 1
            else:
                auc = roc_auc_score(period_data[target_column], period_data[score_column])
                gini = 2 * auc - 1

            print(f"{period_name}: {round(gini, 4)} (Sample size: {len(period_data):,})")

            gini_results.append({
                'Period': period_name,
                'Start_Date': start_date,
                'End_Date': end_date,
                'Sample_Size': sample_size,
                'Bad Rate': bad_rate,
                'Gini_Coefficient': round(gini, 4)
            })

        except Exception as e:
            print(f"{period_name}: Error calculating Gini - {str(e)}")
            gini_results.append({
                'Period': period_name,
                'Start_Date': start_date,
                'End_Date': end_date,
                'Sample_Size': sample_size,
                'Bad Rate': bad_rate,
                'Gini_Coefficient': None
            })

    # Create results DataFrame
    results_df = pd.DataFrame(gini_results)

    print("\n" + "=" * 50)
    print("Summary Table:")
    print(results_df.to_string(index=False))

    return results_df

# Data loading

## PROD API

In [5]:
# PROD API
sql_query_prod_api = """
SELECT
  employee_id as ee_customer_id,
  run_date,
  ee_attrition_risk_segment as attrition_risk_segment,
  ee_attrition_time_to_leave as attrition_time_to_leave,
  oop_score as oop_score_prod,
  oop_risk_segment as oop_risk_segment_prod
FROM `prj-prod-dataplatform.tendo_mart.tendo_scorecard_master_table_api`
"""

dt_prod_api = client.query(sql_query_prod_api).to_dataframe()

In [6]:
dt_prod_api.sample(3)

,ee_customer_id,run_date,attrition_risk_segment,attrition_time_to_leave,oop_score_prod,oop_risk_segment_prod
144936,1525053,2025-08-23T05:35:38.399735,3,3,0.521454,B
50973,1207241,2025-11-08T06:08:22.786061,6,6,0.435917,D
118406,1551886,2025-09-16T06:43:20.777843,10-12,10,0.580491,A


## PROD BATCH

In [7]:
# PROD BATCH
sql_query_prod_batch = """
SELECT
  employee_id as ee_customer_id,
  run_date,
  ee_attrition_risk_segment as attrition_risk_segment,
  ee_attrition_time_to_leave as attrition_time_to_leave,
  oop_score as oop_score_prod,
  oop_risk_segment as oop_risk_segment_prod
FROM `prj-prod-dataplatform.tendo_mart.tendo_scorecard_master_table`
"""

dt_prod_batch = client.query(sql_query_prod_batch).to_dataframe()

In [8]:
dt_prod_batch.sample(3)

,ee_customer_id,run_date,attrition_risk_segment,attrition_time_to_leave,oop_score_prod,oop_risk_segment_prod
15947,1184815,2026-03-15,Average,8,0.719535,A
339937,1212566,2025-09-08,High,6,0.463088,E
76605,1104115,2026-02-02,Very low,12+,0.410247,D


## OOP Latest targets

In [9]:
# OOP Latest targets
sql_query_oop_targets = """
SELECT
  user_id as ee_customer_id,
  target as oop_target
FROM `prj-prod-dataplatform.tendo_mart.tendo_collection_target_master`
WHERE target_maturity_flag = 1
"""

dt_oop_targets = client.query(sql_query_oop_targets).to_dataframe()

In [10]:
dt_oop_targets.groupby('oop_target')['ee_customer_id'].nunique()

oop_target
0    28072
1     7994
Name: ee_customer_id, dtype: int64

## tendo_scorecard_features_data

In [11]:
dt = client.query("""select * from `prj-prod-dataplatform.worktable_data_analysis.tendo_scorecard_features_data_23-03-2026`;""").to_dataframe()

print(f"The shape of the tendo scorecard feature data is: {dt.shape}")

The shape of the tendo scorecard feature data is: (4168379, 100)


In [12]:
dt.head()

,ee_customer_id,ee_email,ee_phone_number,ee_firstname,ee_middlename,ee_lastname,ee_birthdate,ee_gender,ee_email_verified_flag,ee_telephone_verified_flag,ee_id_verified_flag,ee_income_verified_flag,ee_morning_time_contact_time,ee_afternoon_time_contact_time,ee_account_activated_flag,ee_region_name,ee_city_name,ee_barangay,ee_address_line_1,ee_address_line_2,ee_postal_code,ee_landmark,ee_residing_date,ee_employment_status,ee_civil_status,ee_kyc_doc_name,ee_is_citizen_flag,ee_onboarding_date,ee_employment_date,ee_fraud_status,ee_job_type,ee_comment,ee_department,ee_recommended_ir,ee_job_title,ee_employment_type,ee_nature_of_work,ee_permanent_freeze_date,ee_resignation_date,ee_product_type,ee_frozen_status,cust_risk_employer_time_score_v1,cust_risk_gender_score_v1,cust_risk_declared_income_score_v1,cust_risk_civil_status_score_v1,cust_risk_combined_score_v1,cust_risk_cat_v1,er_employer_id,er_employer_group_id,er_employer_name,er_email_domain,er_repayment_days_month,er_custom_email,er_payment_reminders,er_address,er_postal_code_id,er_max_base_interest_rate,er_employer_industry,er_employer_status,er_activated_at,er_deleted_at,er_created_at,er_updated_at,cl_monthly_utility_bills_amount,cl_monthly_income_gross,cl_monthly_income_net,cl_multiple_purchases_enabled_flag,cl_max_credit_limit_multiplier,cl_max_debt_income_ratio,cl_matured_fpd30_flag,cl_matured_fspd30_flag,cl_matured_fstpd30_flag,cl_matured_fstfpd30_flag,cl_fpd30_flag,cl_fspd30_flag,cl_fstpd30_flag,cl_fstfpd30_flag,ln_loan_id,ln_loan_type,ln_loan_status,ln_disbursement_channel,ln_original_principal,ln_orig_interest_fees,ln_orig_tenor,ln_loan_application_datetime,ln_repaid_full_flag,ln_fully_repaid_date,ln_fpd30_flag,ln_fspd30_flag,ln_fstpd30_flag,ln_fstfpd30_flag,ln_min_loan_due_date,ln_os_principal_at_fpd30,ln_os_principal_at_fspd30,ln_os_principal_at_fstpd30,ln_os_principal_at_fstfpd30,ln_matured_fpd30_flag,ln_matured_fspd30_flag,ln_matured_fstpd30_flag,ln_matured_fstfpd30_flag
0,243592,charlene.soan@withelp.games,+63 (917) 709 0644,Charlene Claire,Enriquez,Soan,1983-05-25,Female,1,1,2,2,None,None,2,None,None,None,404 Asiana Bldg. Mayfield Park Residences Rosa...,None,None,None,None,None,None,Payslip,None,2021-01-24 09:17:36+00:00,None,Risk,Regular employment,"900,Other",None,None,Regional HR and Operations Head,Permanent Job (Private sector),None,2025-09-08 17:43:13+00:00,2025-09-09,employer,Frozen,123,120,146,127,516,B,10068.000000000,None,Withelp Corp.,None,"[""15"",""30""]",1,5,Rockwell business center,379,4.0,8,PARKED,None,NaT,2021-01-22 18:34:29+00:00,2025-01-23 12:33:46+00:00,<NA>,100000,100000,1,125,60,1,1,1,1,1,1,1,1,491365,Tendo,Approved/Disbursed,None,799.0,0.0,2,2022-10-03 22:27:54+00:00,1,2022-12-14,0,0,0,0,2022-10-30,0.000,0.0,0.0,0.0,1,1,1,1
1,319435,oliver.dy6@gmail.com,+63 (917) 122 4227,Jon,Magdaluyo,Dy,1991-11-06,Male,1,1,2,2,None,None,2,None,Taguig City,None,14 M.H. del Pilar St. Zone 6 South Signal Village,None,1631,Jollibee triumph,None,regular,Married,Barangay Certificate w/ picture,true,2022-10-07 11:38:21+00:00,None,Risk,None,None,None,None,Business Analyst,Permanent Job (Private sector),None,NaT,None,employer,Frozen,183,118,110,127,538,A,10070.000000000,None,Carparts.Com,None,"[""15"",""30""]",1,8,"9th Floor, Unit 3,4,5 and 6 Milestone At Fifth...",406,4.0,38,IN_PROGRESS,2021-03-17 11:26:16.000000,NaT,2021-03-17 11:26:16+00:00,2025-11-03 12:42:04+00:00,3000,24500,21000,1,120,50,1,1,1,1,0,1,1,1,2500283,Tendo,Approved/Disbursed,PH_GRABPAY,1200.0,403.2,12,2024-11-26 12:52:24+00:00,0,None,0,0,0,0,2024-12-15,0.000,0.0,0.0,0.0,1,1,1,1
2,1272958,bernadithpaden@yahoo.com,+63 (917) 638 3320,Ma Bernadith,Salvo,Paden,1995-09-23,Female,1,1,2,2,None,None,2,NCR - NATIONAL CAPITAL REGION,None,SANTOLAN,88 Sampaguita St,None,None,Che Midwife Lying-in Clinic,None,probationary,Single,Payslip,true,2024-09-17 18:34:54+00:00,2024-07-15,Risk,None,None,None,None,Team Manager,Permanent Job (Private sector),None,NaT,None,employer,Frozen,98,120,146,117,481,C,10268.000000000,

In [13]:
dt.columns.values

array(['ee_customer_id', 'ee_email', 'ee_phone_number', 'ee_firstname',
       'ee_middlename', 'ee_lastname', 'ee_birthdate', 'ee_gender',
       'ee_email_verified_flag', 'ee_telephone_verified_flag',
       'ee_id_verified_flag', 'ee_income_verified_flag',
       'ee_morning_time_contact_time', 'ee_afternoon_time_contact_time',
       'ee_account_activated_flag', 'ee_region_name', 'ee_city_name',
       'ee_barangay', 'ee_address_line_1', 'ee_address_line_2',
       'ee_postal_code', 'ee_landmark', 'ee_residing_date',
       'ee_employment_status', 'ee_civil_status', 'ee_kyc_doc_name',
       'ee_is_citizen_flag', 'ee_onboarding_date', 'ee_employment_date',
       'ee_fraud_status', 'ee_job_type', 'ee_comment', 'ee_department',
       'ee_recommended_ir', 'ee_job_title', 'ee_employment_type',
       'ee_nature_of_work', 'ee_permanent_freeze_date',
       'ee_resignation_date', 'ee_product_type', 'ee_frozen_status',
       'cust_risk_employer_time_score_v1', 'cust_risk_gender_score_v

In [14]:
dt_raw = dt.copy()
dt_raw["ee_customer_id"] = dt_raw["ee_customer_id"].astype(str)
dt_raw["ee_onboarding_date"] = pd.to_datetime(dt_raw["ee_onboarding_date"]).dt.tz_localize(None)
print(f"The shape of the dt_raw a copy of dt is:\t{dt_raw.shape}")
# Keep only the two columns we need from dt_raw to avoid any future collision
dt_raw_slim = dt_raw[["ee_customer_id", "ee_onboarding_date"]].drop_duplicates()
print(f"The shape of the dt_raw_slim after droping the duplicates is:\t{dt_raw_slim.shape}")

The shape of the dt_raw a copy of dt is:	(4168379, 100)
The shape of the dt_raw_slim after droping the duplicates is:	(1742823, 2)


In [15]:
dd.query("""
    SELECT ee_customer_id, count(ee_customer_id) as cnt 
    FROM dt_raw 
    GROUP BY 1 
    HAVING count(ee_customer_id) > 1
    order by 2 desc;
""").to_df()


,ee_customer_id,cnt
0,1065228,1370
1,949503,718
2,1123001,712
3,1094726,698
4,529907,590
...,...,...
239837,553909,2
239838,1689148,2
239839,323629,2
239840,1688094,2


In [16]:
dt_raw_slim.sample(3)

,ee_customer_id,ee_onboarding_date
2013149,672501,NaT
2217137,1282220,2024-10-09 15:17:46
882161,293566,NaT


## tendo_scorecard_loan_repayment_data

In [17]:
dt_repayments = client.query("""select * from `prj-prod-dataplatform.worktable_data_analysis.tendo_scorecard_loan_repayment_data_23-03-2026`;""").to_dataframe()
print(f"The shape of the tendo scorecard repayment data is: {dt_repayments.shape}")

The shape of the tendo scorecard repayment data is: (16086906, 15)


In [18]:
dt_repayments.sample(3)

,user_id,loan_id,installment_number,vendor_id,subfamily,repaid_date,due_date,due_principal,due_interest,due_amount,paid_principal,paid_interest,paid_amount,outstanding_balance,DPD
6751930,1143383,1754113,23,TPSD,COPA,2025-07-15,2025-07-15,155.534,6.966,162.50,155.534,6.966,162.50,158.979,0
15295441,1388174,4433464,2,TPSD,COPA,2025-11-21,2025-10-30,145.291,15.959,161.25,145.291,15.959,161.25,612.416,22
3916656,1205057,3639042,17,TPSD,COPA,2026-02-27,2026-02-28,28.274,5.416,33.69,28.274,5.416,33.69,216.287,-1


## Resignation code

In [19]:
def to_date_str(val) -> str | None:
    # missing
    if val is None or pd.isna(val):
        return None

    # already a string
    if isinstance(val, str):
        return val.strip()

    # datetime-like (Timestamp, datetime, date, numpy datetime64)
    if isinstance(val, (pd.Timestamp, datetime, date, np.datetime64)):
        return pd.Timestamp(val).strftime("%Y-%m-%d")

    # fallback (numbers, objects, etc.)
    return str(val).strip()

def validate_and_convert_dates(df, column_name, output_tag_col, output_date_col,
                                min_date='1990-01-01',
                                max_date='2025-10-01',
                                min_date_col=None,
                                max_date_col=None):
    """
    Validates dates and creates tags with converted dates.

    Parameters:
    -----------
    df : pd.DataFrame
        DataFrame with date column
    column_name : str
        Name of the date column to validate
    min_date : str
        Default minimum date threshold (default: '1990-01-01')
    max_date : str
        Default maximum date threshold (default: '2025-10-01')
    min_date_col : str, optional
        Column name with per-row minimum date thresholds (overrides min_date when not null)
    max_date_col : str, optional
        Column name with per-row maximum date thresholds (overrides max_date when not null)

    Tags:
    - 'valid': Already in correct format and within range
    - 'convertible': Can be fixed (missing day, wrong format) and within range
    - 'invalid': Cannot be converted or outside valid range
    - 'null': NULL/NaN/empty values

    Returns DataFrame with:
    - date_tag: validation tag
    - date_converted: standardized date in YYYY-MM-DD format or None
    """

    default_min_threshold = pd.Timestamp(min_date)
    default_max_threshold = pd.Timestamp(max_date)

    def process_date(row):
        """Process a single date value with per-row thresholds"""

        val = row[column_name]

        # Determine thresholds for this row
        # Use per-row threshold if available and not null, otherwise use default
        if min_date_col and min_date_col in df.columns and pd.notna(row[min_date_col]):
            min_threshold = pd.Timestamp(row[min_date_col])
            # Remove timezone info if present to allow comparison
            if min_threshold.tzinfo is not None:
                min_threshold = min_threshold.tz_localize(None)
        else:
            min_threshold = default_min_threshold

        if max_date_col and max_date_col in df.columns and pd.notna(row[max_date_col]):
            max_threshold = pd.Timestamp(row[max_date_col])
            # Remove timezone info if present to allow comparison
            if max_threshold.tzinfo is not None:
                max_threshold = max_threshold.tz_localize(None)
        else:
            max_threshold = default_max_threshold

        # Handle NULL values
        if pd.isna(val):
            return 'null', None

        date_str = to_date_str(val)

        # Handle empty or 'nan' strings
        if date_str == '' or date_str.lower() == 'nan':
            return 'null', None

        # Check for invalid year pattern (must start with 19 or 20)
        year_match = re.search(r'(\d{4})', date_str)
        if year_match:
            year_str = year_match.group(1)
            first_digit = year_str[0]
            second_digit = year_str[1]

            # Year must start with 1 or 2
            if first_digit not in ['1', '2']:
                return 'invalid', None

            # If starts with 1, second digit must be 9 (1900-1999)
            if first_digit == '1' and second_digit != '9':
                return 'invalid', None

            # If starts with 2, second digit must be 0 (2000-2099)
            if first_digit == '2' and second_digit != '0':
                return 'invalid', None

        # Try to parse and convert the date
        converted_date = None
        needs_conversion = False

        # Pattern 1: YYYY-MM-DD (standard format with various separators)
        for sep in ['-', '/', '.']:
            pattern = f'^(\\d{{4}}){re.escape(sep)}(\\d{{1,2}}){re.escape(sep)}(\\d{{1,2}})$'
            match = re.match(pattern, date_str)
            if match:
                year, month, day = match.groups()
                needs_conversion = False  # Reset for each pattern

                # Store original to check if padding needed
                original_month_len = len(month)
                original_day_len = len(day)

                # Pad month and day if needed
                month = month.zfill(2)
                day = day.zfill(2)

                # Check if padding was needed
                if original_month_len == 1 or original_day_len == 1:
                    needs_conversion = True

                # DON'T swap - the input format is already YYYY-MM-DD
                # (We only swap for DD-MM-YYYY format in Pattern 4)

                try:
                    parsed = pd.Timestamp(f"{year}-{month}-{day}")
                    converted_date = f"{year}-{month}-{day}"

                    # Check if within valid range
                    if parsed < min_threshold or parsed > max_threshold:
                        return 'invalid', None

                    # Different separator means needs conversion
                    if sep != '-':
                        needs_conversion = True

                    return 'convertible' if needs_conversion else 'valid', converted_date
                except:
                    return 'invalid', None

        # Pattern 2: YYYY-MM (missing day)
        for sep in ['-', '/', '.']:
            pattern = f'^(\\d{{4}}){re.escape(sep)}(\\d{{1,2}})$'
            match = re.match(pattern, date_str)
            if match:
                year, month = match.groups()
                month = month.zfill(2)
                day = '01'  # Default to first day of month

                try:
                    parsed = pd.Timestamp(f"{year}-{month}-{day}")
                    converted_date = f"{year}-{month}-{day}"

                    # Check if within valid range
                    if parsed < min_threshold or parsed > max_threshold:
                        return 'invalid', None

                    return 'convertible', converted_date
                except:
                    return 'invalid', None

        # Pattern 3: YYYYMMDD (no separator)
        if re.match(r'^\d{8}$', date_str):
            year = date_str[0:4]
            month = date_str[4:6]
            day = date_str[6:8]

            try:
                parsed = pd.Timestamp(f"{year}-{month}-{day}")
                converted_date = f"{year}-{month}-{day}"

                # Check if within valid range
                if parsed < min_threshold or parsed > max_threshold:
                    return 'invalid', None

                return 'convertible', converted_date
            except:
                return 'invalid', None

        # Pattern 4: DD-MM-YYYY or MM-DD-YYYY (need to detect which)
        for sep in ['-', '/', '.']:
            pattern = f'^(\\d{{1,2}}){re.escape(sep)}(\\d{{1,2}}){re.escape(sep)}(\\d{{4}})$'
            match = re.match(pattern, date_str)
            if match:
                part1, part2, year = match.groups()
                part1 = part1.zfill(2)
                part2 = part2.zfill(2)

                # Determine if DD-MM-YYYY or MM-DD-YYYY
                # If part1 > 12, it must be day, so DD-MM-YYYY
                if int(part1) > 12:
                    day, month = part1, part2
                # If part2 > 12, it must be day, so MM-DD-YYYY
                elif int(part2) > 12:
                    month, day = part1, part2
                # Ambiguous case: assume MM-DD-YYYY (common US format)
                else:
                    # Could be either, default to MM-DD-YYYY but mark as needing review
                    month, day = part1, part2

                try:
                    parsed = pd.Timestamp(f"{year}-{month}-{day}")
                    converted_date = f"{year}-{month}-{day}"

                    # Check if within valid range
                    if parsed < min_threshold or parsed > max_threshold:
                        return 'invalid', None

                    return 'convertible', converted_date
                except:
                    return 'invalid', None

        # If nothing matched, it's invalid
        return 'invalid', None

    # Apply processing to all rows
    results = df.apply(process_date, axis=1)

    df_result = pd.DataFrame(index=df.index)
    df_result[output_tag_col] = results.apply(lambda x: x[0])
    df_result[output_date_col] = pd.to_datetime(results.apply(lambda x: x[1]))

    return df_result

# Filtering
dt = dt[dt['ee_product_type'] == 'employer']
dt = dt[dt['ee_onboarding_date'].notna()].reset_index(drop=True)

# loan table preparation
str_columns = ['ee_customer_id', 'ln_loan_id'] 

missing_values = ['<NA>', 'None', 'NaN', 'nan', '', 'null', 'NULL']
dt[str_columns] = dt[str_columns].replace(missing_values, np.nan).astype('string')

localize_date_cols = ['ee_permanent_freeze_date','ee_onboarding_date']
for col in localize_date_cols:
    print(col)
    dt[col] = pd.to_datetime(dt[col]).dt.tz_localize(None)

date_col_format = ['ee_resignation_date']
for date_col in date_col_format:
    dt[date_col] = pd.to_datetime(dt[date_col], format='%Y-%m-%d', errors='coerce')

# repayment table preparation
dt_repayments = dt_repayments.sort_values(
    ['loan_id', 'repaid_date', 'installment_number', 'outstanding_balance', 'paid_amount'],
    ascending=[True, True, True, False, False]
)

dt_repayments['repaid_date'] = pd.to_datetime(dt_repayments['repaid_date'], format='%Y-%m-%d')
dt_repayments['due_date'] = pd.to_datetime(dt_repayments['due_date'], format='%Y-%m-%d')

dt_repayments['loan_id'] = dt_repayments['loan_id'].astype('str')
dt_repayments['user_id'] = dt_repayments['user_id'].astype('str')



resignation_dates = validate_and_convert_dates(
    df=dt.groupby('ee_customer_id')[['ee_resignation_date']].first(),
    column_name='ee_resignation_date',
    output_tag_col='ee_resignation_date_tag',
    output_date_col='ee_resignation_date_validated',
    min_date=dt['ee_onboarding_date'].min().strftime(format='%Y-%m-%d'),
    max_date='2026-03-23',
    min_date_col=None,
    max_date_col=None
)

permanent_freeze_dates = validate_and_convert_dates(
    df=dt.groupby('ee_customer_id')[['ee_permanent_freeze_date']].first(),
    column_name='ee_permanent_freeze_date',
    output_tag_col='ee_permanent_freeze_date_tag',
    output_date_col='ee_permanent_freeze_date_validated',
    min_date=dt['ee_onboarding_date'].min().strftime(format='%Y-%m-%d'),
    max_date='2026-03-23',
    min_date_col=None,
    max_date_col=None
)

resignation_dates = resignation_dates.merge(
        permanent_freeze_dates['ee_permanent_freeze_date_validated'],
        how='left',
        on='ee_customer_id'
)

resignation_dates['ee_resignation_date_correct'] = resignation_dates.loc[:, 'ee_resignation_date_validated'].fillna(resignation_dates['ee_permanent_freeze_date_validated'])

dt = dt.merge(
    resignation_dates['ee_resignation_date_correct'].reset_index(),
    how='left',on='ee_customer_id'
)

dt_repayments['vendor_id_shifted'] = dt_repayments.groupby('loan_id')[['vendor_id']].shift(-1)
dt_repayments['repaid_date_shifted'] = dt_repayments.groupby('loan_id')[['repaid_date']].shift(-1)

resignation_dates_repayments = (
    dt_repayments[['user_id','loan_id','vendor_id','vendor_id_shifted','repaid_date']]
    .query('vendor_id == "TPSD"')
    .fillna('TPSD')
    .query('vendor_id != vendor_id_shifted & vendor_id_shifted == "TPFP"')
    .assign(resignation_date_fp_new = lambda x: x['repaid_date'] + pd.DateOffset(months=1))
    .groupby('user_id')[['resignation_date_fp_new']]
    .max()
    .reset_index()
)

dt_resignation = dt.merge(resignation_dates_repayments, how='left', left_on='ee_customer_id', right_on='user_id')
dt_resignation['ee_resignation_date_correct'] = dt_resignation['ee_resignation_date_correct'].fillna(dt_resignation['resignation_date_fp_new'])

dt_res = dt_resignation[['ee_customer_id','ee_resignation_date_correct']].drop_duplicates() 

ee_permanent_freeze_date
ee_onboarding_date


In [20]:
dt_resignation.sample(3)

,ee_customer_id,ee_email,ee_phone_number,ee_firstname,ee_middlename,ee_lastname,ee_birthdate,ee_gender,ee_email_verified_flag,ee_telephone_verified_flag,ee_id_verified_flag,ee_income_verified_flag,ee_morning_time_contact_time,ee_afternoon_time_contact_time,ee_account_activated_flag,ee_region_name,ee_city_name,ee_barangay,ee_address_line_1,ee_address_line_2,ee_postal_code,ee_landmark,ee_residing_date,ee_employment_status,ee_civil_status,ee_kyc_doc_name,ee_is_citizen_flag,ee_onboarding_date,ee_employment_date,ee_fraud_status,ee_job_type,ee_comment,ee_department,ee_recommended_ir,ee_job_title,ee_employment_type,ee_nature_of_work,ee_permanent_freeze_date,ee_resignation_date,ee_product_type,ee_frozen_status,cust_risk_employer_time_score_v1,cust_risk_gender_score_v1,cust_risk_declared_income_score_v1,cust_risk_civil_status_score_v1,cust_risk_combined_score_v1,cust_risk_cat_v1,er_employer_id,er_employer_group_id,er_employer_name,er_email_domain,er_repayment_days_month,er_custom_email,er_payment_reminders,er_address,er_postal_code_id,er_max_base_interest_rate,er_employer_industry,er_employer_status,er_activated_at,er_deleted_at,er_created_at,er_updated_at,cl_monthly_utility_bills_amount,cl_monthly_income_gross,cl_monthly_income_net,cl_multiple_purchases_enabled_flag,cl_max_credit_limit_multiplier,cl_max_debt_income_ratio,cl_matured_fpd30_flag,cl_matured_fspd30_flag,cl_matured_fstpd30_flag,cl_matured_fstfpd30_flag,cl_fpd30_flag,cl_fspd30_flag,cl_fstpd30_flag,cl_fstfpd30_flag,ln_loan_id,ln_loan_type,ln_loan_status,ln_disbursement_channel,ln_original_principal,ln_orig_interest_fees,ln_orig_tenor,ln_loan_application_datetime,ln_repaid_full_flag,ln_fully_repaid_date,ln_fpd30_flag,ln_fspd30_flag,ln_fstpd30_flag,ln_fstfpd30_flag,ln_min_loan_due_date,ln_os_principal_at_fpd30,ln_os_principal_at_fspd30,ln_os_principal_at_fstpd30,ln_os_principal_at_fstfpd30,ln_matured_fpd30_flag,ln_matured_fspd30_flag,ln_matured_fstpd30_flag,ln_matured_fstfpd30_flag,ee_resignation_date_correct,user_id,resignation_date_fp_new
1462255,1045355,jeansilida@gmail.com,+63 (905) 405 2600,Mecell Jean,Mendez,Silida,1981-05-31,Female,1,1,2,2,None,None,2,Region VII,None,TALAMBAN,"11767-A-1 Hillside, Talamban Cebu City",None,None,Archival's Apartment,None,regular,Married,None,true,2023-09-04 09:53:00,None,Normal,None,None,None,None,Associate,Permanent Job (Private sector),None,NaT,NaT,employer,Not Frozen,123,120,119,127,489,C,10158.000000000,None,Innodata,None,"[""7"",""22""]",1,-1,None,None,3.5,8,IN_PROGRESS,2023-04-20 16:29:28.000000,NaT,2023-04-20 16:29:28+00:00,2025-08-28 02:24:43+00:00,1500,14000,9000,1,188,40,1,1,1,1,0,0,0,0,8215729,Tendo,Approved/Disbursed,PH_BDO,10000.0,325.00,1,2026-03-12 12:52:13+00:00,0,None,0,0,0,0,2026-04-07,0.0,0.0,0.0,0.0,0,0,0,0,NaT,NaN,NaT
1633460,1294454,rangerjennifer04@gmail.com,+63 (962) 647 0343,Jennifer,Dungca,Yumul,1995-10-04,Female,0,1,2,2,None,None,2,Region III,None,ESCALER,#307 Purok 4,None,None,Near Escaler Elementary School,None,probationary,Married,Payslip,true,2024-11-16 17:11:02,2024-10-01,Normal,None,None,None,None,Jr HR Administrator,Permanent Job (Private sector),None,NaT,NaT,employer,Not Frozen,98,120,146,127,491,C,10162.000000000,None,ConnectOS,None,"[""15"",""30""]",1,7,None,None,3.0,8,IN_PROGRESS,2023-05-30 12:34:48.000000,NaT,2023-05-30 12:34:48+00:00,2025-03-17 18:35:54+00:00,2000,52500,45000,1,280,60,1,1,1,1,0,0,0,0,5018229,Tendo,Approved/Disbursed,PH_UBP,16700.0,2296.25,5,2025-11-19 09:19:31+00:00,0,None,0,0,0,0,2025-12-15,0.0,0.0,0.0,0.0,1,1,1,0,NaT,NaN,NaT
962384,1201985,kurt.zapata@supportninja.com,+63 (977) 745 1932,Kurt Russel,Dalmacio,Zapata,1994-04-19,Male,1,1,2,2,None,None,2,NCR - NATIONAL CAPITAL REGION,None,BARANGAY 409,305 A H Lacson,None,None,Barangay 409 Day Care Center,None,regular,Single,Barangay Certificate w/ picture,true,2024-05-26 21:39:44,None,None,None,None,None,None,Customer Support Representative,Permanent Job (Private sector),None,NaT,NaT,employer,Not Frozen,139,118,118,117,492,C,10065.000000

In [21]:
print(f"The shape of the dt_resignation is:\{dt_resignation.shape}")

The shape of the dt_resignation is:\(2509446, 103)


<string>:1: SyntaxWarning: invalid escape sequence '\{'
<>:1: SyntaxWarning: invalid escape sequence '\{'
<string>:1: SyntaxWarning: invalid escape sequence '\{'
<>:1: SyntaxWarning: invalid escape sequence '\{'
C:\Users\Dwaipayan\AppData\Local\Temp\ipykernel_21304\467682833.py:1: SyntaxWarning: invalid escape sequence '\{'
  print(f"The shape of the dt_resignation is:\{dt_resignation.shape}")


In [22]:
# Your modified code
bucket_name = GS_BUCKET
pickle_filename = f"resignation_data_{CURRENT_DATE}"

# Construct the new filename with .pkl extension
data_filename = f"{pickle_filename}.pkl"
print(data_filename)

destination_blob_name = f"{CLOUDPATH}/{data_filename}"

# Save the DataFrame (or any object) as pickle to GCS
save_pickle_to_gcs(dt_resignation[['ee_customer_id','ee_resignation_date_correct']], bucket_name, destination_blob_name)



resignation_data_20260330.pkl
Pickle file uploaded to gs://prod-asia-southeast1-tonik-aiml-workspace/DC/Tendo_Model_Monitoring_Data/resignation_data_20260330.pkl


## Outstanding balance code

In [23]:
dt_repayments = dt_repayments.sort_values(
    ['loan_id', 'repaid_date', 'installment_number', 'outstanding_balance', 'paid_amount'],
    ascending=[True, True, True, False, False]
)

# converting loan_id and user_id to str in order to be aligned with loan data
dt_repayments['loan_id'] = dt_repayments['loan_id'].astype('str')
dt_repayments['user_id'] = dt_repayments['user_id'].astype('str')

# merging with loan data to get columns for actual osbal calculations
dt_repayments = dt_repayments.merge(dt[['ln_loan_id','ln_original_principal', 'ln_orig_interest_fees']].drop_duplicates(),
                                    how='left', left_on='loan_id', right_on='ln_loan_id')

# converting dates
dt_repayments['repaid_date'] = pd.to_datetime(dt_repayments['repaid_date'], format='%Y-%m-%d')
dt_repayments['due_date'] = pd.to_datetime(dt_repayments['due_date'], format='%Y-%m-%d')

# Calculate cumulative paid amount per loan
dt_repayments['cumulative_paid_principal'] = dt_repayments.groupby('loan_id')['paid_principal'].cumsum()

# Calculate osbal_after directly from loan_amount - cumulative_paid
dt_repayments['osbal_after'] = dt_repayments['ln_original_principal'] - dt_repayments['cumulative_paid_principal']

# Calculate osbal_before by shifting osbal_after and using loan_amount for first payment
dt_repayments['osbal_before'] = dt_repayments.groupby('loan_id')['osbal_after'].shift(1)

# Fill first payment osbal_before with loan_amount
mask = dt_repayments['osbal_before'].isna()
dt_repayments.loc[mask, 'osbal_before'] = dt_repayments.loc[mask, 'ln_original_principal']

In [24]:
dt_repayments.sample(2)

,user_id,loan_id,installment_number,vendor_id,subfamily,repaid_date,due_date,due_principal,due_interest,due_amount,paid_principal,paid_interest,paid_amount,outstanding_balance,DPD,vendor_id_shifted,repaid_date_shifted,ln_loan_id,ln_original_principal,ln_orig_interest_fees,cumulative_paid_principal,osbal_after,osbal_before
1100148,748620,1302778,1,XEDD,COPA,2024-03-16,2024-04-15,135.736,33.014,168.75,111.09,0.0,111.09,1364.264,-30,XEDD,2024-03-25,1302778,1500.0,187.50,135.736,1364.264,1475.354
5366131,1167954,2526168,4,TPSD,COPA,2025-01-30,2025-01-30,156.530,10.100,166.63,156.53,10.1,166.63,322.990,0,TPSD,2025-02-14,2526168,930.0,69.75,607.010,322.990,479.520


In [25]:
dt_raw.sample(3)

,ee_customer_id,ee_email,ee_phone_number,ee_firstname,ee_middlename,ee_lastname,ee_birthdate,ee_gender,ee_email_verified_flag,ee_telephone_verified_flag,ee_id_verified_flag,ee_income_verified_flag,ee_morning_time_contact_time,ee_afternoon_time_contact_time,ee_account_activated_flag,ee_region_name,ee_city_name,ee_barangay,ee_address_line_1,ee_address_line_2,ee_postal_code,ee_landmark,ee_residing_date,ee_employment_status,ee_civil_status,ee_kyc_doc_name,ee_is_citizen_flag,ee_onboarding_date,ee_employment_date,ee_fraud_status,ee_job_type,ee_comment,ee_department,ee_recommended_ir,ee_job_title,ee_employment_type,ee_nature_of_work,ee_permanent_freeze_date,ee_resignation_date,ee_product_type,ee_frozen_status,cust_risk_employer_time_score_v1,cust_risk_gender_score_v1,cust_risk_declared_income_score_v1,cust_risk_civil_status_score_v1,cust_risk_combined_score_v1,cust_risk_cat_v1,er_employer_id,er_employer_group_id,er_employer_name,er_email_domain,er_repayment_days_month,er_custom_email,er_payment_reminders,er_address,er_postal_code_id,er_max_base_interest_rate,er_employer_industry,er_employer_status,er_activated_at,er_deleted_at,er_created_at,er_updated_at,cl_monthly_utility_bills_amount,cl_monthly_income_gross,cl_monthly_income_net,cl_multiple_purchases_enabled_flag,cl_max_credit_limit_multiplier,cl_max_debt_income_ratio,cl_matured_fpd30_flag,cl_matured_fspd30_flag,cl_matured_fstpd30_flag,cl_matured_fstfpd30_flag,cl_fpd30_flag,cl_fspd30_flag,cl_fstpd30_flag,cl_fstfpd30_flag,ln_loan_id,ln_loan_type,ln_loan_status,ln_disbursement_channel,ln_original_principal,ln_orig_interest_fees,ln_orig_tenor,ln_loan_application_datetime,ln_repaid_full_flag,ln_fully_repaid_date,ln_fpd30_flag,ln_fspd30_flag,ln_fstpd30_flag,ln_fstfpd30_flag,ln_min_loan_due_date,ln_os_principal_at_fpd30,ln_os_principal_at_fspd30,ln_os_principal_at_fstpd30,ln_os_principal_at_fstfpd30,ln_matured_fpd30_flag,ln_matured_fspd30_flag,ln_matured_fstpd30_flag,ln_matured_fstfpd30_flag
2038107,548336,arbhiemananguite2021@gmail.com,+63 (956) 609 7586,joicylyn,None,delmundo,1999-09-07,Female,1,1,1,1,None,None,3,NCR - NATIONAL CAPITAL REGION,None,BAGONG SILANGAN,road 1 area c bagong silangan qc,None,1119,None,None,None,Single,Barangay Certificate w/ picture,None,NaT,None,None,None,None,None,None,Remittance Beneficiary,Remittance Beneficiary,None,NaT,None,regular,None,<NA>,<NA>,<NA>,<NA>,<NA>,None,None,None,None,None,None,<NA>,<NA>,None,None,NaN,None,None,None,NaT,NaT,NaT,<NA>,None,None,0,<NA>,None,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,None,None,None,NaN,NaN,<NA>,NaT,<NA>,None,0,0,0,0,None,NaN,NaN,NaN,NaN,0,0,0,0
1728002,1688286,jcrivera168@gmail.com,+63 (945) 881 4856,None,None,None,None,None,1,1,0,0,None,None,0,None,None,None,None,None,None,None,None,None,None,None,None,NaT,None,Normal,None,None,None,None,None,None,None,NaT,None,regular,None,<NA>,<NA>,<NA>,<NA>,<NA>,None,None,None,None,None,None,<NA>,<NA>,None,None,NaN,None,None,None,NaT,NaT,NaT,<NA>,None,None,0,<NA>,None,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,None,None,None,NaN,NaN,<NA>,NaT,<NA>,None,0,0,0,0,None,NaN,NaN,NaN,NaN,0,0,0,0
1280635,637407,meebucog@gmail.com,+63 (917) 143 9687,AIMEE,BUCOG,KONG,1988-04-12,Female,1,1,2,2,None,None,2,None,None,None,Purok Sampaguita Sta Cruz,None,6600,None,None,None,Married,Payslip,None,2021-11-21 11:01:41,None,Risk,None,None,None,None,REATAIL ASSISTANT,Permanent Job (Private sector),None,2023-04-06 16:18:22+00:00,None,regular,None,<NA>,<NA>,<NA>,<NA>,<NA>,None,None,None,None,None,None,<NA>,<NA>,None,None,NaN,None,None,None,NaT,NaT,NaT,<NA>,36000,32000,0,<NA>,None,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,None,None,None,NaN,NaN,<NA>,NaT,<NA>,None,0,0,0,0,None,NaN,NaN,NaN,NaN,0,0,0,0


# OOP

In [26]:
# BS OOP new
sql_query_oop_new = """
SELECT *
FROM `prj-prod-dataplatform.risk_mart.tendo_backscored_new_users_jan23_feb26_20260301_oop_with_osbal`
"""

dt_bs_oop_new = client.query(sql_query_oop_new).to_dataframe()
print(f"The shape of the dataframe after reading tendo_backscored_new_users_jan23_feb26_20260301_oop_with_osbal table is :\t {dt_bs_oop_new.shape}")

The shape of the dataframe after reading tendo_backscored_new_users_jan23_feb26_20260301_oop_with_osbal table is :	 (169297, 7)


In [27]:
dt_bs_oop_new.sample(3)

,ee_customer_id,calc_date,target_dev,score_oop,osbal_as_of_resignation_date,osbal_as_of_oop_eligible_date,osbal_as_of_current_date
5516,991854,2023-03-01,0.0,0.413880,12011.694,11836.368,12011.694
71866,1579497,2025-12-01,NaN,0.480093,NaN,NaN,2628.456
99293,1462615,2025-07-01,NaN,0.533538,NaN,NaN,1555.954


In [28]:
# BS OOP existing
sql_query_oop_existing = """
SELECT *
FROM `prj-prod-dataplatform.risk_mart.tendo_backscored_existing_users_jan23_feb26_20260301_oop`
"""

dt_bs_oop_ex = client.query(sql_query_oop_existing).to_dataframe()
print(f"The shape of dataframe after reading table `prj-prod-dataplatform.risk_mart.tendo_backscored_existing_users_jan23_feb26_20260301_oop` is:\t{dt_bs_oop_ex.shape}")

The shape of dataframe after reading table `prj-prod-dataplatform.risk_mart.tendo_backscored_existing_users_jan23_feb26_20260301_oop` is:	(1531733, 4)


In [29]:
dt_bs_oop_ex.sample(3)

,ee_customer_id,calc_date,target_dev,score_oop
1054185,1367254,2026-02-01,NaN,0.641151
1030854,1020331,2026-02-01,NaN,0.465786
1179550,1211318,2024-11-01,NaN,0.423076


In [30]:
dt_bs_oop_new['ee_customer_id'] = dt_bs_oop_new['ee_customer_id'].astype('str')
dt_bs_oop_ex['ee_customer_id'] = dt_bs_oop_ex['ee_customer_id'].astype('str')

In [31]:
dt_bs_oop_ex["calc_date"] = pd.to_datetime(dt_bs_oop_ex["calc_date"], errors="coerce")
dt_bs_oop_ex["calc_date_correct"] = pd.to_datetime(dt_bs_oop_ex["calc_date"], errors="coerce") - pd.DateOffset(months=1)
dt_bs_oop_ex['calc_date_ym'] = dt_bs_oop_ex['calc_date_correct'].dt.year*100 + dt_bs_oop_ex['calc_date_correct'].dt.month
dt_bs_oop_ex.sample(3)

,ee_customer_id,calc_date,target_dev,score_oop,calc_date_correct,calc_date_ym
639787,1253266,2025-06-01,NaN,0.700640,2025-05-01,202505
95844,883736,2023-12-01,NaN,0.324459,2023-11-01,202311
366058,1131363,2025-03-01,NaN,0.604654,2025-02-01,202502


# Gini

## prod new

In [32]:
dt_prod_api = dt_prod_api.merge(
    dt_raw_slim[['ee_customer_id','ee_onboarding_date']].drop_duplicates(),
    how='left',
    on='ee_customer_id'
).merge(
    dt_oop_targets,
    how='left',
    on='ee_customer_id'
).merge(
    dt_bs_oop_new[['ee_customer_id', 'score_oop', 'osbal_as_of_resignation_date', 'osbal_as_of_oop_eligible_date', 'osbal_as_of_current_date']],
    how='left',
    on='ee_customer_id'
)

dt_prod_api.sample(3)

,ee_customer_id,run_date,attrition_risk_segment,attrition_time_to_leave,oop_score_prod,oop_risk_segment_prod,ee_onboarding_date,oop_target,score_oop,osbal_as_of_resignation_date,osbal_as_of_oop_eligible_date,osbal_as_of_current_date
41230,1633176,2025-11-22T01:09:10.690786,5,5,0.500944,C,2025-12-22 14:44:17,<NA>,0.576986,14353.611,2884.76,2884.76
120464,1628084,2025-11-17T02:52:18.335277,3,3,0.482380,C,NaT,<NA>,NaN,NaN,NaN,NaN
51796,1494857,2025-08-26T01:59:31.462325,2,2,0.557827,B,NaT,<NA>,NaN,NaN,NaN,NaN


In [33]:
dt_prod_api['onb_rd_diff'] = (abs(pd.to_datetime(dt_prod_api['run_date']).dt.normalize() - pd.to_datetime(dt_prod_api['ee_onboarding_date']).dt.normalize())).dt.days
dt_prod_api['ee_onboarding_date'] = pd.to_datetime(dt_prod_api['ee_onboarding_date']).dt.normalize()
dt_prod_api['run_date'] = pd.to_datetime(dt_prod_api['run_date']).dt.normalize()
dt_prod_api['onboarding_date_ym'] = dt_prod_api['ee_onboarding_date'].dt.year*100 + dt_prod_api['ee_onboarding_date'].dt.month
dt_prod_api['run_date_ym'] = dt_prod_api['run_date'].dt.year*100 + dt_prod_api['run_date'].dt.month

dt_prod_api.sample(3)

,ee_customer_id,run_date,attrition_risk_segment,attrition_time_to_leave,oop_score_prod,oop_risk_segment_prod,ee_onboarding_date,oop_target,score_oop,osbal_as_of_resignation_date,osbal_as_of_oop_eligible_date,osbal_as_of_current_date,onb_rd_diff,onboarding_date_ym,run_date_ym
70985,1518449,2025-08-31,3,3,0.578527,A,2025-08-31,<NA>,0.598311,NaN,NaN,0.000,0.0,202508.0,202508
39919,1337080,2025-12-12,6,6,0.410585,D,NaT,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,202512
159858,1211192,2026-03-28,8,8,0.243131,F,2024-06-07,<NA>,0.247114,NaN,NaN,11667.451,659.0,202406.0,202603


In [34]:
dt_prod_api_calc = (dt_prod_api
                    .dropna(subset=['ee_onboarding_date','oop_target'])
                    .sort_values(['onb_rd_diff','run_date'])
                    .drop_duplicates(subset=['ee_customer_id'], keep='first'))

dt_prod_api_calc.sample(3)

,ee_customer_id,run_date,attrition_risk_segment,attrition_time_to_leave,oop_score_prod,oop_risk_segment_prod,ee_onboarding_date,oop_target,score_oop,osbal_as_of_resignation_date,osbal_as_of_oop_eligible_date,osbal_as_of_current_date,onb_rd_diff,onboarding_date_ym,run_date_ym
66525,522308,2026-01-09,5,5,0.392244,E,2021-08-20,0,NaN,NaN,NaN,NaN,1603.0,202108.0,202601
68915,1051081,2025-09-27,7,7,0.666605,A,2023-05-23,0,0.632121,73589.011,73589.011,29845.794,858.0,202305.0,202509
25725,1193802,2025-06-27,6,6,0.442401,F,2024-05-08,0,0.394847,39100.141,39100.141,39100.141,415.0,202405.0,202506


### OOP New Users Prod

In [35]:
print('OOP New Users Prod')
print("\n" + "=" * 50)

data_periods_dict = {
    'Jun 2025': {'start': '2025-06-01', 'end': '2025-06-30'},
    'Jul 2025': {'start': '2025-07-01', 'end': '2025-07-31'},
    'Aug 2025': {'start': '2025-08-01', 'end': '2025-08-31'},
    'Sep 2025': {'start': '2025-09-01', 'end': '2025-09-30'},
    'Jun 2025 - Sep 2025': {'start': '2025-06-01', 'end': '2025-09-30'}
}

oopNewUserProd = calculate_gini_for_table(
    data = dt_prod_api_calc,
    date_column = 'ee_onboarding_date',
    score_column = 'oop_score_prod',
    target_column = 'oop_target',
    data_periods_dict = data_periods_dict
)

oopNewUserProd

OOP New Users Prod

Gini Coefficient Results:
Jun 2025: 0.25 (Sample size: 15)
Jul 2025: 0.025 (Sample size: 117)
Aug 2025: 0.0593 (Sample size: 85)
Sep 2025: -0.2441 (Sample size: 106)
Jun 2025 - Sep 2025: -0.0396 (Sample size: 323)

Summary Table:
             Period Start_Date   End_Date  Sample_Size  Bad Rate  Gini_Coefficient
           Jun 2025 2025-06-01 2025-06-30           15 53.333333            0.2500
           Jul 2025 2025-07-01 2025-07-31          117 66.666667            0.0250
           Aug 2025 2025-08-01 2025-08-31           85 70.588235            0.0593
           Sep 2025 2025-09-01 2025-09-30          106 72.641509           -0.2441
Jun 2025 - Sep 2025 2025-06-01 2025-09-30          323 69.040248           -0.0396


,Period,Start_Date,End_Date,Sample_Size,Bad Rate,Gini_Coefficient
0,Jun 2025,2025-06-01,2025-06-30,15,53.333333,0.2500
1,Jul 2025,2025-07-01,2025-07-31,117,66.666667,0.0250
2,Aug 2025,2025-08-01,2025-08-31,85,70.588235,0.0593
3,Sep 2025,2025-09-01,2025-09-30,106,72.641509,-0.2441
4,Jun 2025 - Sep 2025,2025-06-01,2025-09-30,323,69.040248,-0.0396


### OOP New Users Prod BS

In [36]:
print('OOP New Users Prod BS')
print("\n" + "=" * 50)

data_periods_dict = {
    'Jun 2025': {'start': '2025-06-01', 'end': '2025-06-30'},
    'Jul 2025': {'start': '2025-07-01', 'end': '2025-07-31'},
    'Aug 2025': {'start': '2025-08-01', 'end': '2025-08-31'},
    'Sep 2025': {'start': '2025-09-01', 'end': '2025-09-30'},
    'Jun 2025 - Sep 2025': {'start': '2025-06-01', 'end': '2025-09-30'}
}

oopNewUsersProdBS = calculate_gini_for_table(
    data = dt_prod_api_calc[dt_prod_api_calc['score_oop'].notna()],
    date_column = 'ee_onboarding_date',
    score_column = 'score_oop',
    target_column = 'oop_target',
    data_periods_dict = data_periods_dict
)

oopNewUsersProdBS

OOP New Users Prod BS

Gini Coefficient Results:
Jun 2025: 0.0357 (Sample size: 15)
Jul 2025: 0.3217 (Sample size: 110)
Aug 2025: 0.0647 (Sample size: 80)
Sep 2025: -0.1887 (Sample size: 101)
Jun 2025 - Sep 2025: 0.0586 (Sample size: 306)

Summary Table:
             Period Start_Date   End_Date  Sample_Size  Bad Rate  Gini_Coefficient
           Jun 2025 2025-06-01 2025-06-30           15 53.333333            0.0357
           Jul 2025 2025-07-01 2025-07-31          110 66.363636            0.3217
           Aug 2025 2025-08-01 2025-08-31           80 70.000000            0.0647
           Sep 2025 2025-09-01 2025-09-30          101 73.267327           -0.1887
Jun 2025 - Sep 2025 2025-06-01 2025-09-30          306 68.954248            0.0586


,Period,Start_Date,End_Date,Sample_Size,Bad Rate,Gini_Coefficient
0,Jun 2025,2025-06-01,2025-06-30,15,53.333333,0.0357
1,Jul 2025,2025-07-01,2025-07-31,110,66.363636,0.3217
2,Aug 2025,2025-08-01,2025-08-31,80,70.000000,0.0647
3,Sep 2025,2025-09-01,2025-09-30,101,73.267327,-0.1887
4,Jun 2025 - Sep 2025,2025-06-01,2025-09-30,306,68.954248,0.0586


### OOP New Users Prod, weighted as-is

In [37]:
print('OOP New Users Prod, weighted as-is')
print("\n" + "=" * 50)

data_periods_dict = {
    'Jun 2025': {'start': '2025-06-01', 'end': '2025-06-30'},
    'Jul 2025': {'start': '2025-07-01', 'end': '2025-07-31'},
    'Aug 2025': {'start': '2025-08-01', 'end': '2025-08-31'},
    'Sep 2025': {'start': '2025-09-01', 'end': '2025-09-30'},
    'Jun 2025 - Sep 2025': {'start': '2025-06-01', 'end': '2025-09-30'}
}

oopNewUsersProdwgasis = calculate_gini_for_table(
    data = dt_prod_api_calc[dt_prod_api_calc['osbal_as_of_oop_eligible_date'].notna()],
    date_column = 'ee_onboarding_date',
    score_column = 'oop_score_prod',
    target_column = 'oop_target',
    data_periods_dict = data_periods_dict,
    weights_col='osbal_as_of_oop_eligible_date'
)

oopNewUsersProdwgasis

OOP New Users Prod, weighted as-is

Gini Coefficient Results:
Jun 2025: 0.3099 (Sample size: 14)
Jul 2025: -0.1667 (Sample size: 101)
Aug 2025: 0.098 (Sample size: 76)
Sep 2025: -0.2237 (Sample size: 100)
Jun 2025 - Sep 2025: -0.093 (Sample size: 291)

Summary Table:
             Period Start_Date   End_Date  Sample_Size  Bad Rate  Gini_Coefficient
           Jun 2025 2025-06-01 2025-06-30           14 57.142857            0.3099
           Jul 2025 2025-07-01 2025-07-31          101 68.316832           -0.1667
           Aug 2025 2025-08-01 2025-08-31           76 71.052632            0.0980
           Sep 2025 2025-09-01 2025-09-30          100 74.000000           -0.2237
Jun 2025 - Sep 2025 2025-06-01 2025-09-30          291 70.446735           -0.0930


,Period,Start_Date,End_Date,Sample_Size,Bad Rate,Gini_Coefficient
0,Jun 2025,2025-06-01,2025-06-30,14,57.142857,0.3099
1,Jul 2025,2025-07-01,2025-07-31,101,68.316832,-0.1667
2,Aug 2025,2025-08-01,2025-08-31,76,71.052632,0.0980
3,Sep 2025,2025-09-01,2025-09-30,100,74.000000,-0.2237
4,Jun 2025 - Sep 2025,2025-06-01,2025-09-30,291,70.446735,-0.0930


In [38]:
dt_prod_api_calc['osbal_as_of_oop_eligible_date_log'] = np.log1p(dt_prod_api_calc['osbal_as_of_oop_eligible_date'])

In [39]:
dt_prod_api_calc.sample(5)

,ee_customer_id,run_date,attrition_risk_segment,attrition_time_to_leave,oop_score_prod,oop_risk_segment_prod,ee_onboarding_date,oop_target,score_oop,osbal_as_of_resignation_date,osbal_as_of_oop_eligible_date,osbal_as_of_current_date,onb_rd_diff,onboarding_date_ym,run_date_ym,osbal_as_of_oop_eligible_date_log
25727,1193750,2025-06-27,6,6,0.458825,E,2024-05-10,0,0.441948,27580.858,27580.858,27580.858,413.0,202405.0,202506,10.224914
148275,646524,2025-10-08,8,8,0.620673,B,2022-01-14,1,NaN,NaN,NaN,NaN,1363.0,202201.0,202510,NaN
65622,968490,2025-10-06,8,8,0.411161,D,2023-01-16,0,0.419874,102081.776,86123.776,86123.776,994.0,202301.0,202510,11.363552
25336,1665798,2025-12-26,4,4,0.722708,A,2025-12-26,1,0.700180,21026.784,21026.784,0.001,0.0,202512.0,202512,9.953600
16767,1470980,2025-07-02,3,3,0.515797,B,2025-07-02,1,0.533586,6033.194,6033.194,5190.839,0.0,202507.0,202507,8.705198


### OOP New Users Prod, weighted log

In [40]:
print('OOP New Users Prod, weighted log')
print("\n" + "=" * 50)

data_periods_dict = {
    'Jun 2025': {'start': '2025-06-01', 'end': '2025-06-30'},
    'Jul 2025': {'start': '2025-07-01', 'end': '2025-07-31'},
    'Aug 2025': {'start': '2025-08-01', 'end': '2025-08-31'},
    'Sep 2025': {'start': '2025-09-01', 'end': '2025-09-30'},
    'Oct 2025': {'start': '2025-10-01', 'end': '2025-10-31'},
    'Nov 2025': {'start': '2025-11-01', 'end': '2025-11-30'},
    'Dec 2025': {'start': '2025-12-01', 'end': '2025-12-31'},
    'Jun 2025 - Dec 2025': {'start': '2025-06-01', 'end': '2025-12-31'}
}

oopNewUsersProdwglog = calculate_gini_for_table(
    data = dt_prod_api_calc[dt_prod_api_calc['osbal_as_of_oop_eligible_date_log'].notna()],
    date_column = 'ee_onboarding_date',
    score_column = 'oop_score_prod',
    target_column = 'oop_target',
    data_periods_dict = data_periods_dict,
    weights_col='osbal_as_of_oop_eligible_date_log'
)

oopNewUsersProdwglog

OOP New Users Prod, weighted log

Gini Coefficient Results:
Jun 2025: 0.212 (Sample size: 14)
Jul 2025: 0.0025 (Sample size: 101)
Aug 2025: 0.0833 (Sample size: 76)
Sep 2025: -0.262 (Sample size: 100)
Oct 2025: 0.0809 (Sample size: 37)
Nov 2025: 0.3789 (Sample size: 60)
Dec 2025: -0.2167 (Sample size: 24)
Jun 2025 - Dec 2025: 0.0047 (Sample size: 412)

Summary Table:
             Period Start_Date   End_Date  Sample_Size  Bad Rate  Gini_Coefficient
           Jun 2025 2025-06-01 2025-06-30           14 57.142857            0.2120
           Jul 2025 2025-07-01 2025-07-31          101 68.316832            0.0025
           Aug 2025 2025-08-01 2025-08-31           76 71.052632            0.0833
           Sep 2025 2025-09-01 2025-09-30          100 74.000000           -0.2620
           Oct 2025 2025-10-01 2025-10-31           37 64.864865            0.0809
           Nov 2025 2025-11-01 2025-11-30           60 68.333333            0.3789
           Dec 2025 2025-12-01 2025-12-31        

,Period,Start_Date,End_Date,Sample_Size,Bad Rate,Gini_Coefficient
0,Jun 2025,2025-06-01,2025-06-30,14,57.142857,0.2120
1,Jul 2025,2025-07-01,2025-07-31,101,68.316832,0.0025
2,Aug 2025,2025-08-01,2025-08-31,76,71.052632,0.0833
3,Sep 2025,2025-09-01,2025-09-30,100,74.000000,-0.2620
4,Oct 2025,2025-10-01,2025-10-31,37,64.864865,0.0809
5,Nov 2025,2025-11-01,2025-11-30,60,68.333333,0.3789
6,Dec 2025,2025-12-01,2025-12-31,24,50.000000,-0.2167
7,Jun 2025 - Dec 2025,2025-06-01,2025-12-31,412,68.446602,0.0047


### OOP New Users Prod, weighted log - 4 months

In [41]:
print('OOP New Users Prod, weighted log - 4 months')
print("\n" + "=" * 50)

data_periods_dict = {
    'Jun 2025': {'start': '2025-06-01', 'end': '2025-06-30'},
    'Jul 2025': {'start': '2025-07-01', 'end': '2025-07-31'},
    'Aug 2025': {'start': '2025-08-01', 'end': '2025-08-31'},
    'Sep 2025': {'start': '2025-09-01', 'end': '2025-09-30'},
    'Jun 2025 - Sep 2025': {'start': '2025-06-01', 'end': '2025-09-30'}
}

oopNewUsersProdwglog4mnt = calculate_gini_for_table(
    data = dt_prod_api_calc[dt_prod_api_calc['osbal_as_of_oop_eligible_date_log'].notna()],
    date_column = 'ee_onboarding_date',
    score_column = 'oop_score_prod',
    target_column = 'oop_target',
    data_periods_dict = data_periods_dict,
    weights_col='osbal_as_of_oop_eligible_date_log'
)

oopNewUsersProdwglog4mnt

OOP New Users Prod, weighted log - 4 months

Gini Coefficient Results:
Jun 2025: 0.212 (Sample size: 14)
Jul 2025: 0.0025 (Sample size: 101)
Aug 2025: 0.0833 (Sample size: 76)
Sep 2025: -0.262 (Sample size: 100)
Jun 2025 - Sep 2025: -0.054 (Sample size: 291)

Summary Table:
             Period Start_Date   End_Date  Sample_Size  Bad Rate  Gini_Coefficient
           Jun 2025 2025-06-01 2025-06-30           14 57.142857            0.2120
           Jul 2025 2025-07-01 2025-07-31          101 68.316832            0.0025
           Aug 2025 2025-08-01 2025-08-31           76 71.052632            0.0833
           Sep 2025 2025-09-01 2025-09-30          100 74.000000           -0.2620
Jun 2025 - Sep 2025 2025-06-01 2025-09-30          291 70.446735           -0.0540


,Period,Start_Date,End_Date,Sample_Size,Bad Rate,Gini_Coefficient
0,Jun 2025,2025-06-01,2025-06-30,14,57.142857,0.2120
1,Jul 2025,2025-07-01,2025-07-31,101,68.316832,0.0025
2,Aug 2025,2025-08-01,2025-08-31,76,71.052632,0.0833
3,Sep 2025,2025-09-01,2025-09-30,100,74.000000,-0.2620
4,Jun 2025 - Sep 2025,2025-06-01,2025-09-30,291,70.446735,-0.0540


### OOP New Users Prod BS, weighted as-is

In [42]:
print('OOP New Users Prod BS, weighted as-is')
print("\n" + "=" * 50)

data_periods_dict = {
    'Jun 2025': {'start': '2025-06-01', 'end': '2025-06-30'},
    'Jul 2025': {'start': '2025-07-01', 'end': '2025-07-31'},
    'Aug 2025': {'start': '2025-08-01', 'end': '2025-08-31'},
    'Sep 2025': {'start': '2025-09-01', 'end': '2025-09-30'},
    'Jun 2025 - Sep 2025': {'start': '2025-06-01', 'end': '2025-09-30'}
}

oopNewUsersProdBSwgasis = calculate_gini_for_table(
    data = dt_prod_api_calc[(dt_prod_api_calc['score_oop'].notna()) & dt_prod_api_calc['osbal_as_of_oop_eligible_date'].notna()],
    date_column = 'ee_onboarding_date',
    score_column = 'score_oop',
    target_column = 'oop_target',
    data_periods_dict = data_periods_dict,
    weights_col='osbal_as_of_oop_eligible_date'
)

oopNewUsersProdBSwgasis

OOP New Users Prod BS, weighted as-is

Gini Coefficient Results:
Jun 2025: 0.2166 (Sample size: 14)
Jul 2025: 0.0882 (Sample size: 101)
Aug 2025: -0.2214 (Sample size: 76)
Sep 2025: -0.2322 (Sample size: 100)
Jun 2025 - Sep 2025: -0.0452 (Sample size: 291)

Summary Table:
             Period Start_Date   End_Date  Sample_Size  Bad Rate  Gini_Coefficient
           Jun 2025 2025-06-01 2025-06-30           14 57.142857            0.2166
           Jul 2025 2025-07-01 2025-07-31          101 68.316832            0.0882
           Aug 2025 2025-08-01 2025-08-31           76 71.052632           -0.2214
           Sep 2025 2025-09-01 2025-09-30          100 74.000000           -0.2322
Jun 2025 - Sep 2025 2025-06-01 2025-09-30          291 70.446735           -0.0452


,Period,Start_Date,End_Date,Sample_Size,Bad Rate,Gini_Coefficient
0,Jun 2025,2025-06-01,2025-06-30,14,57.142857,0.2166
1,Jul 2025,2025-07-01,2025-07-31,101,68.316832,0.0882
2,Aug 2025,2025-08-01,2025-08-31,76,71.052632,-0.2214
3,Sep 2025,2025-09-01,2025-09-30,100,74.000000,-0.2322
4,Jun 2025 - Sep 2025,2025-06-01,2025-09-30,291,70.446735,-0.0452


### OOP New Users Prod BS, weighted log transformed

In [43]:
print('OOP New Users Prod BS, weighted log transformed')
print("\n" + "=" * 50)

data_periods_dict = {
    'Jun 2025': {'start': '2025-06-01', 'end': '2025-06-30'},
    'Jul 2025': {'start': '2025-07-01', 'end': '2025-07-31'},
    'Aug 2025': {'start': '2025-08-01', 'end': '2025-08-31'},
    'Sep 2025': {'start': '2025-09-01', 'end': '2025-09-30'},
    'Oct 2025': {'start': '2025-10-01', 'end': '2025-10-31'},
    'Nov 2025': {'start': '2025-11-01', 'end': '2025-11-30'},
    'Dec 2025': {'start': '2025-12-01', 'end': '2025-12-31'},
    'Jun 2025 - Dec 2025': {'start': '2025-06-01', 'end': '2025-12-31'}
}

oopNewUsersProdBSwglogtranformed = calculate_gini_for_table(
    data = dt_prod_api_calc[(dt_prod_api_calc['score_oop'].notna()) & dt_prod_api_calc['osbal_as_of_oop_eligible_date_log'].notna()],
    date_column = 'ee_onboarding_date',
    score_column = 'score_oop',
    target_column = 'oop_target',
    data_periods_dict = data_periods_dict,
    weights_col='osbal_as_of_oop_eligible_date_log'
)

oopNewUsersProdBSwglogtranformed

OOP New Users Prod BS, weighted log transformed

Gini Coefficient Results:
Jun 2025: 0.0937 (Sample size: 14)
Jul 2025: 0.253 (Sample size: 101)
Aug 2025: 0.0318 (Sample size: 76)
Sep 2025: -0.2194 (Sample size: 100)
Oct 2025: 0.0886 (Sample size: 37)
Nov 2025: 0.3944 (Sample size: 60)
Dec 2025: -0.0069 (Sample size: 24)
Jun 2025 - Dec 2025: 0.0687 (Sample size: 412)

Summary Table:
             Period Start_Date   End_Date  Sample_Size  Bad Rate  Gini_Coefficient
           Jun 2025 2025-06-01 2025-06-30           14 57.142857            0.0937
           Jul 2025 2025-07-01 2025-07-31          101 68.316832            0.2530
           Aug 2025 2025-08-01 2025-08-31           76 71.052632            0.0318
           Sep 2025 2025-09-01 2025-09-30          100 74.000000           -0.2194
           Oct 2025 2025-10-01 2025-10-31           37 64.864865            0.0886
           Nov 2025 2025-11-01 2025-11-30           60 68.333333            0.3944
           Dec 2025 2025-12-01 20

,Period,Start_Date,End_Date,Sample_Size,Bad Rate,Gini_Coefficient
0,Jun 2025,2025-06-01,2025-06-30,14,57.142857,0.0937
1,Jul 2025,2025-07-01,2025-07-31,101,68.316832,0.2530
2,Aug 2025,2025-08-01,2025-08-31,76,71.052632,0.0318
3,Sep 2025,2025-09-01,2025-09-30,100,74.000000,-0.2194
4,Oct 2025,2025-10-01,2025-10-31,37,64.864865,0.0886
5,Nov 2025,2025-11-01,2025-11-30,60,68.333333,0.3944
6,Dec 2025,2025-12-01,2025-12-31,24,50.000000,-0.0069
7,Jun 2025 - Dec 2025,2025-06-01,2025-12-31,412,68.446602,0.0687


### OOP New Users Prod BS, weighted log transformed - 4 months

In [44]:
print('OOP New Users Prod BS, weighted log transformed - 4 months')
print("\n" + "=" * 50)

data_periods_dict = {
    'Jun 2025': {'start': '2025-06-01', 'end': '2025-06-30'},
    'Jul 2025': {'start': '2025-07-01', 'end': '2025-07-31'},
    'Aug 2025': {'start': '2025-08-01', 'end': '2025-08-31'},
    'Sep 2025': {'start': '2025-09-01', 'end': '2025-09-30'},
    'Jun 2025 - Sep 2025': {'start': '2025-06-01', 'end': '2025-09-30'}
}

oopNewUsersProdBSwglogtransformed4mnt = calculate_gini_for_table(
    data = dt_prod_api_calc[(dt_prod_api_calc['score_oop'].notna()) & dt_prod_api_calc['osbal_as_of_oop_eligible_date_log'].notna()],
    date_column = 'ee_onboarding_date',
    score_column = 'score_oop',
    target_column = 'oop_target',
    data_periods_dict = data_periods_dict,
    weights_col='osbal_as_of_oop_eligible_date_log'
)

oopNewUsersProdBSwglogtransformed4mnt

OOP New Users Prod BS, weighted log transformed - 4 months

Gini Coefficient Results:
Jun 2025: 0.0937 (Sample size: 14)
Jul 2025: 0.253 (Sample size: 101)
Aug 2025: 0.0318 (Sample size: 76)
Sep 2025: -0.2194 (Sample size: 100)
Jun 2025 - Sep 2025: 0.0148 (Sample size: 291)

Summary Table:
             Period Start_Date   End_Date  Sample_Size  Bad Rate  Gini_Coefficient
           Jun 2025 2025-06-01 2025-06-30           14 57.142857            0.0937
           Jul 2025 2025-07-01 2025-07-31          101 68.316832            0.2530
           Aug 2025 2025-08-01 2025-08-31           76 71.052632            0.0318
           Sep 2025 2025-09-01 2025-09-30          100 74.000000           -0.2194
Jun 2025 - Sep 2025 2025-06-01 2025-09-30          291 70.446735            0.0148


,Period,Start_Date,End_Date,Sample_Size,Bad Rate,Gini_Coefficient
0,Jun 2025,2025-06-01,2025-06-30,14,57.142857,0.0937
1,Jul 2025,2025-07-01,2025-07-31,101,68.316832,0.2530
2,Aug 2025,2025-08-01,2025-08-31,76,71.052632,0.0318
3,Sep 2025,2025-09-01,2025-09-30,100,74.000000,-0.2194
4,Jun 2025 - Sep 2025,2025-06-01,2025-09-30,291,70.446735,0.0148


## prod existing

In [45]:
dt_prod_batch.shape

(405731, 6)

In [46]:
dt_prod_batch['ee_customer_id'].nunique()

80293

In [47]:
dt_prod_batch = dt_prod_batch.drop_duplicates(subset=['ee_customer_id','run_date','attrition_time_to_leave','oop_score_prod'])

In [48]:
dt_prod_batch.shape

(404256, 6)

In [49]:
dt_prod_batch["run_date"] = pd.to_datetime(dt_prod_batch["run_date"], errors="coerce")
dt_prod_batch['run_date_ym'] = dt_prod_batch['run_date'].dt.year*100 + dt_prod_batch['run_date'].dt.month

In [50]:
dt_prod_batch.sample(5)

,ee_customer_id,run_date,attrition_risk_segment,attrition_time_to_leave,oop_score_prod,oop_risk_segment_prod,run_date_ym
126254,1081109,2026-03-08,Very low,12+,0.466714,C,202603
245318,276818,2025-11-06,Average,9,0.604527,B,202511
221511,1344164,2025-10-21,Very low,12+,0.499197,C,202510
183665,1149574,2025-10-24,Low,10,0.382661,E,202510
50768,1156567,2025-10-19,Very low,12+,0.437725,D,202510


In [51]:
dt_prod_batch = dt_prod_batch.merge(
    dt_raw_slim[['ee_customer_id','ee_onboarding_date']].drop_duplicates(),
    how='left',
    on='ee_customer_id'
).merge(
    dt_oop_targets,
    how='left',
    on='ee_customer_id'
).merge(
    dt_bs_oop_new[['ee_customer_id', 'osbal_as_of_resignation_date', 'osbal_as_of_oop_eligible_date', 'osbal_as_of_current_date']],
    how='left',
    on='ee_customer_id'
).merge(
    dt_bs_oop_ex[['ee_customer_id', 'calc_date_ym', 'score_oop']],
    how='left',
    left_on=['ee_customer_id','run_date_ym'],
    right_on=['ee_customer_id','calc_date_ym']
)

In [52]:
dt_prod_batch.sample(5)

,ee_customer_id,run_date,attrition_risk_segment,attrition_time_to_leave,oop_score_prod,oop_risk_segment_prod,run_date_ym,ee_onboarding_date,oop_target,osbal_as_of_resignation_date,osbal_as_of_oop_eligible_date,osbal_as_of_current_date,calc_date_ym,score_oop
237925,1253751,2026-01-16,Low,11,0.457784,D,202601,2024-08-16 17:25:52,<NA>,NaN,NaN,14171.801,202601.0,0.395140
5736,1072182,2025-12-24,Very low,12+,0.633293,B,202512,2023-06-24 11:59:47,<NA>,NaN,NaN,107503.363,202512.0,0.595388
318215,1317838,2025-07-14,High,4,0.481429,C,202507,2025-01-14 17:49:21,<NA>,NaN,NaN,31802.068,202507.0,0.593634
239939,638210,2026-01-10,High,6,0.698275,B,202601,2021-12-10 11:16:41,<NA>,NaN,NaN,NaN,202601.0,0.696596
104534,1095842,2025-12-06,Very low,12+,0.429880,D,202512,2023-08-06 15:00:03,<NA>,NaN,NaN,1936.386,202512.0,0.409313


In [53]:
dt_prod_batch.rename(columns={'ee_onboarding_date_x':'ee_onboarding_date', }, inplace=True)

In [54]:
dt_prod_batch['ee_onboarding_date'] = pd.to_datetime(dt_prod_batch['ee_onboarding_date']).dt.normalize()
dt_prod_batch['onboarding_date_ym'] = dt_prod_batch['ee_onboarding_date'].dt.year*100 + dt_prod_batch['ee_onboarding_date'].dt.month
dt_prod_batch['onb_rd_diff'] = (abs(dt_prod_batch['run_date'] - dt_prod_batch['ee_onboarding_date'])).dt.days

In [55]:
dt_prod_batch.sample(5)

,ee_customer_id,run_date,attrition_risk_segment,attrition_time_to_leave,oop_score_prod,oop_risk_segment_prod,run_date_ym,ee_onboarding_date,oop_target,osbal_as_of_resignation_date,osbal_as_of_oop_eligible_date,osbal_as_of_current_date,calc_date_ym,score_oop,onboarding_date_ym,onb_rd_diff
314597,1473253,2025-09-10,Very low,12+,0.531673,B,202509,2025-07-10,<NA>,NaN,NaN,29191.693,NaN,NaN,202507,62
238641,1148532,2026-01-17,Average,7,0.642319,B,202601,2024-02-17,<NA>,NaN,NaN,13458.442,202601.0,0.490205,202402,700
189871,1014081,2025-12-12,Very low,12+,0.693225,B,202512,2023-04-12,<NA>,NaN,NaN,2935.302,202512.0,0.670569,202304,975
120615,1569036,2026-03-01,Very high,3,0.707576,A,202603,2025-11-01,<NA>,NaN,NaN,0.000,NaN,NaN,202511,120
133529,1011844,2026-03-22,Average,7,0.574346,B,202603,2023-03-22,<NA>,NaN,NaN,23645.618,NaN,NaN,202303,1096


In [56]:
dt_prod_batch_calc = (dt_prod_batch
                    .dropna(subset=['ee_onboarding_date','oop_target'])
                    .sort_values(['ee_customer_id','run_date']))

In [57]:
print(f"The shape of dt_prod_batch is:\t{dt_prod_batch.shape}")
print(f"The shape of dt_prod_batch_calc after dropping na is:\t{dt_prod_batch_calc.shape}")

The shape of dt_prod_batch is:	(404256, 16)
The shape of dt_prod_batch_calc after dropping na is:	(16648, 16)


### OOP Existing Users Prod

In [58]:
print('OOP Existing Users Prod')
print("\n" + "=" * 50)

data_periods_dict = {
    'Jun 2025': {'start': '2025-06-01', 'end': '2025-06-30'},
    'Jul 2025': {'start': '2025-07-01', 'end': '2025-07-31'},
    'Aug 2025': {'start': '2025-08-01', 'end': '2025-08-31'},
    'Sep 2025': {'start': '2025-09-01', 'end': '2025-09-30'},
    'Jun 2025 - Sep 2025': {'start': '2025-06-01', 'end': '2025-09-30'}
}

oopExtUsersProd = calculate_gini_for_table(
    data = dt_prod_batch_calc,
    date_column = 'run_date',
    score_column = 'oop_score_prod',
    target_column = 'oop_target',
    data_periods_dict = data_periods_dict
)

oopExtUsersProd

OOP Existing Users Prod

Gini Coefficient Results:
Jun 2025: 0.2667 (Sample size: 2,124)
Jul 2025: 0.2726 (Sample size: 3,429)
Aug 2025: 0.2618 (Sample size: 2,734)
Sep 2025: 0.2658 (Sample size: 2,121)
Jun 2025 - Sep 2025: 0.2657 (Sample size: 10,408)

Summary Table:
             Period Start_Date   End_Date  Sample_Size  Bad Rate  Gini_Coefficient
           Jun 2025 2025-06-01 2025-06-30         2109 75.011854            0.2667
           Jul 2025 2025-07-01 2025-07-31         3429 72.761738            0.2726
           Aug 2025 2025-08-01 2025-08-31         2734 70.592538            0.2618
           Sep 2025 2025-09-01 2025-09-30         2121 68.646865            0.2658
Jun 2025 - Sep 2025 2025-06-01 2025-09-30         4381 73.316594            0.2657


,Period,Start_Date,End_Date,Sample_Size,Bad Rate,Gini_Coefficient
0,Jun 2025,2025-06-01,2025-06-30,2109,75.011854,0.2667
1,Jul 2025,2025-07-01,2025-07-31,3429,72.761738,0.2726
2,Aug 2025,2025-08-01,2025-08-31,2734,70.592538,0.2618
3,Sep 2025,2025-09-01,2025-09-30,2121,68.646865,0.2658
4,Jun 2025 - Sep 2025,2025-06-01,2025-09-30,4381,73.316594,0.2657


### OOP Existing Users Prod BS

In [59]:
print('OOP Existing Users Prod BS')
print("\n" + "=" * 50)

data_periods_dict = {
    'Jun 2025': {'start': '2025-06-01', 'end': '2025-06-30'},
    'Jul 2025': {'start': '2025-07-01', 'end': '2025-07-31'},
    'Aug 2025': {'start': '2025-08-01', 'end': '2025-08-31'},
    'Sep 2025': {'start': '2025-09-01', 'end': '2025-09-30'},
    'Jun 2025 - Sep 2025': {'start': '2025-06-01', 'end': '2025-09-30'}
}

oopExtUsersProdBS = calculate_gini_for_table(
    data = dt_prod_batch_calc[dt_prod_batch_calc['score_oop'].notna()],
    date_column = 'run_date',
    score_column = 'score_oop',
    target_column = 'oop_target',
    data_periods_dict = data_periods_dict
)

oopExtUsersProdBS

OOP Existing Users Prod BS

Gini Coefficient Results:
Jun 2025: 0.2987 (Sample size: 1,569)
Jul 2025: 0.3015 (Sample size: 2,418)
Aug 2025: 0.3134 (Sample size: 1,902)
Sep 2025: 0.3112 (Sample size: 1,504)
Jun 2025 - Sep 2025: 0.3061 (Sample size: 7,393)

Summary Table:
             Period Start_Date   End_Date  Sample_Size  Bad Rate  Gini_Coefficient
           Jun 2025 2025-06-01 2025-06-30         1556 75.192802            0.2987
           Jul 2025 2025-07-01 2025-07-31         2418 71.464020            0.3015
           Aug 2025 2025-08-01 2025-08-31         1902 68.401682            0.3134
           Sep 2025 2025-09-01 2025-09-30         1504 67.021277            0.3112
Jun 2025 - Sep 2025 2025-06-01 2025-09-30         3153 72.851253            0.3061


,Period,Start_Date,End_Date,Sample_Size,Bad Rate,Gini_Coefficient
0,Jun 2025,2025-06-01,2025-06-30,1556,75.192802,0.2987
1,Jul 2025,2025-07-01,2025-07-31,2418,71.464020,0.3015
2,Aug 2025,2025-08-01,2025-08-31,1902,68.401682,0.3134
3,Sep 2025,2025-09-01,2025-09-30,1504,67.021277,0.3112
4,Jun 2025 - Sep 2025,2025-06-01,2025-09-30,3153,72.851253,0.3061


### OOP Existing Users Prod, weighted as-is

In [60]:
print('OOP Existing Users Prod, weighted as-is')
print("\n" + "=" * 50)

data_periods_dict = {
    'Jun 2025': {'start': '2025-06-01', 'end': '2025-06-30'},
    'Jul 2025': {'start': '2025-07-01', 'end': '2025-07-31'},
    'Aug 2025': {'start': '2025-08-01', 'end': '2025-08-31'},
    'Sep 2025': {'start': '2025-09-01', 'end': '2025-09-30'},
    'Jun 2025 - Sep 2025': {'start': '2025-06-01', 'end': '2025-09-30'}
}

oopExtusersProdwgasis = calculate_gini_for_table(
    data = dt_prod_batch_calc[dt_prod_batch_calc['osbal_as_of_oop_eligible_date'].notna()],
    date_column = 'run_date',
    score_column = 'oop_score_prod',
    target_column = 'oop_target',
    data_periods_dict = data_periods_dict,
    weights_col = 'osbal_as_of_oop_eligible_date'
)

oopExtusersProdwgasis

OOP Existing Users Prod, weighted as-is

Gini Coefficient Results:
Jun 2025: 0.2796 (Sample size: 1,891)
Jul 2025: 0.2229 (Sample size: 3,035)
Aug 2025: 0.2186 (Sample size: 2,402)
Sep 2025: 0.2109 (Sample size: 1,882)
Jun 2025 - Sep 2025: 0.2272 (Sample size: 9,210)

Summary Table:
             Period Start_Date   End_Date  Sample_Size  Bad Rate  Gini_Coefficient
           Jun 2025 2025-06-01 2025-06-30         1880 76.329787            0.2796
           Jul 2025 2025-07-01 2025-07-31         3035 73.937397            0.2229
           Aug 2025 2025-08-01 2025-08-31         2402 71.981682            0.2186
           Sep 2025 2025-09-01 2025-09-30         1882 70.616366            0.2109
Jun 2025 - Sep 2025 2025-06-01 2025-09-30         3886 74.626866            0.2272


,Period,Start_Date,End_Date,Sample_Size,Bad Rate,Gini_Coefficient
0,Jun 2025,2025-06-01,2025-06-30,1880,76.329787,0.2796
1,Jul 2025,2025-07-01,2025-07-31,3035,73.937397,0.2229
2,Aug 2025,2025-08-01,2025-08-31,2402,71.981682,0.2186
3,Sep 2025,2025-09-01,2025-09-30,1882,70.616366,0.2109
4,Jun 2025 - Sep 2025,2025-06-01,2025-09-30,3886,74.626866,0.2272


In [61]:
dt_prod_batch_calc['osbal_as_of_oop_eligible_date_log'] = np.log1p(dt_prod_batch_calc['osbal_as_of_oop_eligible_date'])

### OOP Existing Users Prod, weighted as-is - jun25 - Dec25

In [62]:
print('OOP Existing Users Prod, weighted as-is')
print("\n" + "=" * 50)

data_periods_dict = {
    'Jun 2025': {'start': '2025-06-01', 'end': '2025-06-30'},
    'Jul 2025': {'start': '2025-07-01', 'end': '2025-07-31'},
    'Aug 2025': {'start': '2025-08-01', 'end': '2025-08-31'},
    'Sep 2025': {'start': '2025-09-01', 'end': '2025-09-30'},
    'Oct 2025': {'start': '2025-10-01', 'end': '2025-10-31'},
    'Nov 2025': {'start': '2025-11-01', 'end': '2025-11-30'},
    'Dec 2025': {'start': '2025-12-01', 'end': '2025-12-31'},
    'Jun 2025 - Dec 2025': {'start': '2025-06-01', 'end': '2025-12-31'}
}

oopExtUsersProdwgasisjun25dec25 = calculate_gini_for_table(
    data = dt_prod_batch_calc[dt_prod_batch_calc['osbal_as_of_oop_eligible_date'].notna()],
    date_column = 'run_date',
    score_column = 'oop_score_prod',
    target_column = 'oop_target',
    data_periods_dict = data_periods_dict,
    weights_col = 'osbal_as_of_oop_eligible_date'
)

oopExtUsersProdwgasisjun25dec25

OOP Existing Users Prod, weighted as-is

Gini Coefficient Results:
Jun 2025: 0.2796 (Sample size: 1,891)
Jul 2025: 0.2229 (Sample size: 3,035)
Aug 2025: 0.2186 (Sample size: 2,402)
Sep 2025: 0.2109 (Sample size: 1,882)
Oct 2025: 0.1706 (Sample size: 1,557)
Nov 2025: 0.2293 (Sample size: 1,321)
Dec 2025: 0.2447 (Sample size: 950)
Jun 2025 - Dec 2025: 0.2129 (Sample size: 13,038)

Summary Table:
             Period Start_Date   End_Date  Sample_Size  Bad Rate  Gini_Coefficient
           Jun 2025 2025-06-01 2025-06-30         1880 76.329787            0.2796
           Jul 2025 2025-07-01 2025-07-31         3035 73.937397            0.2229
           Aug 2025 2025-08-01 2025-08-31         2402 71.981682            0.2186
           Sep 2025 2025-09-01 2025-09-30         1882 70.616366            0.2109
           Oct 2025 2025-10-01 2025-10-31         1557 68.593449            0.1706
           Nov 2025 2025-11-01 2025-11-30         1321 68.130204            0.2293
           Dec 2025 20

,Period,Start_Date,End_Date,Sample_Size,Bad Rate,Gini_Coefficient
0,Jun 2025,2025-06-01,2025-06-30,1880,76.329787,0.2796
1,Jul 2025,2025-07-01,2025-07-31,3035,73.937397,0.2229
2,Aug 2025,2025-08-01,2025-08-31,2402,71.981682,0.2186
3,Sep 2025,2025-09-01,2025-09-30,1882,70.616366,0.2109
4,Oct 2025,2025-10-01,2025-10-31,1557,68.593449,0.1706
5,Nov 2025,2025-11-01,2025-11-30,1321,68.130204,0.2293
6,Dec 2025,2025-12-01,2025-12-31,950,63.684211,0.2447
7,Jun 2025 - Dec 2025,2025-06-01,2025-12-31,4330,74.318707,0.2129


### OOP Existing Users Prod, weighted log transformed

In [63]:
print('OOP Existing Users Prod, weighted log transformed')
print("\n" + "=" * 50)

data_periods_dict = {
    'Jun 2025': {'start': '2025-06-01', 'end': '2025-06-30'},
    'Jul 2025': {'start': '2025-07-01', 'end': '2025-07-31'},
    'Aug 2025': {'start': '2025-08-01', 'end': '2025-08-31'},
    'Sep 2025': {'start': '2025-09-01', 'end': '2025-09-30'},
    'Oct 2025': {'start': '2025-10-01', 'end': '2025-10-31'},
    'Nov 2025': {'start': '2025-11-01', 'end': '2025-11-30'},
    'Dec 2025': {'start': '2025-12-01', 'end': '2025-12-31'},
    'Jun 2025 - Dec 2025': {'start': '2025-06-01', 'end': '2025-12-31'}
}

oopExtUsersProdwglogtransformed = calculate_gini_for_table(
    data = dt_prod_batch_calc[dt_prod_batch_calc['osbal_as_of_oop_eligible_date_log'].notna()],
    date_column = 'run_date',
    score_column = 'oop_score_prod',
    target_column = 'oop_target',
    data_periods_dict = data_periods_dict,
    weights_col = 'osbal_as_of_oop_eligible_date_log'
)

oopExtUsersProdwglogtransformed

OOP Existing Users Prod, weighted log transformed

Gini Coefficient Results:
Jun 2025: 0.2734 (Sample size: 1,891)
Jul 2025: 0.2734 (Sample size: 3,035)
Aug 2025: 0.2722 (Sample size: 2,402)
Sep 2025: 0.2776 (Sample size: 1,882)
Oct 2025: 0.264 (Sample size: 1,557)
Nov 2025: 0.2317 (Sample size: 1,321)
Dec 2025: 0.2289 (Sample size: 950)
Jun 2025 - Dec 2025: 0.2523 (Sample size: 13,038)

Summary Table:
             Period Start_Date   End_Date  Sample_Size  Bad Rate  Gini_Coefficient
           Jun 2025 2025-06-01 2025-06-30         1880 76.329787            0.2734
           Jul 2025 2025-07-01 2025-07-31         3035 73.937397            0.2734
           Aug 2025 2025-08-01 2025-08-31         2402 71.981682            0.2722
           Sep 2025 2025-09-01 2025-09-30         1882 70.616366            0.2776
           Oct 2025 2025-10-01 2025-10-31         1557 68.593449            0.2640
           Nov 2025 2025-11-01 2025-11-30         1321 68.130204            0.2317
           De

,Period,Start_Date,End_Date,Sample_Size,Bad Rate,Gini_Coefficient
0,Jun 2025,2025-06-01,2025-06-30,1880,76.329787,0.2734
1,Jul 2025,2025-07-01,2025-07-31,3035,73.937397,0.2734
2,Aug 2025,2025-08-01,2025-08-31,2402,71.981682,0.2722
3,Sep 2025,2025-09-01,2025-09-30,1882,70.616366,0.2776
4,Oct 2025,2025-10-01,2025-10-31,1557,68.593449,0.2640
5,Nov 2025,2025-11-01,2025-11-30,1321,68.130204,0.2317
6,Dec 2025,2025-12-01,2025-12-31,950,63.684211,0.2289
7,Jun 2025 - Dec 2025,2025-06-01,2025-12-31,4330,74.318707,0.2523


### OOP Existing Users Prod, weighted log transformed - 4 months

In [64]:
print('OOP Existing Users Prod, weighted log transformed - 4 months')
print("\n" + "=" * 50)

data_periods_dict = {
    'Jun 2025': {'start': '2025-06-01', 'end': '2025-06-30'},
    'Jul 2025': {'start': '2025-07-01', 'end': '2025-07-31'},
    'Aug 2025': {'start': '2025-08-01', 'end': '2025-08-31'},
    'Sep 2025': {'start': '2025-09-01', 'end': '2025-09-30'},
    'Jun 2025 - Sep 2025': {'start': '2025-06-01', 'end': '2025-09-30'}
}

oopExtUsersProdwglogtransformed4mnt = calculate_gini_for_table(
    data = dt_prod_batch_calc[dt_prod_batch_calc['osbal_as_of_oop_eligible_date_log'].notna()],
    date_column = 'run_date',
    score_column = 'oop_score_prod',
    target_column = 'oop_target',
    data_periods_dict = data_periods_dict,
    weights_col = 'osbal_as_of_oop_eligible_date_log'
)

oopExtUsersProdwglogtransformed4mnt

OOP Existing Users Prod, weighted log transformed - 4 months

Gini Coefficient Results:
Jun 2025: 0.2734 (Sample size: 1,891)
Jul 2025: 0.2734 (Sample size: 3,035)
Aug 2025: 0.2722 (Sample size: 2,402)
Sep 2025: 0.2776 (Sample size: 1,882)
Jun 2025 - Sep 2025: 0.2729 (Sample size: 9,210)

Summary Table:
             Period Start_Date   End_Date  Sample_Size  Bad Rate  Gini_Coefficient
           Jun 2025 2025-06-01 2025-06-30         1880 76.329787            0.2734
           Jul 2025 2025-07-01 2025-07-31         3035 73.937397            0.2734
           Aug 2025 2025-08-01 2025-08-31         2402 71.981682            0.2722
           Sep 2025 2025-09-01 2025-09-30         1882 70.616366            0.2776
Jun 2025 - Sep 2025 2025-06-01 2025-09-30         3886 74.626866            0.2729


,Period,Start_Date,End_Date,Sample_Size,Bad Rate,Gini_Coefficient
0,Jun 2025,2025-06-01,2025-06-30,1880,76.329787,0.2734
1,Jul 2025,2025-07-01,2025-07-31,3035,73.937397,0.2734
2,Aug 2025,2025-08-01,2025-08-31,2402,71.981682,0.2722
3,Sep 2025,2025-09-01,2025-09-30,1882,70.616366,0.2776
4,Jun 2025 - Sep 2025,2025-06-01,2025-09-30,3886,74.626866,0.2729


### OOP Existing Users Prod BS, weighted as-is

In [65]:
print('OOP Existing Users Prod BS, weighted as-is')
print("\n" + "=" * 50)

data_periods_dict = {
    'Jun 2025': {'start': '2025-06-01', 'end': '2025-06-30'},
    'Jul 2025': {'start': '2025-07-01', 'end': '2025-07-31'},
    'Aug 2025': {'start': '2025-08-01', 'end': '2025-08-31'},
    'Sep 2025': {'start': '2025-09-01', 'end': '2025-09-30'},
    'Jun 2025 - Sep 2025': {'start': '2025-06-01', 'end': '2025-09-30'}
}

oopExtUsersProdBSwgasis = calculate_gini_for_table(
    data = dt_prod_batch_calc[(dt_prod_batch_calc['score_oop'].notna()) & (dt_prod_batch_calc['osbal_as_of_oop_eligible_date'].notna())],
    date_column = 'run_date',
    score_column = 'score_oop',
    target_column = 'oop_target',
    data_periods_dict = data_periods_dict,
    weights_col = 'osbal_as_of_oop_eligible_date'
)

oopExtUsersProdBSwgasis

OOP Existing Users Prod BS, weighted as-is

Gini Coefficient Results:
Jun 2025: 0.3174 (Sample size: 1,379)
Jul 2025: 0.2285 (Sample size: 2,112)
Aug 2025: 0.2509 (Sample size: 1,646)
Sep 2025: 0.2592 (Sample size: 1,333)
Jun 2025 - Sep 2025: 0.2572 (Sample size: 6,470)

Summary Table:
             Period Start_Date   End_Date  Sample_Size  Bad Rate  Gini_Coefficient
           Jun 2025 2025-06-01 2025-06-30         1369 76.625274            0.3174
           Jul 2025 2025-07-01 2025-07-31         2112 72.916667            0.2285
           Aug 2025 2025-08-01 2025-08-31         1646 69.987849            0.2509
           Sep 2025 2025-09-01 2025-09-30         1333 68.642161            0.2592
Jun 2025 - Sep 2025 2025-06-01 2025-09-30         2777 74.288801            0.2572


,Period,Start_Date,End_Date,Sample_Size,Bad Rate,Gini_Coefficient
0,Jun 2025,2025-06-01,2025-06-30,1369,76.625274,0.3174
1,Jul 2025,2025-07-01,2025-07-31,2112,72.916667,0.2285
2,Aug 2025,2025-08-01,2025-08-31,1646,69.987849,0.2509
3,Sep 2025,2025-09-01,2025-09-30,1333,68.642161,0.2592
4,Jun 2025 - Sep 2025,2025-06-01,2025-09-30,2777,74.288801,0.2572


### OOP Existing Users Prod BSm weighted as-is - Jun25-Dec25

In [66]:
print('OOP Existing Users Prod BSm weighted as-is - Jun25-Dec25')
print("\n" + "=" * 50)

data_periods_dict = {
    'Jun 2025': {'start': '2025-06-01', 'end': '2025-06-30'},
    'Jul 2025': {'start': '2025-07-01', 'end': '2025-07-31'},
    'Aug 2025': {'start': '2025-08-01', 'end': '2025-08-31'},
    'Sep 2025': {'start': '2025-09-01', 'end': '2025-09-30'},
    'Oct 2025': {'start': '2025-10-01', 'end': '2025-10-31'},
    'Nov 2025': {'start': '2025-11-01', 'end': '2025-11-30'},
    'Dec 2025': {'start': '2025-12-01', 'end': '2025-12-31'},
    'Jun 2025 - Dec 2025': {'start': '2025-06-01', 'end': '2025-12-31'}
}

oopExtUsersProdBSwgasisjun25dec25 = calculate_gini_for_table(
    data = dt_prod_batch_calc[(dt_prod_batch_calc['score_oop'].notna()) & (dt_prod_batch_calc['osbal_as_of_oop_eligible_date'].notna())],
    date_column = 'run_date',
    score_column = 'score_oop',
    target_column = 'oop_target',
    data_periods_dict = data_periods_dict,
    weights_col = 'osbal_as_of_oop_eligible_date'
)

oopExtUsersProdBSwgasisjun25dec25

OOP Existing Users Prod BSm weighted as-is - Jun25-Dec25

Gini Coefficient Results:
Jun 2025: 0.3174 (Sample size: 1,379)
Jul 2025: 0.2285 (Sample size: 2,112)
Aug 2025: 0.2509 (Sample size: 1,646)
Sep 2025: 0.2592 (Sample size: 1,333)
Oct 2025: 0.307 (Sample size: 1,118)
Nov 2025: 0.2705 (Sample size: 1,013)
Dec 2025: 0.2612 (Sample size: 607)
Jun 2025 - Dec 2025: 0.2666 (Sample size: 9,208)

Summary Table:
             Period Start_Date   End_Date  Sample_Size  Bad Rate  Gini_Coefficient
           Jun 2025 2025-06-01 2025-06-30         1369 76.625274            0.3174
           Jul 2025 2025-07-01 2025-07-31         2112 72.916667            0.2285
           Aug 2025 2025-08-01 2025-08-31         1646 69.987849            0.2509
           Sep 2025 2025-09-01 2025-09-30         1333 68.642161            0.2592
           Oct 2025 2025-10-01 2025-10-31         1118 67.441860            0.3070
           Nov 2025 2025-11-01 2025-11-30         1013 67.719645            0.2705
       

,Period,Start_Date,End_Date,Sample_Size,Bad Rate,Gini_Coefficient
0,Jun 2025,2025-06-01,2025-06-30,1369,76.625274,0.3174
1,Jul 2025,2025-07-01,2025-07-31,2112,72.916667,0.2285
2,Aug 2025,2025-08-01,2025-08-31,1646,69.987849,0.2509
3,Sep 2025,2025-09-01,2025-09-30,1333,68.642161,0.2592
4,Oct 2025,2025-10-01,2025-10-31,1118,67.441860,0.3070
5,Nov 2025,2025-11-01,2025-11-30,1013,67.719645,0.2705
6,Dec 2025,2025-12-01,2025-12-31,607,57.495881,0.2612
7,Jun 2025 - Dec 2025,2025-06-01,2025-12-31,3078,74.171540,0.2666


### OOP Existing Users Prod BS, weighted log transformed

In [67]:
print('OOP Existing Users Prod BS, weighted log transformed - 4 months')
print("\n" + "=" * 50)

data_periods_dict = {
    'Jun 2025': {'start': '2025-06-01', 'end': '2025-06-30'},
    'Jul 2025': {'start': '2025-07-01', 'end': '2025-07-31'},
    'Aug 2025': {'start': '2025-08-01', 'end': '2025-08-31'},
    'Sep 2025': {'start': '2025-09-01', 'end': '2025-09-30'},
    'Jun 2025 - Sep 2025': {'start': '2025-06-01', 'end': '2025-09-30'}
}

oopExtUsersProdBSwglogtransformed = calculate_gini_for_table(
    data = dt_prod_batch_calc[(dt_prod_batch_calc['score_oop'].notna()) & (dt_prod_batch_calc['osbal_as_of_oop_eligible_date_log'].notna())],
    date_column = 'run_date',
    score_column = 'score_oop',
    target_column = 'oop_target',
    data_periods_dict = data_periods_dict,
    weights_col = 'osbal_as_of_oop_eligible_date_log'
)

oopExtUsersProdBSwglogtransformed

OOP Existing Users Prod BS, weighted log transformed - 4 months

Gini Coefficient Results:
Jun 2025: 0.3152 (Sample size: 1,379)
Jul 2025: 0.3008 (Sample size: 2,112)
Aug 2025: 0.3211 (Sample size: 1,646)
Sep 2025: 0.3131 (Sample size: 1,333)
Jun 2025 - Sep 2025: 0.3117 (Sample size: 6,470)

Summary Table:
             Period Start_Date   End_Date  Sample_Size  Bad Rate  Gini_Coefficient
           Jun 2025 2025-06-01 2025-06-30         1369 76.625274            0.3152
           Jul 2025 2025-07-01 2025-07-31         2112 72.916667            0.3008
           Aug 2025 2025-08-01 2025-08-31         1646 69.987849            0.3211
           Sep 2025 2025-09-01 2025-09-30         1333 68.642161            0.3131
Jun 2025 - Sep 2025 2025-06-01 2025-09-30         2777 74.288801            0.3117


,Period,Start_Date,End_Date,Sample_Size,Bad Rate,Gini_Coefficient
0,Jun 2025,2025-06-01,2025-06-30,1369,76.625274,0.3152
1,Jul 2025,2025-07-01,2025-07-31,2112,72.916667,0.3008
2,Aug 2025,2025-08-01,2025-08-31,1646,69.987849,0.3211
3,Sep 2025,2025-09-01,2025-09-30,1333,68.642161,0.3131
4,Jun 2025 - Sep 2025,2025-06-01,2025-09-30,2777,74.288801,0.3117


### OOP Existing Users Prod BSm weighted log transformed

In [68]:
print('OOP Existing Users Prod BSm weighted log transformed jun25-dec25')
print("\n" + "=" * 50)

data_periods_dict = {
    'Jun 2025': {'start': '2025-06-01', 'end': '2025-06-30'},
    'Jul 2025': {'start': '2025-07-01', 'end': '2025-07-31'},
    'Aug 2025': {'start': '2025-08-01', 'end': '2025-08-31'},
    'Sep 2025': {'start': '2025-09-01', 'end': '2025-09-30'},
    'Oct 2025': {'start': '2025-10-01', 'end': '2025-10-31'},
    'Nov 2025': {'start': '2025-11-01', 'end': '2025-11-30'},
    'Dec 2025': {'start': '2025-12-01', 'end': '2025-12-31'},
    'Jun 2025 - Dec 2025': {'start': '2025-06-01', 'end': '2025-12-31'}
}

oopExtUsersProdBSwglogtransformed = calculate_gini_for_table(
    data = dt_prod_batch_calc[(dt_prod_batch_calc['score_oop'].notna()) & (dt_prod_batch_calc['osbal_as_of_oop_eligible_date_log'].notna())],
    date_column = 'run_date',
    score_column = 'score_oop',
    target_column = 'oop_target',
    data_periods_dict = data_periods_dict,
    weights_col = 'osbal_as_of_oop_eligible_date_log'
)

oopExtUsersProdBSwglogtransformed

OOP Existing Users Prod BSm weighted log transformed jun25-dec25

Gini Coefficient Results:
Jun 2025: 0.3152 (Sample size: 1,379)
Jul 2025: 0.3008 (Sample size: 2,112)
Aug 2025: 0.3211 (Sample size: 1,646)
Sep 2025: 0.3131 (Sample size: 1,333)
Oct 2025: 0.3384 (Sample size: 1,118)
Nov 2025: 0.3177 (Sample size: 1,013)
Dec 2025: 0.348 (Sample size: 607)
Jun 2025 - Dec 2025: 0.3192 (Sample size: 9,208)

Summary Table:
             Period Start_Date   End_Date  Sample_Size  Bad Rate  Gini_Coefficient
           Jun 2025 2025-06-01 2025-06-30         1369 76.625274            0.3152
           Jul 2025 2025-07-01 2025-07-31         2112 72.916667            0.3008
           Aug 2025 2025-08-01 2025-08-31         1646 69.987849            0.3211
           Sep 2025 2025-09-01 2025-09-30         1333 68.642161            0.3131
           Oct 2025 2025-10-01 2025-10-31         1118 67.441860            0.3384
           Nov 2025 2025-11-01 2025-11-30         1013 67.719645            0.3177

,Period,Start_Date,End_Date,Sample_Size,Bad Rate,Gini_Coefficient
0,Jun 2025,2025-06-01,2025-06-30,1369,76.625274,0.3152
1,Jul 2025,2025-07-01,2025-07-31,2112,72.916667,0.3008
2,Aug 2025,2025-08-01,2025-08-31,1646,69.987849,0.3211
3,Sep 2025,2025-09-01,2025-09-30,1333,68.642161,0.3131
4,Oct 2025,2025-10-01,2025-10-31,1118,67.441860,0.3384
5,Nov 2025,2025-11-01,2025-11-30,1013,67.719645,0.3177
6,Dec 2025,2025-12-01,2025-12-31,607,57.495881,0.3480
7,Jun 2025 - Dec 2025,2025-06-01,2025-12-31,3078,74.171540,0.3192


## bs new

In [69]:
dt_bs_oop_new.shape

(169297, 7)

In [70]:
dt_bs_oop_new['ee_customer_id'].nunique()

169297

In [71]:
dt_bs_oop_new = dt_bs_oop_new.merge(
    dt_raw[['ee_customer_id','ee_onboarding_date']].drop_duplicates(),
    how='left',
    on='ee_customer_id'
).merge(
    dt_oop_targets,
    how='left',
    on='ee_customer_id'
)

dt_bs_oop_new.sample(5)

,ee_customer_id,calc_date,target_dev,score_oop,osbal_as_of_resignation_date,osbal_as_of_oop_eligible_date,osbal_as_of_current_date,ee_onboarding_date,oop_target
21712,1141397,2024-05-01,0.0,0.538868,9379.935,9379.935,9379.935,2024-04-30 10:40:01,0
26048,1142954,2024-05-01,0.0,0.566176,24662.852,24662.852,24662.852,2024-04-30 11:25:13,0
16254,1182790,2024-05-01,0.0,0.506191,12669.713,12669.713,12669.713,2024-04-22 13:10:40,0
23939,1573508,2025-11-01,NaN,0.552877,29632.466,29632.466,29632.466,2025-10-07 15:01:46,0
153411,1159346,2025-03-01,NaN,0.620011,NaN,NaN,19444.845,2025-02-06 16:52:44,<NA>


In [72]:
dt_bs_oop_new['ee_onboarding_date'] = pd.to_datetime(dt_bs_oop_new['ee_onboarding_date']).dt.normalize()
dt_bs_oop_new['onboarding_date_ym'] = dt_bs_oop_new['ee_onboarding_date'].dt.year*100 + dt_bs_oop_new['ee_onboarding_date'].dt.month

In [73]:
dt_bs_oop_new.sample(5)

,ee_customer_id,calc_date,target_dev,score_oop,osbal_as_of_resignation_date,osbal_as_of_oop_eligible_date,osbal_as_of_current_date,ee_onboarding_date,oop_target,onboarding_date_ym
20086,1049619,2023-06-01,0.0,0.528843,0.002,978.352,978.352,2023-05-24,0,202305
56273,1681967,2026-02-01,NaN,0.423986,NaN,NaN,22895.242,2026-01-13,<NA>,202601
168761,1551851,2025-10-01,NaN,0.753986,NaN,NaN,0.000,2025-09-19,<NA>,202509
68992,1229948,2024-08-01,NaN,0.471732,NaN,NaN,0.000,2024-07-12,<NA>,202407
52242,1244472,2024-09-01,NaN,0.401516,NaN,NaN,28521.132,2024-08-01,<NA>,202408


In [74]:
dt_bs_oop_new_calc = dt_bs_oop_new.dropna(subset=['oop_target'])

print(f"The shape of the dataframe dt_bs_oop_new is:\t {dt_bs_oop_new.shape} ")
print(f"The shape of the dataframe after dropping na from oop_target is:\t {dt_bs_oop_new_calc.shape}")

The shape of the dataframe dt_bs_oop_new is:	 (169297, 10) 
The shape of the dataframe after dropping na from oop_target is:	 (31364, 10)


### OOP New Users BS

In [75]:
print('OOP New Users BS')
print("\n" + "=" * 50)

data_periods_dict = {
    'Jun 2025': {'start': '2025-06-01', 'end': '2025-06-30'},
    'Jul 2025': {'start': '2025-07-01', 'end': '2025-07-31'},
    'Aug 2025': {'start': '2025-08-01', 'end': '2025-08-31'},
    'Sep 2025': {'start': '2025-09-01', 'end': '2025-09-30'},
    'Jun 2025 - Sep 2025': {'start': '2025-06-01', 'end': '2025-09-30'}
}

oopNewUsersBS = calculate_gini_for_table(
    data = dt_bs_oop_new_calc,
    date_column = 'ee_onboarding_date',
    score_column = 'score_oop',
    target_column = 'oop_target',
    data_periods_dict = data_periods_dict
)

oopNewUsersBS

OOP New Users BS

Gini Coefficient Results:
Jun 2025: 0.0759 (Sample size: 464)
Jul 2025: 0.0866 (Sample size: 441)
Aug 2025: 0.0584 (Sample size: 292)
Sep 2025: -0.0155 (Sample size: 631)
Jun 2025 - Sep 2025: 0.044 (Sample size: 1,828)

Summary Table:
             Period Start_Date   End_Date  Sample_Size  Bad Rate  Gini_Coefficient
           Jun 2025 2025-06-01 2025-06-30          464 68.965517            0.0759
           Jul 2025 2025-07-01 2025-07-31          441 63.492063            0.0866
           Aug 2025 2025-08-01 2025-08-31          292 64.383562            0.0584
           Sep 2025 2025-09-01 2025-09-30          631 68.621236           -0.0155
Jun 2025 - Sep 2025 2025-06-01 2025-09-30         1828 66.794311            0.0440


,Period,Start_Date,End_Date,Sample_Size,Bad Rate,Gini_Coefficient
0,Jun 2025,2025-06-01,2025-06-30,464,68.965517,0.0759
1,Jul 2025,2025-07-01,2025-07-31,441,63.492063,0.0866
2,Aug 2025,2025-08-01,2025-08-31,292,64.383562,0.0584
3,Sep 2025,2025-09-01,2025-09-30,631,68.621236,-0.0155
4,Jun 2025 - Sep 2025,2025-06-01,2025-09-30,1828,66.794311,0.0440


In [76]:
dt_bs_oop_new_calc['osbal_as_of_oop_eligible_date_log'] = np.log1p(dt_bs_oop_new_calc['osbal_as_of_oop_eligible_date'])

C:\Users\Dwaipayan\AppData\Local\Temp\ipykernel_21304\3448610563.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dt_bs_oop_new_calc['osbal_as_of_oop_eligible_date_log'] = np.log1p(dt_bs_oop_new_calc['osbal_as_of_oop_eligible_date'])


### OOP New Users BS, weighted as-is 4 months

In [77]:
print('OOP New Users BS, weighted as-is')
print("\n" + "=" * 50)

data_periods_dict = {
    'Jun 2025': {'start': '2025-06-01', 'end': '2025-06-30'},
    'Jul 2025': {'start': '2025-07-01', 'end': '2025-07-31'},
    'Aug 2025': {'start': '2025-08-01', 'end': '2025-08-31'},
    'Sep 2025': {'start': '2025-09-01', 'end': '2025-09-30'},
    'Jun 2025 - Sep 2025': {'start': '2025-06-01', 'end': '2025-09-30'}
}

oopNewUsersBSwgasis4mnt = calculate_gini_for_table(
    data = dt_bs_oop_new_calc[dt_bs_oop_new_calc['osbal_as_of_oop_eligible_date'].notna()],
    date_column = 'ee_onboarding_date',
    score_column = 'score_oop',
    target_column = 'oop_target',
    data_periods_dict = data_periods_dict,
    weights_col = 'osbal_as_of_oop_eligible_date'
)

oopNewUsersBSwgasis4mnt

OOP New Users BS, weighted as-is

Gini Coefficient Results:
Jun 2025: 0.0023 (Sample size: 440)
Jul 2025: 0.0668 (Sample size: 399)
Aug 2025: 0.0864 (Sample size: 271)
Sep 2025: -0.0414 (Sample size: 606)
Jun 2025 - Sep 2025: 0.0097 (Sample size: 1,716)

Summary Table:
             Period Start_Date   End_Date  Sample_Size  Bad Rate  Gini_Coefficient
           Jun 2025 2025-06-01 2025-06-30          440 70.000000            0.0023
           Jul 2025 2025-07-01 2025-07-31          399 65.664160            0.0668
           Aug 2025 2025-08-01 2025-08-31          271 67.158672            0.0864
           Sep 2025 2025-09-01 2025-09-30          606 70.132013           -0.0414
Jun 2025 - Sep 2025 2025-06-01 2025-09-30         1716 68.589744            0.0097


,Period,Start_Date,End_Date,Sample_Size,Bad Rate,Gini_Coefficient
0,Jun 2025,2025-06-01,2025-06-30,440,70.000000,0.0023
1,Jul 2025,2025-07-01,2025-07-31,399,65.664160,0.0668
2,Aug 2025,2025-08-01,2025-08-31,271,67.158672,0.0864
3,Sep 2025,2025-09-01,2025-09-30,606,70.132013,-0.0414
4,Jun 2025 - Sep 2025,2025-06-01,2025-09-30,1716,68.589744,0.0097


### OOP New Users BS, weighted log - 4 months

In [78]:
print('OOP New Users BS, weighted log 4 months')
print("\n" + "=" * 50)

data_periods_dict = {
    'Jun 2025': {'start': '2025-06-01', 'end': '2025-06-30'},
    'Jul 2025': {'start': '2025-07-01', 'end': '2025-07-31'},
    'Aug 2025': {'start': '2025-08-01', 'end': '2025-08-31'},
    'Sep 2025': {'start': '2025-09-01', 'end': '2025-09-30'},
    'Jun 2025 - Sep 2025': {'start': '2025-06-01', 'end': '2025-09-30'}
}

oopNewUsersBSwglog4mnt = calculate_gini_for_table(
    data = dt_bs_oop_new_calc[dt_bs_oop_new_calc['osbal_as_of_oop_eligible_date_log'].notna()],
    date_column = 'ee_onboarding_date',
    score_column = 'score_oop',
    target_column = 'oop_target',
    data_periods_dict = data_periods_dict,
    weights_col = 'osbal_as_of_oop_eligible_date_log'
)

oopNewUsersBSwglog4mnt

OOP New Users BS, weighted log 4 months

Gini Coefficient Results:
Jun 2025: 0.0579 (Sample size: 440)
Jul 2025: 0.0896 (Sample size: 399)
Aug 2025: 0.0515 (Sample size: 271)
Sep 2025: -0.0391 (Sample size: 606)
Jun 2025 - Sep 2025: 0.0305 (Sample size: 1,716)

Summary Table:
             Period Start_Date   End_Date  Sample_Size  Bad Rate  Gini_Coefficient
           Jun 2025 2025-06-01 2025-06-30          440 70.000000            0.0579
           Jul 2025 2025-07-01 2025-07-31          399 65.664160            0.0896
           Aug 2025 2025-08-01 2025-08-31          271 67.158672            0.0515
           Sep 2025 2025-09-01 2025-09-30          606 70.132013           -0.0391
Jun 2025 - Sep 2025 2025-06-01 2025-09-30         1716 68.589744            0.0305


,Period,Start_Date,End_Date,Sample_Size,Bad Rate,Gini_Coefficient
0,Jun 2025,2025-06-01,2025-06-30,440,70.000000,0.0579
1,Jul 2025,2025-07-01,2025-07-31,399,65.664160,0.0896
2,Aug 2025,2025-08-01,2025-08-31,271,67.158672,0.0515
3,Sep 2025,2025-09-01,2025-09-30,606,70.132013,-0.0391
4,Jun 2025 - Sep 2025,2025-06-01,2025-09-30,1716,68.589744,0.0305


### OOP New Users BS, weighted as-is

In [79]:
print('OOP New Users BS, weighted as-is')
print("\n" + "=" * 50)

data_periods_dict = {
    'Jun 2025': {'start': '2025-06-01', 'end': '2025-06-30'},
    'Jul 2025': {'start': '2025-07-01', 'end': '2025-07-31'},
    'Aug 2025': {'start': '2025-08-01', 'end': '2025-08-31'},
    'Sep 2025': {'start': '2025-09-01', 'end': '2025-09-30'},
    'Oct 2025': {'start': '2025-10-01', 'end': '2025-10-31'},
    'Nov 2025': {'start': '2025-11-01', 'end': '2025-11-30'},
    'Dec 2025': {'start': '2025-12-01', 'end': '2025-12-31'},
    'Jun 2025 - Dec 2025': {'start': '2025-06-01', 'end': '2025-12-31'}
}

oopNewUsersBSwgasis = calculate_gini_for_table(
    data = dt_bs_oop_new_calc[dt_bs_oop_new_calc['osbal_as_of_oop_eligible_date'].notna()],
    date_column = 'ee_onboarding_date',
    score_column = 'score_oop',
    target_column = 'oop_target',
    data_periods_dict = data_periods_dict,
    weights_col = 'osbal_as_of_oop_eligible_date'
)

oopNewUsersBSwgasis

OOP New Users BS, weighted as-is

Gini Coefficient Results:
Jun 2025: 0.0023 (Sample size: 440)
Jul 2025: 0.0668 (Sample size: 399)
Aug 2025: 0.0864 (Sample size: 271)
Sep 2025: -0.0414 (Sample size: 606)
Oct 2025: 0.2513 (Sample size: 930)
Nov 2025: 0.2402 (Sample size: 444)
Dec 2025: 0.1203 (Sample size: 123)
Jun 2025 - Dec 2025: 0.1402 (Sample size: 3,213)

Summary Table:
             Period Start_Date   End_Date  Sample_Size  Bad Rate  Gini_Coefficient
           Jun 2025 2025-06-01 2025-06-30          440 70.000000            0.0023
           Jul 2025 2025-07-01 2025-07-31          399 65.664160            0.0668
           Aug 2025 2025-08-01 2025-08-31          271 67.158672            0.0864
           Sep 2025 2025-09-01 2025-09-30          606 70.132013           -0.0414
           Oct 2025 2025-10-01 2025-10-31          930 76.881720            0.2513
           Nov 2025 2025-11-01 2025-11-30          444 72.747748            0.2402
           Dec 2025 2025-12-01 2025-12-31

,Period,Start_Date,End_Date,Sample_Size,Bad Rate,Gini_Coefficient
0,Jun 2025,2025-06-01,2025-06-30,440,70.000000,0.0023
1,Jul 2025,2025-07-01,2025-07-31,399,65.664160,0.0668
2,Aug 2025,2025-08-01,2025-08-31,271,67.158672,0.0864
3,Sep 2025,2025-09-01,2025-09-30,606,70.132013,-0.0414
4,Oct 2025,2025-10-01,2025-10-31,930,76.881720,0.2513
5,Nov 2025,2025-11-01,2025-11-30,444,72.747748,0.2402
6,Dec 2025,2025-12-01,2025-12-31,123,53.658537,0.1203
7,Jun 2025 - Dec 2025,2025-06-01,2025-12-31,3213,70.992842,0.1402


### OOP New Users BS, weighted log

In [80]:
print('OOP New Users BS, weighted log')
print("\n" + "=" * 50)

data_periods_dict = {
    'Jun 2025': {'start': '2025-06-01', 'end': '2025-06-30'},
    'Jul 2025': {'start': '2025-07-01', 'end': '2025-07-31'},
    'Aug 2025': {'start': '2025-08-01', 'end': '2025-08-31'},
    'Sep 2025': {'start': '2025-09-01', 'end': '2025-09-30'},
    'Oct 2025': {'start': '2025-10-01', 'end': '2025-10-31'},
    'Nov 2025': {'start': '2025-11-01', 'end': '2025-11-30'},
    'Dec 2025': {'start': '2025-12-01', 'end': '2025-12-31'},
    'Jun 2025 - Dec 2025': {'start': '2025-06-01', 'end': '2025-12-31'}
}

oopNewUsersBSwglog = calculate_gini_for_table(
    data = dt_bs_oop_new_calc[dt_bs_oop_new_calc['osbal_as_of_oop_eligible_date_log'].notna()],
    date_column = 'ee_onboarding_date',
    score_column = 'score_oop',
    target_column = 'oop_target',
    data_periods_dict = data_periods_dict,
    weights_col = 'osbal_as_of_oop_eligible_date_log'
)

oopNewUsersBSwglog

OOP New Users BS, weighted log

Gini Coefficient Results:
Jun 2025: 0.0579 (Sample size: 440)
Jul 2025: 0.0896 (Sample size: 399)
Aug 2025: 0.0515 (Sample size: 271)
Sep 2025: -0.0391 (Sample size: 606)
Oct 2025: 0.2725 (Sample size: 930)
Nov 2025: 0.2945 (Sample size: 444)
Dec 2025: 0.1253 (Sample size: 123)
Jun 2025 - Dec 2025: 0.1466 (Sample size: 3,213)

Summary Table:
             Period Start_Date   End_Date  Sample_Size  Bad Rate  Gini_Coefficient
           Jun 2025 2025-06-01 2025-06-30          440 70.000000            0.0579
           Jul 2025 2025-07-01 2025-07-31          399 65.664160            0.0896
           Aug 2025 2025-08-01 2025-08-31          271 67.158672            0.0515
           Sep 2025 2025-09-01 2025-09-30          606 70.132013           -0.0391
           Oct 2025 2025-10-01 2025-10-31          930 76.881720            0.2725
           Nov 2025 2025-11-01 2025-11-30          444 72.747748            0.2945
           Dec 2025 2025-12-01 2025-12-31  

,Period,Start_Date,End_Date,Sample_Size,Bad Rate,Gini_Coefficient
0,Jun 2025,2025-06-01,2025-06-30,440,70.000000,0.0579
1,Jul 2025,2025-07-01,2025-07-31,399,65.664160,0.0896
2,Aug 2025,2025-08-01,2025-08-31,271,67.158672,0.0515
3,Sep 2025,2025-09-01,2025-09-30,606,70.132013,-0.0391
4,Oct 2025,2025-10-01,2025-10-31,930,76.881720,0.2725
5,Nov 2025,2025-11-01,2025-11-30,444,72.747748,0.2945
6,Dec 2025,2025-12-01,2025-12-31,123,53.658537,0.1253
7,Jun 2025 - Dec 2025,2025-06-01,2025-12-31,3213,70.992842,0.1466


## bs existing

In [81]:
dt_bs_oop_ex.shape

(1531733, 6)

In [82]:
dt_bs_oop_ex['ee_customer_id'].nunique()

144497

In [83]:
dt_bs_oop_ex = dt_bs_oop_ex.merge(
    dt_raw[['ee_customer_id','ee_onboarding_date']].drop_duplicates(),
    how='left',
    on='ee_customer_id'
).merge(
    dt_oop_targets,
    how='left',
    on='ee_customer_id'
).merge(
    dt_bs_oop_new[['ee_customer_id', 'osbal_as_of_resignation_date', 'osbal_as_of_oop_eligible_date', 'osbal_as_of_current_date']],
    how='left',
    on='ee_customer_id'
)

dt_bs_oop_ex.sample(5)

,ee_customer_id,calc_date,target_dev,score_oop,calc_date_correct,calc_date_ym,ee_onboarding_date,oop_target,osbal_as_of_resignation_date,osbal_as_of_oop_eligible_date,osbal_as_of_current_date
899853,1428032,2025-11-01,NaN,0.549598,2025-10-01,202510,2025-05-10 09:41:24,<NA>,NaN,NaN,12725.930
947034,613602,2023-01-01,0.0,0.600975,2022-12-01,202212,2021-10-30 01:34:49,0,NaN,NaN,NaN
1225446,1195852,2025-02-01,NaN,0.403368,2025-01-01,202501,2024-05-10 19:44:42,<NA>,NaN,NaN,6769.743
325818,1259152,2025-01-01,0.0,0.640802,2024-12-01,202412,2024-08-27 17:14:44,0,4541.660,4541.660,4541.660
1121205,1149576,2024-06-01,NaN,0.456610,2024-05-01,202405,2024-01-23 19:08:18,1,10618.038,10618.038,14387.847


In [84]:
dt_bs_oop_ex.shape

(1531733, 11)

In [85]:
dt_bs_oop_ex_calc = dt_bs_oop_ex.dropna(subset=['oop_target'])
dt_bs_oop_ex_calc.shape

(191419, 11)

### OOP Existing Users BS

In [86]:
print('OOP Existing Users BS')
print("\n" + "=" * 50)

data_periods_dict = {
    'Jun 2025': {'start': '2025-06-01', 'end': '2025-06-30'},
    'Jul 2025': {'start': '2025-07-01', 'end': '2025-07-31'},
    'Aug 2025': {'start': '2025-08-01', 'end': '2025-08-31'},
    'Sep 2025': {'start': '2025-09-01', 'end': '2025-09-30'},
    'Jun 2025 - Sep 2025': {'start': '2025-06-01', 'end': '2025-09-30'}
}

oopExtusersBS = calculate_gini_for_table(
    data = dt_bs_oop_ex_calc,
    date_column = 'calc_date_correct',
    score_column = 'score_oop',
    target_column = 'oop_target',
    data_periods_dict = data_periods_dict
)
oopExtusersBS

OOP Existing Users BS

Gini Coefficient Results:
Jun 2025: 0.2809 (Sample size: 4,767)
Jul 2025: 0.2761 (Sample size: 5,374)
Aug 2025: 0.2906 (Sample size: 4,651)
Sep 2025: 0.2842 (Sample size: 3,823)
Jun 2025 - Sep 2025: 0.2889 (Sample size: 18,615)

Summary Table:
             Period Start_Date   End_Date  Sample_Size  Bad Rate  Gini_Coefficient
           Jun 2025 2025-06-01 2025-06-30         4767 77.407174            0.2809
           Jul 2025 2025-07-01 2025-07-31         5374 73.427614            0.2761
           Aug 2025 2025-08-01 2025-08-31         4651 71.253494            0.2906
           Sep 2025 2025-09-01 2025-09-30         3823 68.924928            0.2842
Jun 2025 - Sep 2025 2025-06-01 2025-09-30         7064 74.009060            0.2889


,Period,Start_Date,End_Date,Sample_Size,Bad Rate,Gini_Coefficient
0,Jun 2025,2025-06-01,2025-06-30,4767,77.407174,0.2809
1,Jul 2025,2025-07-01,2025-07-31,5374,73.427614,0.2761
2,Aug 2025,2025-08-01,2025-08-31,4651,71.253494,0.2906
3,Sep 2025,2025-09-01,2025-09-30,3823,68.924928,0.2842
4,Jun 2025 - Sep 2025,2025-06-01,2025-09-30,7064,74.009060,0.2889


In [87]:
dt_bs_oop_ex_calc['osbal_as_of_oop_eligible_date_log'] = np.log1p(dt_bs_oop_ex_calc['osbal_as_of_oop_eligible_date'])

C:\Users\Dwaipayan\AppData\Local\Temp\ipykernel_21304\3634989807.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dt_bs_oop_ex_calc['osbal_as_of_oop_eligible_date_log'] = np.log1p(dt_bs_oop_ex_calc['osbal_as_of_oop_eligible_date'])


### OOP Existing Users BS, weighted as-is 4 months

In [88]:
print('OOP Existing Users BS, weighted as-is')
print("\n" + "=" * 50)

data_periods_dict = {
    'Jun 2025': {'start': '2025-06-01', 'end': '2025-06-30'},
    'Jul 2025': {'start': '2025-07-01', 'end': '2025-07-31'},
    'Aug 2025': {'start': '2025-08-01', 'end': '2025-08-31'},
    'Sep 2025': {'start': '2025-09-01', 'end': '2025-09-30'},
    'Jun 2025 - Sep 2025': {'start': '2025-06-01', 'end': '2025-09-30'}
}

oopExtUsersBSwgasis4mnt = calculate_gini_for_table(
    data = dt_bs_oop_ex_calc[dt_bs_oop_ex_calc['osbal_as_of_oop_eligible_date'].notna()],
    date_column = 'calc_date_correct',
    score_column = 'score_oop',
    target_column = 'oop_target',
    data_periods_dict = data_periods_dict,
    weights_col = 'osbal_as_of_oop_eligible_date'
)

oopExtUsersBSwgasis4mnt

OOP Existing Users BS, weighted as-is

Gini Coefficient Results:
Jun 2025: 0.2687 (Sample size: 3,884)
Jul 2025: 0.2842 (Sample size: 4,525)
Aug 2025: 0.317 (Sample size: 3,914)
Sep 2025: 0.2981 (Sample size: 3,283)
Jun 2025 - Sep 2025: 0.2992 (Sample size: 15,606)

Summary Table:
             Period Start_Date   End_Date  Sample_Size  Bad Rate  Gini_Coefficient
           Jun 2025 2025-06-01 2025-06-30         3884 77.960865            0.2687
           Jul 2025 2025-07-01 2025-07-31         4525 73.900552            0.2842
           Aug 2025 2025-08-01 2025-08-31         3914 71.972407            0.3170
           Sep 2025 2025-09-01 2025-09-30         3283 69.753274            0.2981
Jun 2025 - Sep 2025 2025-06-01 2025-09-30         6059 74.434725            0.2992


,Period,Start_Date,End_Date,Sample_Size,Bad Rate,Gini_Coefficient
0,Jun 2025,2025-06-01,2025-06-30,3884,77.960865,0.2687
1,Jul 2025,2025-07-01,2025-07-31,4525,73.900552,0.2842
2,Aug 2025,2025-08-01,2025-08-31,3914,71.972407,0.3170
3,Sep 2025,2025-09-01,2025-09-30,3283,69.753274,0.2981
4,Jun 2025 - Sep 2025,2025-06-01,2025-09-30,6059,74.434725,0.2992


### OOP Existing Users BS, weighted log transform 4 months

In [89]:
print('OOP Existing Users BS, weighted log transform')
print("\n" + "=" * 50)

data_periods_dict = {
    'Jun 2025': {'start': '2025-06-01', 'end': '2025-06-30'},
    'Jul 2025': {'start': '2025-07-01', 'end': '2025-07-31'},
    'Aug 2025': {'start': '2025-08-01', 'end': '2025-08-31'},
    'Sep 2025': {'start': '2025-09-01', 'end': '2025-09-30'},
    'Jun 2025 - Sep 2025': {'start': '2025-06-01', 'end': '2025-09-30'}
}

oopExtUsersBSwglog4mnt = calculate_gini_for_table(
    data = dt_bs_oop_ex_calc[dt_bs_oop_ex_calc['osbal_as_of_oop_eligible_date_log'].notna()],
    date_column = 'calc_date_correct',
    score_column = 'score_oop',
    target_column = 'oop_target',
    data_periods_dict = data_periods_dict,
    weights_col = 'osbal_as_of_oop_eligible_date_log'
)

oopExtUsersBSwglog4mnt

OOP Existing Users BS, weighted log transform

Gini Coefficient Results:
Jun 2025: 0.2927 (Sample size: 3,884)
Jul 2025: 0.2826 (Sample size: 4,525)
Aug 2025: 0.3018 (Sample size: 3,914)
Sep 2025: 0.2853 (Sample size: 3,283)
Jun 2025 - Sep 2025: 0.2977 (Sample size: 15,606)

Summary Table:
             Period Start_Date   End_Date  Sample_Size  Bad Rate  Gini_Coefficient
           Jun 2025 2025-06-01 2025-06-30         3884 77.960865            0.2927
           Jul 2025 2025-07-01 2025-07-31         4525 73.900552            0.2826
           Aug 2025 2025-08-01 2025-08-31         3914 71.972407            0.3018
           Sep 2025 2025-09-01 2025-09-30         3283 69.753274            0.2853
Jun 2025 - Sep 2025 2025-06-01 2025-09-30         6059 74.434725            0.2977


,Period,Start_Date,End_Date,Sample_Size,Bad Rate,Gini_Coefficient
0,Jun 2025,2025-06-01,2025-06-30,3884,77.960865,0.2927
1,Jul 2025,2025-07-01,2025-07-31,4525,73.900552,0.2826
2,Aug 2025,2025-08-01,2025-08-31,3914,71.972407,0.3018
3,Sep 2025,2025-09-01,2025-09-30,3283,69.753274,0.2853
4,Jun 2025 - Sep 2025,2025-06-01,2025-09-30,6059,74.434725,0.2977


### OOP Existing Users BS, weighted as-is jun25-dec25

In [90]:
print('OOP Existing Users BS, weighted as-is')
print("\n" + "=" * 50)

data_periods_dict = {
    'Jun 2025': {'start': '2025-06-01', 'end': '2025-06-30'},
    'Jul 2025': {'start': '2025-07-01', 'end': '2025-07-31'},
    'Aug 2025': {'start': '2025-08-01', 'end': '2025-08-31'},
    'Sep 2025': {'start': '2025-09-01', 'end': '2025-09-30'},
    'Oct 2025': {'start': '2025-10-01', 'end': '2025-10-31'},
    'Nov 2025': {'start': '2025-11-01', 'end': '2025-11-30'},
    'Dec 2025': {'start': '2025-12-01', 'end': '2025-12-31'},
    'Jun 2025 - Dec 2025': {'start': '2025-06-01', 'end': '2025-12-31'}
}

oopExtUsersBSwgasisjun25dec25 = calculate_gini_for_table(
    data = dt_bs_oop_ex_calc[dt_bs_oop_ex_calc['osbal_as_of_oop_eligible_date'].notna()],
    date_column = 'calc_date_correct',
    score_column = 'score_oop',
    target_column = 'oop_target',
    data_periods_dict = data_periods_dict,
    weights_col = 'osbal_as_of_oop_eligible_date'
)

oopExtUsersBSwgasisjun25dec25

OOP Existing Users BS, weighted as-is

Gini Coefficient Results:
Jun 2025: 0.2687 (Sample size: 3,884)
Jul 2025: 0.2842 (Sample size: 4,525)
Aug 2025: 0.317 (Sample size: 3,914)
Sep 2025: 0.2981 (Sample size: 3,283)
Oct 2025: 0.2735 (Sample size: 2,864)
Nov 2025: 0.2955 (Sample size: 2,534)
Dec 2025: 0.3147 (Sample size: 1,821)
Jun 2025 - Dec 2025: 0.2967 (Sample size: 22,825)

Summary Table:
             Period Start_Date   End_Date  Sample_Size  Bad Rate  Gini_Coefficient
           Jun 2025 2025-06-01 2025-06-30         3884 77.960865            0.2687
           Jul 2025 2025-07-01 2025-07-31         4525 73.900552            0.2842
           Aug 2025 2025-08-01 2025-08-31         3914 71.972407            0.3170
           Sep 2025 2025-09-01 2025-09-30         3283 69.753274            0.2981
           Oct 2025 2025-10-01 2025-10-31         2864 67.493017            0.2735
           Nov 2025 2025-11-01 2025-11-30         2534 66.337806            0.2955
           Dec 2025 202

,Period,Start_Date,End_Date,Sample_Size,Bad Rate,Gini_Coefficient
0,Jun 2025,2025-06-01,2025-06-30,3884,77.960865,0.2687
1,Jul 2025,2025-07-01,2025-07-31,4525,73.900552,0.2842
2,Aug 2025,2025-08-01,2025-08-31,3914,71.972407,0.3170
3,Sep 2025,2025-09-01,2025-09-30,3283,69.753274,0.2981
4,Oct 2025,2025-10-01,2025-10-31,2864,67.493017,0.2735
5,Nov 2025,2025-11-01,2025-11-30,2534,66.337806,0.2955
6,Dec 2025,2025-12-01,2025-12-31,1821,60.076881,0.3147
7,Jun 2025 - Dec 2025,2025-06-01,2025-12-31,6739,73.215611,0.2967


### OOP Existing Users BS, weighted log transform jun25-dec25

In [91]:
print('OOP Existing Users BS, weighted log transform')
print("\n" + "=" * 50)

data_periods_dict = {
    'Jun 2025': {'start': '2025-06-01', 'end': '2025-06-30'},
    'Jul 2025': {'start': '2025-07-01', 'end': '2025-07-31'},
    'Aug 2025': {'start': '2025-08-01', 'end': '2025-08-31'},
    'Sep 2025': {'start': '2025-09-01', 'end': '2025-09-30'},
    'Oct 2025': {'start': '2025-10-01', 'end': '2025-10-31'},
    'Nov 2025': {'start': '2025-11-01', 'end': '2025-11-30'},
    'Dec 2025': {'start': '2025-12-01', 'end': '2025-12-31'},
    'Jun 2025 - Dec 2025': {'start': '2025-06-01', 'end': '2025-12-31'}
}

oopExtUsersBSwglogjun25dec25 = calculate_gini_for_table(
    data = dt_bs_oop_ex_calc[dt_bs_oop_ex_calc['osbal_as_of_oop_eligible_date_log'].notna()],
    date_column = 'calc_date_correct',
    score_column = 'score_oop',
    target_column = 'oop_target',
    data_periods_dict = data_periods_dict,
    weights_col = 'osbal_as_of_oop_eligible_date_log'
)

oopExtUsersBSwglogjun25dec25

OOP Existing Users BS, weighted log transform

Gini Coefficient Results:
Jun 2025: 0.2927 (Sample size: 3,884)
Jul 2025: 0.2826 (Sample size: 4,525)
Aug 2025: 0.3018 (Sample size: 3,914)
Sep 2025: 0.2853 (Sample size: 3,283)
Oct 2025: 0.2714 (Sample size: 2,864)
Nov 2025: 0.3008 (Sample size: 2,534)
Dec 2025: 0.3376 (Sample size: 1,821)
Jun 2025 - Dec 2025: 0.2996 (Sample size: 22,825)

Summary Table:
             Period Start_Date   End_Date  Sample_Size  Bad Rate  Gini_Coefficient
           Jun 2025 2025-06-01 2025-06-30         3884 77.960865            0.2927
           Jul 2025 2025-07-01 2025-07-31         4525 73.900552            0.2826
           Aug 2025 2025-08-01 2025-08-31         3914 71.972407            0.3018
           Sep 2025 2025-09-01 2025-09-30         3283 69.753274            0.2853
           Oct 2025 2025-10-01 2025-10-31         2864 67.493017            0.2714
           Nov 2025 2025-11-01 2025-11-30         2534 66.337806            0.3008
           Dec

,Period,Start_Date,End_Date,Sample_Size,Bad Rate,Gini_Coefficient
0,Jun 2025,2025-06-01,2025-06-30,3884,77.960865,0.2927
1,Jul 2025,2025-07-01,2025-07-31,4525,73.900552,0.2826
2,Aug 2025,2025-08-01,2025-08-31,3914,71.972407,0.3018
3,Sep 2025,2025-09-01,2025-09-30,3283,69.753274,0.2853
4,Oct 2025,2025-10-01,2025-10-31,2864,67.493017,0.2714
5,Nov 2025,2025-11-01,2025-11-30,2534,66.337806,0.3008
6,Dec 2025,2025-12-01,2025-12-31,1821,60.076881,0.3376
7,Jun 2025 - Dec 2025,2025-06-01,2025-12-31,6739,73.215611,0.2996


# Creating Slide 1

In [92]:
# Calculate averages from the assigned results
avg_gini_new = oopNewUsersProdBS[oopNewUsersProdBS['Period'].isin(['Jul 2025', 'Aug 2025'])]['Gini_Coefficient'].mean()
avg_gini_ex = oopExtUsersBSwgasis4mnt[oopExtUsersBSwgasis4mnt['Period'].isin(['Jul 2025', 'Aug 2025'])]['Gini_Coefficient'].mean()

summary_data = {
    'UserType': ['New users (< 3MOB)', 'Existing users (>= 3 MOB)'],
    'OOP_Repay_Dev_Gini': [0.30, 0.32],
    'OOP_Repay_Prod_Gini_Avg': [round(avg_gini_new, 2), round(avg_gini_ex, 2)],
    'Attrition_Dev_C_Index': [0.66, 0.64],
    'Attrition_Prod_C_Index': [0.60, 0.59]
}

df_summary = pd.DataFrame(summary_data)
df_summary['run_date'] = CURRENT_DATE
df_summary['monthdisplayed'] = 'Jul25-Aug25'
display(df_summary)


,UserType,OOP_Repay_Dev_Gini,OOP_Repay_Prod_Gini_Avg,Attrition_Dev_C_Index,Attrition_Prod_C_Index,run_date,monthdisplayed
0,New users (< 3MOB),0.30,0.19,0.66,0.60,20260330,Jul25-Aug25
1,Existing users (>= 3 MOB),0.32,0.30,0.64,0.59,20260330,Jul25-Aug25


In [93]:
# Upload to BigQuery
table_id = "prj-prod-dataplatform.dap_ds_poweruser_playground.tendomodelmonitoringdashboardslide1"
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_TRUNCATE",  # or "WRITE_APPEND"
)
job = client.load_table_from_dataframe(df_summary, table_id, job_config=job_config)
job.result()  # Wait for the job to complete

LoadJob<project=prj-prod-dataplatform, location=asia-southeast1, id=e30c066e-f18f-4336-aca5-f3c8e552f248>

## Distribution metrics

### prod new

In [94]:
# finding oop score ranges
# 1) observed min/max per segment from production
mm = (
    dt_prod_api.query('run_date >= "2025-10-15"')
      .assign(
          oop_risk_segment_prod=lambda d: d["oop_risk_segment_prod"].astype(str),
          oop_score_prod=lambda d: pd.to_numeric(d["oop_score_prod"], errors="coerce"),
      )
      .pivot_table(index="oop_risk_segment_prod", values="oop_score_prod", aggfunc=["min", "max"])
)

# flatten columns: ('min','oop_score_prod') -> 'min', same for 'max'
mm = mm.droplevel(1, axis=1).rename(columns={"min": "min_score", "max": "max_score"})

# enforce segment order: A highest scores ... F lowest
order = list("ABCDEF")
mm = mm.reindex(order)

# clamp to [0,1] (safe even if scores are slightly outside)
mm["min_score"] = mm["min_score"].clip(0, 1)
mm["max_score"] = mm["max_score"].clip(0, 1)

# if any segments missing in prod sample, fill min/max by interpolation between neighbors
mm[["min_score", "max_score"]] = mm[["min_score", "max_score"]].interpolate(limit_direction="both")

# 2) compute cutpoints between adjacent segments:
# boundary between (A,B) = midpoint between min(A) and max(B), etc.
cut_idx = [f"{hi}/{lo}" for hi, lo in zip(order[:-1], order[1:])]
cuts = pd.Series(
    [(mm.loc[hi, "min_score"] + mm.loc[lo, "max_score"]) / 2 for hi, lo in zip(order[:-1], order[1:])],
    index=cut_idx,
)

# enforce monotonicity (A/B cutoff should be >= B/C cutoff >= ... >= E/F cutoff)
cuts = cuts.cummin()

# 3) build bins for pd.cut (ascending edges) + labels (F..A)
# edges: [0, cut_EF, cut_DE, ..., cut_AB, 1]
edges_new = [0.0] + cuts.iloc[::-1].tolist() + [np.nextafter(1.0, 2.0)]
labels_new = list("FEDCBA")

# Optional: a readable cutoff table
cutoff_table = pd.DataFrame({
    "segment": labels_new,
    "min_inclusive": edges_new[:-1],
    "max_exclusive": edges_new[1:],
})
cutoff_table.loc[cutoff_table["segment"] == "A", "max_exclusive"] = 1.0

In [95]:
dt_prod_api_calc["oop_risk_segment_bs"] = pd.cut(
    dt_prod_api_calc["score_oop"],
    bins=edges_new,
    labels=labels_new,
    right=False,          # intervals are [a, b) so boundary goes to the higher segment
    include_lowest=True,
)

#### New prod users, prod score, June-Aug

## Slide 3

In [96]:
# slide 3 upper part
# New prod users, prod score, June-Aug
df = dt_prod_api_calc.query('onboarding_date_ym >= 202506 & onboarding_date_ym <= 202508').copy()

df["oop_target_bad"] = 1 - df["oop_target"]

df["osbal_current_bad"] = df["oop_target_bad"] * df["osbal_as_of_current_date"]

pt = pd.pivot_table(
    df,
    index="oop_risk_segment_prod",
    values=["oop_target_bad", "osbal_as_of_resignation_date", "osbal_current_bad"],
    aggfunc={
        "oop_target_bad": ["count", "mean"],
        "osbal_as_of_resignation_date": "sum",
        "osbal_current_bad": "sum"
    }
)

pt[("php_weighted_outstanding_bad_rate", "ratio")] = (
    pt[("osbal_current_bad", "sum")] / pt[("osbal_as_of_resignation_date", "sum")]
)

pt
newprodusersprodscorejuneaug25 = pt.reset_index()
newprodusersprodscorejuneaug25

oop_risk_segment_prod oop_target_bad           osbal_as_of_resignation_date  \
                                 count      mean                          sum   
0                     A             78  0.615385                   978836.238   
1                     B             65  0.692308                   630198.853   
2                     C             42  0.738095                   392620.044   
3                     D              7       1.0                    59443.298   
4                     E              9  0.555556                    48530.317   
5                     F             16     0.625                   179410.149   

  osbal_current_bad php_weighted_outstanding_bad_rate  
                sum                             ratio  
0        718432.603                          0.733966  
1         424655.68                          0.673844  
2        290468.873                          0.739822  
3         44854.565                          0.754577  
4         39893.464                          0.822032  
5        102230.745                          0.569816

In [97]:
newprodusersprodscorejuneaug25.columns

newprodusersprodscorejuneaug25.columns = newprodusersprodscorejuneaug25.columns.droplevel(1)
newprodusersprodscorejuneaug25.columns = ['OOP_Risk_Group', 'Count', 'Bad_rate', 'OS_bal_as_of_resignation_date', 'OS_bal_flag_bad_as_of_today', 'Outstanding_balance_bad_rate']
newprodusersprodscorejuneaug25

,OOP_Risk_Group,Count,Bad_rate,OS_bal_as_of_resignation_date,OS_bal_flag_bad_as_of_today,Outstanding_balance_bad_rate
0,A,78,0.615385,978836.238,718432.603,0.733966
1,B,65,0.692308,630198.853,424655.68,0.673844
2,C,42,0.738095,392620.044,290468.873,0.739822
3,D,7,1.0,59443.298,44854.565,0.754577
4,E,9,0.555556,48530.317,39893.464,0.822032
5,F,16,0.625,179410.149,102230.745,0.569816


In [98]:
# Slide 3 lower part
# New prod users, BS score, June-Aug
df = dt_prod_api_calc.query('onboarding_date_ym >= 202506 & onboarding_date_ym <= 202508').copy()

df["oop_target_bad"] = 1 - df["oop_target"]

df["osbal_current_bad"] = df["oop_target_bad"] * df["osbal_as_of_current_date"]

pt = pd.pivot_table(
    df,
    index="oop_risk_segment_bs",
    values=["oop_target_bad", "osbal_as_of_resignation_date", "osbal_current_bad"],
    aggfunc={
        "oop_target_bad": ["count", "mean"],
        "osbal_as_of_resignation_date": "sum",
        "osbal_current_bad": "sum"
    }
)

pt[("php_weighted_outstanding_bad_rate", "ratio")] = (
    pt[("osbal_current_bad", "sum")] / pt[("osbal_as_of_resignation_date", "sum")]
)

pt.sort_index(ascending=False)
newprodusersbsscorejuneaug25 = pt.sort_index(ascending=False).reset_index()
newprodusersbsscorejuneaug25.columns = newprodusersbsscorejuneaug25.columns.droplevel(1)
newprodusersbsscorejuneaug25.columns = ['OOP_Risk_Group', 'Count', 'Bad_rate', 'OS_bal_as_of_resignation_date', 'OS_bal_flag_bad_as_of_today', 'Outstanding_balance_bad_rate']
newprodusersbsscorejuneaug25

,OOP_Risk_Group,Count,Bad_rate,OS_bal_as_of_resignation_date,OS_bal_flag_bad_as_of_today,Outstanding_balance_bad_rate
0,A,6,0.833333,92872.273,71395.594,0.76875
1,B,70,0.557143,774778.162,542748.947,0.700522
2,C,69,0.695652,858157.414,604477.696,0.70439
3,D,29,0.689655,345129.469,215834.334,0.625372
4,E,17,0.647059,135849.186,103826.964,0.764281
5,F,14,1.0,82252.395,82252.395,1.0


### bs new

In [99]:
dt_bs_oop_new_calc["oop_risk_segment_bs"] = pd.cut(
    dt_bs_oop_new_calc["score_oop"],
    bins=edges_new,
    labels=labels_new,
    right=False,          # intervals are [a, b) so boundary goes to the higher segment
    include_lowest=True,
)

C:\Users\Dwaipayan\AppData\Local\Temp\ipykernel_21304\2931598673.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dt_bs_oop_new_calc["oop_risk_segment_bs"] = pd.cut(


In [100]:
# New BS users, BS score, June-Aug
df = dt_bs_oop_new_calc.query('onboarding_date_ym >= 202506 & onboarding_date_ym <= 202508').copy()

df["oop_target_bad"] = 1 - df["oop_target"]

df["osbal_current_bad"] = df["oop_target_bad"] * df["osbal_as_of_current_date"]

pt = pd.pivot_table(
    df,
    index="oop_risk_segment_bs",
    values=["oop_target_bad", "osbal_as_of_resignation_date", "osbal_current_bad"],
    aggfunc={
        "oop_target_bad": ["count", "mean"],
        "osbal_as_of_resignation_date": "sum",
        "osbal_current_bad": "sum"
    }
)

pt[("php_weighted_outstanding_bad_rate", "ratio")] = (
    pt[("osbal_current_bad", "sum")] / pt[("osbal_as_of_resignation_date", "sum")]
)

pt.sort_index(ascending=False)
newbsusersbsscorejuneaug25 = pt.sort_index(ascending=False).reset_index()
newbsusersbsscorejuneaug25.columns = newbsusersbsscorejuneaug25.columns.droplevel(1)
newbsusersbsscorejuneaug25.columns = ['OOP_Risk_Group', 'Count', 'Bad_rate', 'OS_bal_as_of_resignation_date', 'OS_bal_flag_bad_as_of_today', 'Outstanding_balance_bad_rate']
newbsusersbsscorejuneaug25


,OOP_Risk_Group,Count,Bad_rate,OS_bal_as_of_resignation_date,OS_bal_flag_bad_as_of_today,Outstanding_balance_bad_rate
0,A,26,0.576923,422222.996,257628.073,0.610171
1,B,636,0.632075,7752261.328,4311143.994,0.556114
2,C,298,0.697987,3936843.789,2629351.936,0.667883
3,D,135,0.622222,1775993.106,945149.309,0.532181
4,E,69,0.710145,1109077.514,624096.365,0.562717
5,F,33,0.909091,369621.079,280475.128,0.758818


### prod existing

In [101]:
# finding oop score ranges
# 1) observed min/max per segment from production
mm = (
    dt_prod_batch.query('run_date >= "2025-10-15"')
      .assign(
          oop_risk_segment_prod=lambda d: d["oop_risk_segment_prod"].astype(str),
          oop_score_prod=lambda d: pd.to_numeric(d["oop_score_prod"], errors="coerce"),
      )
      .pivot_table(index="oop_risk_segment_prod", values="oop_score_prod", aggfunc=["min", "max"])
)

# flatten columns: ('min','oop_score_prod') -> 'min', same for 'max'
mm = mm.droplevel(1, axis=1).rename(columns={"min": "min_score", "max": "max_score"})

# enforce segment order: A highest scores ... F lowest
order = list("ABCDEF")
mm = mm.reindex(order)

# clamp to [0,1] (safe even if scores are slightly outside)
mm["min_score"] = mm["min_score"].clip(0, 1)
mm["max_score"] = mm["max_score"].clip(0, 1)

# if any segments missing in prod sample, fill min/max by interpolation between neighbors
mm[["min_score", "max_score"]] = mm[["min_score", "max_score"]].interpolate(limit_direction="both")

# 2) compute cutpoints between adjacent segments:
# boundary between (A,B) = midpoint between min(A) and max(B), etc.
cut_idx = [f"{hi}/{lo}" for hi, lo in zip(order[:-1], order[1:])]
cuts = pd.Series(
    [(mm.loc[hi, "min_score"] + mm.loc[lo, "max_score"]) / 2 for hi, lo in zip(order[:-1], order[1:])],
    index=cut_idx,
)

# enforce monotonicity (A/B cutoff should be >= B/C cutoff >= ... >= E/F cutoff)
cuts = cuts.cummin()

# 3) build bins for pd.cut (ascending edges) + labels (F..A)
# edges: [0, cut_EF, cut_DE, ..., cut_AB, 1]
edges_ex = [0.0] + cuts.iloc[::-1].tolist() + [np.nextafter(1.0, 2.0)]
labels_ex = list("FEDCBA")

# Optional: a readable cutoff table
cutoff_table = pd.DataFrame({
    "segment": labels_ex,
    "min_inclusive": edges_ex[:-1],
    "max_exclusive": edges_ex[1:],
})
cutoff_table.loc[cutoff_table["segment"] == "A", "max_exclusive"] = 1.0

In [102]:
dt_prod_batch_calc["oop_risk_segment_bs"] = pd.cut(
    dt_prod_batch_calc["score_oop"],
    bins=edges_ex,
    labels=labels_ex,
    right=False,          # intervals are [a, b) so boundary goes to the higher segment
    include_lowest=True,
)

In [103]:
# Existing prod users, prod score, June
df = dt_prod_batch_calc.query('run_date_ym == 202506').copy()

df["oop_target_bad"] = 1 - df["oop_target"]

df["osbal_current_bad"] = df["oop_target_bad"] * df["osbal_as_of_current_date"]

pt = pd.pivot_table(
    df,
    index="oop_risk_segment_prod",
    values=["oop_target_bad", "osbal_as_of_resignation_date", "osbal_current_bad"],
    aggfunc={
        "oop_target_bad": ["count", "mean"],
        "osbal_as_of_resignation_date": "sum",
        "osbal_current_bad": "sum"
    }
)

pt[("php_weighted_outstanding_bad_rate", "ratio")] = (
    pt[("osbal_current_bad", "sum")] / pt[("osbal_as_of_resignation_date", "sum")]
)

pt.sort_index(ascending=True)
extprodusersprodscorejune = pt.sort_index(ascending=True).reset_index()

extprodusersprodscorejune.columns = extprodusersprodscorejune.columns.droplevel(1)
extprodusersprodscorejune.columns = ['OOP_Risk_Group', 'Count', 'Bad_rate', 'OS_bal_as_of_resignation_date', 'OS_bal_flag_bad_as_of_today', 'Outstanding_balance_bad_rate']
extprodusersprodscorejune


,OOP_Risk_Group,Count,Bad_rate,OS_bal_as_of_resignation_date,OS_bal_flag_bad_as_of_today,Outstanding_balance_bad_rate
0,A,75,0.586667,1.458888e+06,709061.229,0.486029
1,B,471,0.645435,8.208541e+06,4872733.707,0.593618
2,C,678,0.718289,1.054174e+07,6650068.859,0.630832
3,D,511,0.83953,7.538032e+06,5925112.834,0.786029
4,E,204,0.818627,2.647025e+06,2083808.615,0.787227
5,F,185,0.864865,2.178477e+06,1845814.173,0.847296


In [104]:
# Slide 4 upper part
# Existing prod users, prod score, July
df = dt_prod_batch_calc.query('run_date_ym == 202507').copy()

df["oop_target_bad"] = 1 - df["oop_target"]

df["osbal_current_bad"] = df["oop_target_bad"] * df["osbal_as_of_current_date"]

pt = pd.pivot_table(
    df,
    index="oop_risk_segment_prod",
    values=["oop_target_bad", "osbal_as_of_resignation_date", "osbal_current_bad"],
    aggfunc={
        "oop_target_bad": ["count", "mean"],
        "osbal_as_of_resignation_date": "sum",
        "osbal_current_bad": "sum"
    }
)

pt[("php_weighted_outstanding_bad_rate", "ratio")] = (
    pt[("osbal_current_bad", "sum")] / pt[("osbal_as_of_resignation_date", "sum")]
)

pt.sort_index(ascending=True)
extproduserprodscorejuly = pt.sort_index(ascending=True).reset_index()
extproduserprodscorejuly.columns = extproduserprodscorejuly.columns.droplevel(1)
extproduserprodscorejuly.columns = ['OOP_Risk_Group', 'Count', 'Bad_rate', 'OS_bal_as_of_resignation_date', 'OS_bal_flag_bad_as_of_today', 'Outstanding_balance_bad_rate']
extproduserprodscorejuly

,OOP_Risk_Group,Count,Bad_rate,OS_bal_as_of_resignation_date,OS_bal_flag_bad_as_of_today,Outstanding_balance_bad_rate
0,A,107,0.551402,2.035219e+06,1060618.702,0.521132
1,B,768,0.619792,1.359044e+07,8712928.965,0.641107
2,C,1044,0.692529,1.803966e+07,10956507.037,0.607357
3,D,861,0.799071,1.357779e+07,10382111.667,0.764639
4,E,353,0.827195,5.051804e+06,3988532.788,0.789526
5,F,296,0.868243,3.391040e+06,2817575.435,0.830888


In [105]:
# Existing prod users, prod score, Aug
df = dt_prod_batch_calc.query('run_date_ym == 202508').copy()

df["oop_target_bad"] = 1 - df["oop_target"]

df["osbal_current_bad"] = df["oop_target_bad"] * df["osbal_as_of_current_date"]

pt = pd.pivot_table(
    df,
    index="oop_risk_segment_prod",
    values=["oop_target_bad", "osbal_as_of_resignation_date", "osbal_current_bad"],
    aggfunc={
        "oop_target_bad": ["count", "mean"],
        "osbal_as_of_resignation_date": "sum",
        "osbal_current_bad": "sum"
    }
)

pt[("php_weighted_outstanding_bad_rate", "ratio")] = (
    pt[("osbal_current_bad", "sum")] / pt[("osbal_as_of_resignation_date", "sum")]
)

pt.sort_index(ascending=True)
extprodusersprodscoreaug = pt.sort_index(ascending=True).reset_index()
extprodusersprodscoreaug.columns = extprodusersprodscoreaug.columns.droplevel(1)
extprodusersprodscoreaug.columns = ['OOP_Risk_Group', 'Count', 'Bad_rate', 'OS_bal_as_of_resignation_date', 'OS_bal_flag_bad_as_of_today', 'Outstanding_balance_bad_rate']
extprodusersprodscoreaug

,OOP_Risk_Group,Count,Bad_rate,OS_bal_as_of_resignation_date,OS_bal_flag_bad_as_of_today,Outstanding_balance_bad_rate
0,A,88,0.568182,1.605067e+06,872498.768,0.54359
1,B,630,0.607937,1.033497e+07,6537960.987,0.632606
2,C,804,0.646766,1.359418e+07,8173318.157,0.601237
3,D,699,0.788269,1.084022e+07,8272968.112,0.763173
4,E,282,0.815603,3.786675e+06,3061391.769,0.808464
5,F,231,0.848485,2.482998e+06,2137092.277,0.86069


In [106]:
# Existing prod users, BS score, June
df = dt_prod_batch_calc.query('run_date_ym == 202506').copy()

df["oop_target_bad"] = 1 - df["oop_target"]

df["osbal_current_bad"] = df["oop_target_bad"] * df["osbal_as_of_current_date"]

pt = pd.pivot_table(
    df,
    index="oop_risk_segment_bs",
    values=["oop_target_bad", "osbal_as_of_resignation_date", "osbal_current_bad"],
    aggfunc={
        "oop_target_bad": ["count", "mean"],
        "osbal_as_of_resignation_date": "sum",
        "osbal_current_bad": "sum"
    }
)

pt[("php_weighted_outstanding_bad_rate", "ratio")] = (
    pt[("osbal_current_bad", "sum")] / pt[("osbal_as_of_resignation_date", "sum")]
)

pt.sort_index(ascending=False)
extprodusersbsscorejune25 = pt.sort_index(ascending=False).reset_index()
extprodusersbsscorejune25.columns = extprodusersbsscorejune25.columns.droplevel(1)
extprodusersbsscorejune25.columns = ['OOP_Risk_Group', 'Count', 'Bad_rate', 'OS_bal_as_of_resignation_date', 'OS_bal_flag_bad_as_of_today', 'Outstanding_balance_bad_rate']
extprodusersbsscorejune25


,OOP_Risk_Group,Count,Bad_rate,OS_bal_as_of_resignation_date,OS_bal_flag_bad_as_of_today,Outstanding_balance_bad_rate
0,A,58,0.586207,1453432.492,563112.844,0.387437
1,B,330,0.59697,5807941.261,3515407.676,0.605276
2,C,421,0.76247,7045815.932,4602371.902,0.653206
3,D,385,0.794805,5402697.802,4081372.838,0.755432
4,E,220,0.854545,2491042.542,2095789.428,0.84133
5,F,155,0.858065,1890720.283,1639733.635,0.867253


In [107]:
# Slide 4 lower part
# Existing prod users, BS score, July
df = dt_prod_batch_calc.query('run_date_ym == 202507').copy()

df["oop_target_bad"] = 1 - df["oop_target"]

df["osbal_current_bad"] = df["oop_target_bad"] * df["osbal_as_of_current_date"]

pt = pd.pivot_table(
    df,
    index="oop_risk_segment_bs",
    values=["oop_target_bad", "osbal_as_of_resignation_date", "osbal_current_bad"],
    aggfunc={
        "oop_target_bad": ["count", "mean"],
        "osbal_as_of_resignation_date": "sum",
        "osbal_current_bad": "sum"
    }
)

pt[("php_weighted_outstanding_bad_rate", "ratio")] = (
    pt[("osbal_current_bad", "sum")] / pt[("osbal_as_of_resignation_date", "sum")]
)

pt.sort_index(ascending=False)
extprodusersbsscorejuly25 = pt.sort_index(ascending=False).reset_index()
extprodusersbsscorejuly25.columns = extprodusersbsscorejuly25.columns.droplevel(1)
extprodusersbsscorejuly25.columns = ['OOP_Risk_Group', 'Count', 'Bad_rate', 'OS_bal_as_of_resignation_date', 'OS_bal_flag_bad_as_of_today', 'Outstanding_balance_bad_rate']
extprodusersbsscorejuly25

,OOP_Risk_Group,Count,Bad_rate,OS_bal_as_of_resignation_date,OS_bal_flag_bad_as_of_today,Outstanding_balance_bad_rate
0,A,67,0.492537,1.304149e+06,640130.416,0.490841
1,B,521,0.570058,9.512222e+06,5433451.6,0.571207
2,C,660,0.689394,1.255881e+07,8534721.132,0.679581
3,D,588,0.767007,8.876604e+06,6547134.93,0.737572
4,E,317,0.835962,4.282780e+06,3543305.442,0.827338
5,F,265,0.856604,3.150621e+06,2727977.478,0.865854


In [108]:
# Existing prod users, BS score, Aug
df = dt_prod_batch_calc.query('run_date_ym == 202508').copy()

df["oop_target_bad"] = 1 - df["oop_target"]

df["osbal_current_bad"] = df["oop_target_bad"] * df["osbal_as_of_current_date"]

pt = pd.pivot_table(
    df,
    index="oop_risk_segment_bs",
    values=["oop_target_bad", "osbal_as_of_resignation_date", "osbal_current_bad"],
    aggfunc={
        "oop_target_bad": ["count", "mean"],
        "osbal_as_of_resignation_date": "sum",
        "osbal_current_bad": "sum"
    }
)

pt[("php_weighted_outstanding_bad_rate", "ratio")] = (
    pt[("osbal_current_bad", "sum")] / pt[("osbal_as_of_resignation_date", "sum")]
)

pt.sort_index(ascending=False)
extprodusersbsscoreaug25 = pt.sort_index(ascending=False).reset_index()
extprodusersbsscoreaug25.columns = extprodusersbsscoreaug25.columns.droplevel(1)
extprodusersbsscoreaug25.columns = ['OOP_Risk_Group', 'Count', 'Bad_rate', 'OS_bal_as_of_resignation_date', 'OS_bal_flag_bad_as_of_today', 'Outstanding_balance_bad_rate']
extprodusersbsscoreaug25

,OOP_Risk_Group,Count,Bad_rate,OS_bal_as_of_resignation_date,OS_bal_flag_bad_as_of_today,Outstanding_balance_bad_rate
0,A,59,0.389831,9.238067e+05,449721.892,0.486814
1,B,428,0.546729,7.710388e+06,4468893.626,0.579594
2,C,527,0.654649,1.013974e+07,6633828.498,0.65424
3,D,459,0.740741,7.020451e+06,4930936.515,0.702367
4,E,245,0.804082,3.038082e+06,2602366.828,0.856582
5,F,184,0.880435,2.217247e+06,1953892.6,0.881225


### bs existing

In [109]:
dt_bs_oop_ex_calc["oop_risk_segment_bs"] = pd.cut(
    dt_bs_oop_ex_calc["score_oop"],
    bins=edges_ex,
    labels=labels_ex,
    right=False,          # intervals are [a, b) so boundary goes to the higher segment
    include_lowest=True,
)

C:\Users\Dwaipayan\AppData\Local\Temp\ipykernel_21304\125666790.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dt_bs_oop_ex_calc["oop_risk_segment_bs"] = pd.cut(


In [110]:
# Existing BS users, BS score, June
df = dt_bs_oop_ex_calc.query('calc_date_ym == 202506').copy()

df["oop_target_bad"] = 1 - df["oop_target"]

df["osbal_current_bad"] = df["oop_target_bad"] * df["osbal_as_of_current_date"]

pt = pd.pivot_table(
    df,
    index="oop_risk_segment_bs",
    values=["oop_target_bad", "osbal_as_of_resignation_date", "osbal_current_bad"],
    aggfunc={
        "oop_target_bad": ["count", "mean"],
        "osbal_as_of_resignation_date": "sum",
        "osbal_current_bad": "sum"
    }
)

pt[("php_weighted_outstanding_bad_rate", "ratio")] = (
    pt[("osbal_current_bad", "sum")] / pt[("osbal_as_of_resignation_date", "sum")]
)

pt.sort_index(ascending=False)
extbsusersbsscorejune25 = pt.sort_index(ascending=False).reset_index()
extbsusersbsscorejune25.columns = extbsusersbsscorejune25.columns.droplevel(1)
extbsusersbsscorejune25.columns = ['OOP_Risk_Group', 'Count', 'Bad_rate', 'OS_bal_as_of_resignation_date', 'OS_bal_flag_bad_as_of_today', 'Outstanding_balance_bad_rate']
extbsusersbsscorejune25

,OOP_Risk_Group,Count,Bad_rate,OS_bal_as_of_resignation_date,OS_bal_flag_bad_as_of_today,Outstanding_balance_bad_rate
0,A,150,0.526667,2.547140e+06,1059353.077,0.415899
1,B,1054,0.66129,1.651084e+07,10194437.235,0.617439
2,C,1375,0.780364,2.160962e+07,15281791.368,0.707175
3,D,1113,0.806828,1.513597e+07,10994733.162,0.726398
4,E,603,0.870647,7.346968e+06,6149291.482,0.836984
5,F,472,0.885593,5.705456e+06,4791217.979,0.839761


In [111]:
# Existing BS users, BS score, July
df = dt_bs_oop_ex_calc.query('calc_date_ym == 202507').copy()

df["oop_target_bad"] = 1 - df["oop_target"]

df["osbal_current_bad"] = df["oop_target_bad"] * df["osbal_as_of_current_date"]

pt = pd.pivot_table(
    df,
    index="oop_risk_segment_bs",
    values=["oop_target_bad", "osbal_as_of_resignation_date", "osbal_current_bad"],
    aggfunc={
        "oop_target_bad": ["count", "mean"],
        "osbal_as_of_resignation_date": "sum",
        "osbal_current_bad": "sum"
    }
)

pt[("php_weighted_outstanding_bad_rate", "ratio")] = (
    pt[("osbal_current_bad", "sum")] / pt[("osbal_as_of_resignation_date", "sum")]
)

pt.sort_index(ascending=False)
extbsusersbsscorejuly25 = pt.sort_index(ascending=False).reset_index()
extbsusersbsscorejuly25.columns = extbsusersbsscorejuly25.columns.droplevel(1)
extbsusersbsscorejuly25.columns = ['OOP_Risk_Group', 'Count', 'Bad_rate', 'OS_bal_as_of_resignation_date', 'OS_bal_flag_bad_as_of_today', 'Outstanding_balance_bad_rate']
extbsusersbsscorejuly25

,OOP_Risk_Group,Count,Bad_rate,OS_bal_as_of_resignation_date,OS_bal_flag_bad_as_of_today,Outstanding_balance_bad_rate
0,A,209,0.473684,3.524752e+06,1286532.767,0.365
1,B,1550,0.645161,2.403508e+07,13694078.009,0.569754
2,C,1696,0.739387,2.528971e+07,17175931.488,0.679167
3,D,1034,0.801741,1.462373e+07,10664282.185,0.729245
4,E,478,0.853556,5.856246e+06,4795593.933,0.818885
5,F,407,0.874693,4.819724e+06,4081221.207,0.846775


In [112]:
# Existing BS users, BS score, Aug
df = dt_bs_oop_ex_calc.query('calc_date_ym == 202508').copy()

df["oop_target_bad"] = 1 - df["oop_target"]

df["osbal_current_bad"] = df["oop_target_bad"] * df["osbal_as_of_current_date"]

pt = pd.pivot_table(
    df,
    index="oop_risk_segment_bs",
    values=["oop_target_bad", "osbal_as_of_resignation_date", "osbal_current_bad"],
    aggfunc={
        "oop_target_bad": ["count", "mean"],
        "osbal_as_of_resignation_date": "sum",
        "osbal_current_bad": "sum"
    }
)

pt[("php_weighted_outstanding_bad_rate", "ratio")] = (
    pt[("osbal_current_bad", "sum")] / pt[("osbal_as_of_resignation_date", "sum")]
)

pt.sort_index(ascending=False)
extbsusersbsscoreaug25 = pt.sort_index(ascending=False).reset_index()
extbsusersbsscoreaug25.columns = extbsusersbsscoreaug25.columns.droplevel(1)
extbsusersbsscoreaug25.columns = ['OOP_Risk_Group', 'Count', 'Bad_rate', 'OS_bal_as_of_resignation_date', 'OS_bal_flag_bad_as_of_today', 'Outstanding_balance_bad_rate']
extbsusersbsscoreaug25

,OOP_Risk_Group,Count,Bad_rate,OS_bal_as_of_resignation_date,OS_bal_flag_bad_as_of_today,Outstanding_balance_bad_rate
0,A,191,0.450262,2.985153e+06,1121195.355,0.375591
1,B,1347,0.613957,2.041318e+07,11367543.291,0.556873
2,C,1515,0.721452,2.256163e+07,15224510.17,0.674797
3,D,878,0.781321,1.256168e+07,9019983.484,0.718055
4,E,388,0.832474,4.876232e+06,4158499.362,0.85281
5,F,332,0.900602,3.948235e+06,3513789.588,0.889965


# Attrition

In [113]:
# BS Attrition
sql_query_attrition = """
SELECT *
FROM `prj-prod-dataplatform.risk_mart.tendo_backscored_jan24_feb26_20260301_attrition`
"""

dt_bs_attr = client.query(sql_query_attrition).to_dataframe()

In [114]:
print(f"The shape of the dataframe backscored attrition data is:\t  {dt_bs_attr.shape}")

The shape of the dataframe backscored attrition data is:	  (1932839, 8)


In [115]:
dd.query("""select ee_customer_id, count(ee_customer_id)cnt from dt_bs_attr group by 1 having cnt > 1;""").to_df()

,ee_customer_id,cnt
0,1479827,6
1,1488256,8
2,1494882,7
3,1570780,5
4,1579892,2
...,...,...
170619,1691253,2
170620,1513735,2
170621,1667217,2
170622,1667603,2


In [116]:
dd.query("""select * from dt_bs_attr where ee_customer_id = '1202344';""").to_df()

,ee_customer_id,calc_date,ee_onboarding_month,is_new_customer_flag_3m,target_event,target_duration_dev,score_attr,score_attr_segment
0,1202344,2024-08-01,202407,1,0.0,12.0,10.0,Low
1,1202344,2024-09-01,202407,1,0.0,12.0,10.0,Low
2,1202344,2024-10-01,202407,1,0.0,12.0,10.0,Low
3,1202344,2025-01-01,202407,0,0.0,12.0,10.0,Low
4,1202344,2024-11-01,202407,0,0.0,12.0,9.0,Average
5,1202344,2024-12-01,202407,0,0.0,12.0,9.0,Average
6,1202344,2025-02-01,202407,0,0.0,12.0,10.0,Low
7,1202344,2025-03-01,202407,0,0.0,12.0,10.0,Low
8,1202344,2025-04-01,202407,0,0.0,12.0,9.0,Average
9,1202344,2025-05-01,202407,0,0.0,12.0,9.0,Average


In [117]:
dt_bs_attr['ee_customer_id'] = dt_bs_attr['ee_customer_id'].astype('str')

dt_bs_attr["calc_date"] = pd.to_datetime(dt_bs_attr["calc_date"], errors="coerce")
dt_bs_attr["calc_date_correct"] = pd.to_datetime(dt_bs_attr["calc_date"], errors="coerce") - pd.DateOffset(months=1)
dt_bs_attr['calc_date_ym'] = dt_bs_attr['calc_date_correct'].dt.year*100 + dt_bs_attr['calc_date_correct'].dt.month

In [118]:
new_1m = dt_bs_attr['ee_onboarding_month'] == dt_bs_attr['calc_date_ym']
dt_bs_attr['is_new_customer_flag_1m'] = new_1m.astype('int')

In [119]:
dt_bs_attr.sample(5)

,ee_customer_id,calc_date,ee_onboarding_month,is_new_customer_flag_3m,target_event,target_duration_dev,score_attr,score_attr_segment,calc_date_correct,calc_date_ym,is_new_customer_flag_1m
1526869,718148,2025-12-01,202202,0,0.0,12.0,8.0,Average,2025-11-01,202511,0
1930877,1696493,2026-03-01,202602,1,0.0,12.0,inf,Very low,2026-02-01,202602,1
806286,195042,2025-05-01,202011,0,0.0,12.0,6.0,High,2025-04-01,202504,0
983701,1378189,2025-08-01,202506,1,0.0,12.0,9.0,Average,2025-07-01,202507,0
954018,1397610,2025-06-01,202504,1,1.0,0.0,10.0,Low,2025-05-01,202505,0


In [120]:
# dt_res = pd.read_pickle(
#     generate_bucket_url('DC/Tendo_Model_Monitoring_Data/resignation_data_20260330.pkl', GS_BUCKET)
# )

print(f"The shape of the resignation data downloaded from the cloud is:\t {dt_res.shape}")
dt_res.sample(5)

The shape of the resignation data downloaded from the cloud is:	 (197787, 2)


,ee_customer_id,ee_resignation_date_correct
626889,1384611,NaT
1053166,1343412,NaT
964213,1383896,NaT
719170,1117569,NaT
931683,966993,NaT


## Distribution metrics

### prod new

In [121]:
dt_prod_api.shape

(165276, 15)

In [122]:
dt_prod_api.sample(5)

,ee_customer_id,run_date,attrition_risk_segment,attrition_time_to_leave,oop_score_prod,oop_risk_segment_prod,ee_onboarding_date,oop_target,score_oop,osbal_as_of_resignation_date,osbal_as_of_oop_eligible_date,osbal_as_of_current_date,onb_rd_diff,onboarding_date_ym,run_date_ym
57131,1346365,2025-12-16,6,6,0.609415,B,NaT,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,202512
67252,1555648,2025-10-31,4,4,0.592835,B,2025-11-30,<NA>,0.587813,NaN,NaN,2221.846,30.0,202511.0,202510
99235,1507656,2025-11-11,4,4,0.489078,C,NaT,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,202511
148480,1574525,2025-10-08,4,4,0.644240,B,2025-10-08,<NA>,0.644240,NaN,NaN,40275.184,0.0,202510.0,202510
66736,1522236,2026-01-09,10-12,11,0.614184,B,NaT,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,202601


In [123]:
dt_prod_api_attr = dt_prod_api.merge(
    dt_res,
    how='left',
    on='ee_customer_id'
).merge(
    dt_bs_attr[['ee_customer_id', 'calc_date_ym', 'score_attr', 'score_attr_segment', 'is_new_customer_flag_1m']],
    how='left',
    left_on=['ee_customer_id','run_date_ym'],
    right_on=['ee_customer_id','calc_date_ym']
)

In [124]:
print(f"The shape of the dt_prod_api_attr after merging it with dt_res and dt_bs_attr is:\t{dt_prod_api_attr.shape} ")

The shape of the dt_prod_api_attr after merging it with dt_res and dt_bs_attr is:	(165276, 20) 


In [125]:
dt_prod_api_calc = (dt_prod_api_attr
                    .dropna(subset=['ee_onboarding_date'])
                    .sort_values(['onb_rd_diff','run_date'])
                    .drop_duplicates(subset=['ee_customer_id'], keep='first'))

In [126]:
print(f"The shape of dt_prod_api_calc after dropping duplicates and null values in ee_onboarding_date is:\t{dt_prod_api_calc.shape}")

The shape of dt_prod_api_calc after dropping duplicates and null values in ee_onboarding_date is:	(11336, 20)


In [127]:
months_diff = (
    (dt_prod_api_calc['ee_resignation_date_correct'].dt.year - dt_prod_api_calc['run_date'].dt.year) * 12 +
    (dt_prod_api_calc['ee_resignation_date_correct'].dt.month - dt_prod_api_calc['run_date'].dt.month)
)

dt_prod_api_calc['time_to_attrition'] = np.where(dt_prod_api_calc['ee_resignation_date_correct'].isna(), np.nan, months_diff)
dt_prod_api_calc['attrition_event'] = dt_prod_api_calc['ee_resignation_date_correct'].notna().astype('int')

In [128]:
dt_prod_api_calc['score_attr_corrected'] = dt_prod_api_calc['score_attr'].replace(np.inf, 15)
dt_prod_api_calc['attrition_score_prod'] = dt_prod_api_calc['attrition_time_to_leave'].replace('12+', '15').astype('float')

mapping_dict = {
            1: 'Very high',
            2: 'Very high',
            3: 'Very high',
            4: 'High',
            5: 'High',
            6: 'High',
            7: 'Average',
            8: 'Average',
            9: 'Average',
            10: 'Low',
            11: 'Low',
            12: 'Low',
            15: 'Very low'
        }

dt_prod_api_calc['attrition_risk_segment_prod'] = dt_prod_api_calc['attrition_score_prod'].replace(mapping_dict)

In [129]:
# Slide 5 upper part
# New prod users, prod score, June-Aug
df = dt_prod_api_calc.query('onboarding_date_ym >= 202506 & onboarding_date_ym <= 202508').copy()

pt = pd.pivot_table(
    df,
    index="attrition_risk_segment_prod", # score_attr_segment - BS
    values=["attrition_event", "attrition_score_prod", "time_to_attrition", "ee_customer_id"],
    aggfunc={
        "attrition_event": "mean",
        "attrition_score_prod": "mean",
        "time_to_attrition": "mean",
        "ee_customer_id": "count"
    }
).rename(
    columns={"ee_customer_id": "count"}
)[['attrition_event', 'attrition_score_prod', 'time_to_attrition', 'count']]

order = ['Very low', 'Low', 'Average', 'High', 'Very high']

pt.loc[order]

attrnewprodusersprodscorejuneaug25 = pt.loc[order].reset_index()
# attrnewprodusersprodscorejuneaug25.columns = attrnewprodusersprodscorejuneaug25.columns.droplevel(1)
# extbsusersbsscoreaug25.columns = ['OOP_Risk_Group', 'Count', 'Bad_rate', 'OS_bal_as_of_resignation_date', 'OS_bal_flag_bad_as_of_today', 'Outstanding_balance_bad_rate']
attrnewprodusersprodscorejuneaug25 = attrnewprodusersprodscorejuneaug25[['attrition_risk_segment_prod', 'count', 'attrition_event', 'attrition_score_prod', 'time_to_attrition']]
attrnewprodusersprodscorejuneaug25

,attrition_risk_segment_prod,count,attrition_event,attrition_score_prod,time_to_attrition
0,Very low,44,0.227273,15.000000,0.000000
1,Low,185,0.178378,10.254054,4.333333
2,Average,159,0.188679,8.641509,4.433333
3,High,1080,0.214815,4.450000,3.586207
4,Very high,638,0.252351,2.929467,3.422360


In [130]:
# Slide 5 lower part
# New prod users, BS score, June-Aug
df = dt_prod_api_calc.query('onboarding_date_ym >= 202506 & onboarding_date_ym <= 202508').copy()

pt = pd.pivot_table(
    df,
    index="score_attr_segment", # score_attr_segment - BS
    values=["attrition_event", "score_attr_corrected", "time_to_attrition", "ee_customer_id"],
    aggfunc={
        "attrition_event": "mean",
        "score_attr_corrected": "mean",
        "time_to_attrition": "mean",
        "ee_customer_id": "count"
    }
).rename(
    columns={"ee_customer_id": "count"}
)[['attrition_event', 'score_attr_corrected', 'time_to_attrition', 'count']]

order = ['Very low', 'Low', 'Average', 'High', 'Very high']

pt.loc[order]
attrnewprodusersbsscorejuneaug25 = pt.loc[order].reset_index()
attrnewprodusersbsscorejuneaug25 = attrnewprodusersbsscorejuneaug25[['score_attr_segment', 'count', 'attrition_event', 'score_attr_corrected', 'time_to_attrition']]
attrnewprodusersbsscorejuneaug25

,score_attr_segment,count,attrition_event,score_attr_corrected,time_to_attrition
0,Very low,264,0.208333,15.000000,4.690909
1,Low,216,0.152778,10.342593,3.666667
2,Average,615,0.188618,7.912195,4.267241
3,High,890,0.226966,5.174157,4.183168
4,Very high,65,0.246154,2.953846,4.125000


### bs new

In [131]:
dt_bs_attr.sample(5)

,ee_customer_id,calc_date,ee_onboarding_month,is_new_customer_flag_3m,target_event,target_duration_dev,score_attr,score_attr_segment,calc_date_correct,calc_date_ym,is_new_customer_flag_1m
1619563,1206735,2026-02-01,202405,0,0.0,12.0,8.0,Average,2026-01-01,202601,0
474689,1166788,2024-12-01,202408,0,0.0,12.0,6.0,High,2024-11-01,202411,0
783266,1231544,2025-04-01,202407,0,0.0,12.0,inf,Very low,2025-03-01,202503,0
41039,731067,2024-02-01,202203,0,1.0,11.0,5.0,High,2024-01-01,202401,0
277912,1224240,2024-09-01,202407,1,1.0,7.0,5.0,High,2024-08-01,202408,0


In [132]:
for col in dt_bs_attr.select_dtypes(include='datetime').columns:
    dt_bs_attr[col] = dt_bs_attr[col].astype('datetime64[s]')

for col in dt_res.select_dtypes(include='datetime').columns:
    dt_res[col] = dt_res[col].astype('datetime64[s]')

In [133]:
# chunk_size = 500_000
# chunks = []

# for i in range(0, len(dt_bs_attr), chunk_size):
#     chunk = dt_bs_attr.iloc[i:i+chunk_size].merge(
#         dt_res, how='left', on='ee_customer_id'
#     )
#     chunks.append(chunk)

# dt_bs_attr_dev = pd.concat(chunks, ignore_index=True)

In [134]:
dt_bs_attr_dev = dt_bs_attr.merge(
    dt_res,
    how='left',
    on='ee_customer_id'
)

In [135]:
dt_bs_attr_dev_calc = dt_bs_attr_dev.query('is_new_customer_flag_1m == 1').copy()

In [136]:
months_diff = (
    (dt_bs_attr_dev_calc['ee_resignation_date_correct'].dt.year - dt_bs_attr_dev_calc['calc_date_correct'].dt.year) * 12 +
    (dt_bs_attr_dev_calc['ee_resignation_date_correct'].dt.month - dt_bs_attr_dev_calc['calc_date_correct'].dt.month)
)

dt_bs_attr_dev_calc['time_to_attrition'] = np.where(dt_bs_attr_dev_calc['ee_resignation_date_correct'].isna(), np.nan, months_diff)
dt_bs_attr_dev_calc['attrition_event'] = dt_bs_attr_dev_calc['ee_resignation_date_correct'].notna().astype('int')

In [137]:
dt_bs_attr_dev_calc['score_attr_corrected'] = dt_bs_attr_dev_calc['score_attr'].replace(np.inf, 15)

In [138]:
# New BS users, BS score, June-Aug
df = dt_bs_attr_dev_calc.query('ee_onboarding_month >= 202506 & ee_onboarding_month <= 202508').copy()

pt = pd.pivot_table(
    df,
    index="score_attr_segment",
    values=["attrition_event", "score_attr_corrected", "time_to_attrition", "ee_customer_id"],
    aggfunc={
        "attrition_event": "mean",
        "score_attr_corrected": "mean",
        "time_to_attrition": "mean",
        "ee_customer_id": "count"
    }
).rename(
    columns={"ee_customer_id": "count"}
)[['attrition_event', 'score_attr_corrected', 'time_to_attrition', 'count']]

order = ['Very low', 'Low', 'Average', 'High', 'Very high']

pt.loc[order]

attrnewbsuserbsscorejuneaug25 = pt.loc[order].reset_index()
attrnewbsuserbsscorejuneaug25 = attrnewbsuserbsscorejuneaug25[['score_attr_segment', 'count', 'attrition_event', 'score_attr_corrected', 'time_to_attrition']]
attrnewbsuserbsscorejuneaug25

C:\Users\Dwaipayan\AppData\Local\Temp\ipykernel_21304\3184220800.py:2: RuntimeWarning: Engine has switched to 'python' because numexpr does not support extension array dtypes. Please set your engine to python manually.
  df = dt_bs_attr_dev_calc.query('ee_onboarding_month >= 202506 & ee_onboarding_month <= 202508').copy()


,score_attr_segment,count,attrition_event,score_attr_corrected,time_to_attrition
0,Very low,1909,0.137245,15.000000,4.843511
1,Low,2814,0.125800,10.472637,4.437853
2,Average,5862,0.210167,8.138690,4.420455
3,High,2918,0.261480,5.473955,4.159895
4,Very high,438,0.461187,2.582192,4.272277


### prod existing

In [139]:
dt_prod_batch.shape

(404256, 16)

In [140]:
dt_prod_batch.head()

,ee_customer_id,run_date,attrition_risk_segment,attrition_time_to_leave,oop_score_prod,oop_risk_segment_prod,run_date_ym,ee_onboarding_date,oop_target,osbal_as_of_resignation_date,osbal_as_of_oop_eligible_date,osbal_as_of_current_date,calc_date_ym,score_oop,onboarding_date_ym,onb_rd_diff
0,1124842,2026-03-28,Low,10,0.725723,A,202603,2023-11-28,<NA>,NaN,NaN,26123.873,NaN,NaN,202311,851
1,1154423,2026-03-28,Low,10,0.730344,A,202603,2024-02-28,<NA>,NaN,NaN,4753.862,NaN,NaN,202402,759
2,1204583,2026-03-28,Low,10,0.727268,A,202603,2024-05-28,<NA>,NaN,NaN,25853.294,NaN,NaN,202405,669
3,1204015,2026-03-28,Low,10,0.711415,A,202603,2024-05-28,<NA>,NaN,NaN,3783.955,NaN,NaN,202405,669
4,1203525,2026-03-28,Low,10,0.750166,A,202603,2024-05-28,<NA>,NaN,NaN,91560.252,NaN,NaN,202405,669


In [141]:
# import polars as pl

# df_prod = pl.from_pandas(dt_prod_batch)
# df_res = pl.from_pandas(dt_res)
# df_bs = pl.from_pandas(dt_bs_attr[['ee_customer_id', 'calc_date_ym', 
#                                     'score_attr', 'score_attr_segment', 
#                                     'is_new_customer_flag_1m']])

# result = (
#     df_prod
#     .join(df_res, on='ee_customer_id', how='left')
#     .join(df_bs, left_on=['ee_customer_id', 'run_date_ym'], 
#           right_on=['ee_customer_id', 'calc_date_ym'], how='left')
# )

In [142]:
dd.query("""select ee_customer_id, count(ee_customer_id) cnt from dt_res group by 1 having cnt > 1 order by cnt desc;""").to_df()

,ee_customer_id,cnt


In [143]:
dd.query("""select * from dt_res where ee_customer_id = '1065228';""").to_df()

,ee_customer_id,ee_resignation_date_correct
0,1065228,NaT


In [144]:
dd.query("""select ee_customer_id, count(ee_customer_id)cnt from dt_res group by 1 having cnt > 1 order by 2 desc;""").to_df()

,ee_customer_id,cnt


In [145]:
dt_prod_batch_attr = dt_prod_batch.merge(
    dt_res,
    how='left',
    on='ee_customer_id'
).merge(
    dt_bs_attr[['ee_customer_id', 'calc_date_ym', 'score_attr', 'score_attr_segment', 'is_new_customer_flag_1m']],
    how='left',
    left_on=['ee_customer_id','run_date_ym'],
    right_on=['ee_customer_id','calc_date_ym']
)

print(f"The shape of dt_prod_batch_attr after merging dt_prod_batch with dt_res and dt_bs_attr is:\t {dt_prod_batch_attr.shape}")

The shape of dt_prod_batch_attr after merging dt_prod_batch with dt_res and dt_bs_attr is:	 (404256, 21)


In [146]:
dt_prod_batch_attr.dropna(subset=['ee_onboarding_date']).query('run_date_ym == 202507')['ee_customer_id'].nunique()

35684

In [147]:
dt_prod_batch_calc = dt_prod_batch_attr.dropna(subset=['ee_onboarding_date'])
print(f"The shape of the dt_prod_batch_calc after dropping null values from ee_onboarding_date is:\t {dt_prod_batch_calc.shape}")

The shape of the dt_prod_batch_calc after dropping null values from ee_onboarding_date is:	 (404256, 21)


In [148]:
months_diff = (
    (dt_prod_batch_calc['ee_resignation_date_correct'].dt.year - dt_prod_batch_calc['run_date'].dt.year) * 12 +
    (dt_prod_batch_calc['ee_resignation_date_correct'].dt.month - dt_prod_batch_calc['run_date'].dt.month)
)

dt_prod_batch_calc['time_to_attrition'] = np.where(dt_prod_batch_calc['ee_resignation_date_correct'].isna(), np.nan, months_diff)
dt_prod_batch_calc['attrition_event'] = dt_prod_batch_calc['ee_resignation_date_correct'].notna().astype('int')

In [149]:
dt_prod_batch_calc['score_attr_corrected'] = dt_prod_batch_calc['score_attr'].replace(np.inf, 15)
dt_prod_batch_calc['attrition_score_prod'] = dt_prod_batch_calc['attrition_time_to_leave'].replace('12+', '15').astype('float')

mapping_dict = {
            1: 'Very high',
            2: 'Very high',
            3: 'Very high',
            4: 'High',
            5: 'High',
            6: 'High',
            7: 'Average',
            8: 'Average',
            9: 'Average',
            10: 'Low',
            11: 'Low',
            12: 'Low',
            15: 'Very low'
        }

dt_prod_batch_calc['attrition_risk_segment_prod'] = dt_prod_batch_calc['attrition_score_prod'].replace(mapping_dict)

In [150]:
# Existing prod users, prod score, June
df = dt_prod_batch_calc.query('run_date_ym == 202506').copy()

pt = pd.pivot_table(
    df,
    index="attrition_risk_segment_prod", # score_attr_segment - BS
    values=["attrition_event", "attrition_score_prod", "time_to_attrition", "ee_customer_id"],
    aggfunc={
        "attrition_event": "mean",
        "attrition_score_prod": "mean",
        "time_to_attrition": "mean",
        "ee_customer_id": "count"
    }
).rename(
    columns={"ee_customer_id": "count"}
)[['attrition_event', 'attrition_score_prod', 'time_to_attrition', 'count']]

order = ['Very low', 'Low', 'Average', 'High', 'Very high']

pt.loc[order]

,attrition_event,attrition_score_prod,time_to_attrition,count
attrition_risk_segment_prod,,,,
Very low,0.136236,15.000000,4.648012,4984
Low,0.128168,10.613582,4.399254,2091
Average,0.178924,8.030628,3.470986,5779
High,0.246425,5.356121,3.697066,6363
Very high,0.430085,2.891949,3.591133,472


In [151]:
# Slide 6 Upper part
# Existing prod users, prod score, July
df = dt_prod_batch_calc.query('run_date_ym == 202507').copy()

pt = pd.pivot_table(
    df,
    index="attrition_risk_segment_prod", # score_attr_segment - BS
    values=["attrition_event", "attrition_score_prod", "time_to_attrition", "ee_customer_id"],
    aggfunc={
        "attrition_event": "mean",
        "attrition_score_prod": "mean",
        "time_to_attrition": "mean",
        "ee_customer_id": "count"
    }
).rename(
    columns={"ee_customer_id": "count"}
)[['attrition_event', 'attrition_score_prod', 'time_to_attrition', 'count']]

order = ['Very low', 'Low', 'Average', 'High', 'Very high']

pt.loc[order]
attrextprodusersprodscorejuly25 = pt.loc[order].reset_index()
attrextprodusersprodscorejuly25 = attrextprodusersprodscorejuly25[['attrition_risk_segment_prod', 'count', 'attrition_event', 'attrition_score_prod', 'time_to_attrition']]
attrextprodusersprodscorejuly25


,attrition_risk_segment_prod,count,attrition_event,attrition_score_prod,time_to_attrition
0,Very low,8954,0.128769,15.000000,4.124892
1,Low,5054,0.129600,10.443213,3.760305
2,Average,13288,0.164208,8.105433,3.281852
3,High,8261,0.236896,5.500787,3.390393
4,Very high,127,0.385827,3.000000,3.306122


In [152]:
# Existing prod users, prod score, August
df = dt_prod_batch_calc.query('run_date_ym == 202508').copy()

pt = pd.pivot_table(
    df,
    index="attrition_risk_segment_prod", # score_attr_segment - BS
    values=["attrition_event", "attrition_score_prod", "time_to_attrition", "ee_customer_id"],
    aggfunc={
        "attrition_event": "mean",
        "attrition_score_prod": "mean",
        "time_to_attrition": "mean",
        "ee_customer_id": "count"
    }
).rename(
    columns={"ee_customer_id": "count"}
)[['attrition_event', 'attrition_score_prod', 'time_to_attrition', 'count']]

order = ['Very low', 'Low', 'Average', 'High', 'Very high']

pt.loc[order]
attrextprodusersprodscoreaug25 = pt.loc[order].reset_index()
attrextprodusersprodscoreaug25

,attrition_risk_segment_prod,attrition_event,attrition_score_prod,time_to_attrition,count
0,Very low,0.113253,15.000000,3.506424,8247
1,Low,0.109626,10.428627,3.127208,5163
2,Average,0.141134,8.115601,3.052438,13512
3,High,0.203967,5.511141,3.063025,8168
4,Very high,0.274725,3.000000,2.600000,91


In [153]:
# Existing prod users, BS score, June
df = dt_prod_batch_calc.query('run_date_ym == 202506').copy()

pt = pd.pivot_table(
    df,
    index="score_attr_segment", # score_attr_segment - BS
    values=["attrition_event", "score_attr_corrected", "time_to_attrition", "ee_customer_id"],
    aggfunc={
        "attrition_event": "mean",
        "score_attr_corrected": "mean",
        "time_to_attrition": "mean",
        "ee_customer_id": "count"
    }
).rename(
    columns={"ee_customer_id": "count"}
)[['attrition_event', 'score_attr_corrected', 'time_to_attrition', 'count']]

order = ['Very low', 'Low', 'Average', 'High', 'Very high']

pt.loc[order]
attrextproduserbscorejun25 = pt.loc[order].reset_index()
attrextproduserbscorejun25

,score_attr_segment,attrition_event,score_attr_corrected,time_to_attrition,count
0,Very low,0.129559,15.000000,4.988796,5511
1,Low,0.146822,10.454143,4.928767,2486
2,Average,0.172463,8.119974,4.298797,6268
3,High,0.247025,5.393927,3.992525,4874
4,Very high,0.421429,2.907143,4.398305,280


In [154]:
# Slide 6 lower part
# Existing prod users, BS score, July
df = dt_prod_batch_calc.query('run_date_ym == 202507').copy()

pt = pd.pivot_table(
    df,
    index="score_attr_segment", # score_attr_segment - BS
    values=["attrition_event", "score_attr_corrected", "time_to_attrition", "ee_customer_id"],
    aggfunc={
        "attrition_event": "mean",
        "score_attr_corrected": "mean",
        "time_to_attrition": "mean",
        "ee_customer_id": "count"
    }
).rename(
    columns={"ee_customer_id": "count"}
)[['attrition_event', 'score_attr_corrected', 'time_to_attrition', 'count']]

order = ['Very low', 'Low', 'Average', 'High', 'Very high']

attrextproduserbsscorejuly25 = pt.loc[order].reset_index()
attrextproduserbsscorejuly25 = attrextproduserbsscorejuly25[['score_attr_segment', 'count', 'attrition_event', 'score_attr_corrected', 'time_to_attrition']]
attrextproduserbsscorejuly25

,score_attr_segment,count,attrition_event,score_attr_corrected,time_to_attrition
0,Very low,8932,0.123936,15.000000,4.482385
1,Low,5056,0.119066,10.430775,4.455150
2,Average,13023,0.147508,8.126776,4.262363
3,High,7877,0.208201,5.516059,4.025000
4,Very high,103,0.320388,3.000000,3.666667


In [155]:
# Existing prod users, BS score, August
df = dt_prod_batch_calc.query('run_date_ym == 202508').copy()

pt = pd.pivot_table(
    df,
    index="score_attr_segment", # score_attr_segment - BS
    values=["attrition_event", "score_attr_corrected", "time_to_attrition", "ee_customer_id"],
    aggfunc={
        "attrition_event": "mean",
        "score_attr_corrected": "mean",
        "time_to_attrition": "mean",
        "ee_customer_id": "count"
    }
).rename(
    columns={"ee_customer_id": "count"}
)[['attrition_event', 'score_attr_corrected', 'time_to_attrition', 'count']]

order = ['Very low', 'Low', 'Average', 'High', 'Very high']

attrextprodusersbsscoreaug25 = pt.loc[order].reset_index()
attrextprodusersbsscoreaug25


,score_attr_segment,attrition_event,score_attr_corrected,time_to_attrition,count
0,Very low,0.105707,15.000000,3.897110,8183
1,Low,0.103109,10.427784,4.000000,5082
2,Average,0.130833,8.126182,3.879014,13330
3,High,0.180160,5.525737,3.565881,8004
4,Very high,0.271605,3.000000,2.954545,81


### bs existing

In [156]:
print(f"The shape of dt_bs_attr_dev_calc is:\t{dt_bs_attr_dev_calc.shape}")
dt_bs_attr_dev_calc.head()

The shape of dt_bs_attr_dev_calc is:	(156944, 15)


,ee_customer_id,calc_date,ee_onboarding_month,is_new_customer_flag_3m,target_event,target_duration_dev,score_attr,score_attr_segment,calc_date_correct,calc_date_ym,is_new_customer_flag_1m,ee_resignation_date_correct,time_to_attrition,attrition_event,score_attr_corrected
23,1119478,2024-01-01,202312,1,0.0,12.0,2.0,Very high,2023-12-01,202312,1,2025-02-13,14.0,1,2.0
25,1121063,2024-01-01,202312,1,0.0,12.0,2.0,Very high,2023-12-01,202312,1,NaT,NaN,0,2.0
26,1122569,2024-01-01,202312,1,1.0,6.0,2.0,Very high,2023-12-01,202312,1,2024-07-05,7.0,1,2.0
30,1126761,2024-01-01,202312,1,0.0,12.0,2.0,Very high,2023-12-01,202312,1,NaT,NaN,0,2.0
33,1127534,2024-01-01,202312,1,1.0,0.0,2.0,Very high,2023-12-01,202312,1,2024-01-06,1.0,1,2.0


In [157]:
dt_bs_attr_dev_calc = dt_bs_attr_dev.query('is_new_customer_flag_3m == 0').copy()

C:\Users\Dwaipayan\AppData\Local\Temp\ipykernel_21304\3206105625.py:1: RuntimeWarning: Engine has switched to 'python' because numexpr does not support extension array dtypes. Please set your engine to python manually.
  dt_bs_attr_dev_calc = dt_bs_attr_dev.query('is_new_customer_flag_3m == 0').copy()


In [158]:
months_diff = (
    (dt_bs_attr_dev_calc['ee_resignation_date_correct'].dt.year - dt_bs_attr_dev_calc['calc_date_correct'].dt.year) * 12 +
    (dt_bs_attr_dev_calc['ee_resignation_date_correct'].dt.month - dt_bs_attr_dev_calc['calc_date_correct'].dt.month)
)

dt_bs_attr_dev_calc['time_to_attrition'] = np.where(dt_bs_attr_dev_calc['ee_resignation_date_correct'].isna(), np.nan, months_diff)
dt_bs_attr_dev_calc['attrition_event'] = dt_bs_attr_dev_calc['ee_resignation_date_correct'].notna().astype('int')

In [159]:
dt_bs_attr_dev_calc['score_attr_corrected'] = dt_bs_attr_dev_calc['score_attr'].replace(np.inf, 15)

In [160]:
# Existing BS users, BS score, June
df = dt_bs_attr_dev_calc.query('calc_date_ym == 202506').copy()

pt = pd.pivot_table(
    df,
    index="score_attr_segment",
    values=["attrition_event", "score_attr_corrected", "time_to_attrition", "ee_customer_id"],
    aggfunc={
        "attrition_event": "mean",
        "score_attr_corrected": "mean",
        "time_to_attrition": "mean",
        "ee_customer_id": "count"
    }
).rename(
    columns={"ee_customer_id": "count"}
)[['attrition_event', 'score_attr_corrected', 'time_to_attrition', 'count']]

order = ['Very low', 'Low', 'Average', 'High', 'Very high']

attrextbsusersbsscorejun25 = pt.loc[order].reset_index()
attrextbsusersbsscorejun25

,score_attr_segment,attrition_event,score_attr_corrected,time_to_attrition,count
0,Very low,0.132850,15.000000,4.690421,19330
1,Low,0.151732,10.524227,4.661417,7533
2,Average,0.178092,8.126963,4.276794,17446
3,High,0.252867,5.431802,4.296237,12295
4,Very high,0.400763,2.925573,4.119048,524


In [161]:
# Existing BS users, BS score, July
df = dt_bs_attr_dev_calc.query('calc_date_ym == 202507').copy()

pt = pd.pivot_table(
    df,
    index="score_attr_segment",
    values=["attrition_event", "score_attr_corrected", "time_to_attrition", "ee_customer_id"],
    aggfunc={
        "attrition_event": "mean",
        "score_attr_corrected": "mean",
        "time_to_attrition": "mean",
        "ee_customer_id": "count"
    }
).rename(
    columns={"ee_customer_id": "count"}
)[['attrition_event', 'score_attr_corrected', 'time_to_attrition', 'count']]

order = ['Very low', 'Low', 'Average', 'High', 'Very high']

attrextbsusersbsscorejul25 = pt.loc[order].reset_index()
attrextbsusersbsscorejul25

,score_attr_segment,attrition_event,score_attr_corrected,time_to_attrition,count
0,Very low,0.116371,15.000000,4.381773,20933
1,Low,0.121278,10.486485,4.535757,13836
2,Average,0.168933,8.165043,4.225737,27508
3,High,0.229262,5.511287,4.086681,14442
4,Very high,0.369565,2.932609,3.652941,460


In [162]:
# Existing BS users, BS score, August
df = dt_bs_attr_dev_calc.query('calc_date_ym == 202508').copy()

pt = pd.pivot_table(
    df,
    index="score_attr_segment",
    values=["attrition_event", "score_attr_corrected", "time_to_attrition", "ee_customer_id"],
    aggfunc={
        "attrition_event": "mean",
        "score_attr_corrected": "mean",
        "time_to_attrition": "mean",
        "ee_customer_id": "count"
    }
).rename(
    columns={"ee_customer_id": "count"}
)[['attrition_event', 'score_attr_corrected', 'time_to_attrition', 'count']]

order = ['Very low', 'Low', 'Average', 'High', 'Very high']

attrextbsusersbsscoreaug25 = pt.loc[order].reset_index()
attrextbsusersbsscoreaug25

,score_attr_segment,attrition_event,score_attr_corrected,time_to_attrition,count
0,Very low,0.101009,15.000000,3.984665,21305
1,Low,0.107008,10.490364,4.010076,14840
2,Average,0.154484,8.163943,3.793163,29919
3,High,0.203840,5.520497,3.613103,15051
4,Very high,0.366133,2.876430,3.043750,437


In [206]:
def convert_datetime_columns(df):
    for col in df.columns:
        col_type = str(df[col].dtype).lower()
        
        # Handle date/time types
        if 'dbdate' in col_type or 'date' in col_type:
            try:
                df[col] = pd.to_datetime(df[col])
            except:
                pass
        elif 'dbtime' in col_type or 'time' in col_type:
            try:
                df[col] = pd.to_datetime(df[col], format='%H:%M:%S').dt.time
            except:
                pass
        elif 'dbtimestamp' in col_type or 'timestamp' in col_type:
            try:
                df[col] = pd.to_datetime(df[col])
            except:
                pass

# Count of Onboarded customer as per scorecard 2.0 - New logic Three months from Onboarding

In [207]:
sq = """ 
select 
date(user_timelines.first_account_activated_at) as ee_onboarding_date,
format_date('%Y-%m', date(user_timelines.first_account_activated_at)) Onboarding_month,
t1.employee_id,
t1.oop_risk_segment,
  from prj-prod-dataplatform.tendo_mart.tendo_scorecard_master_table_api t1
  inner join tendopay_raw.user_timelines on cast(user_timelines.user_id as string) = cast(t1.employee_id as string)
   where user_timelines.first_account_activated_at >= '2025-06-01'
        and user_timelines.first_account_activated_at < '2026-01-01'
     qualify row_number() over(partition by t1.employee_id order by t1.run_date) = 1;
"""
df1 = client.query(sq).to_dataframe()

In [208]:
df1.groupby(['Onboarding_month', 'oop_risk_segment']).agg({'employee_id':'nunique'}).reset_index()

,Onboarding_month,oop_risk_segment,employee_id
0,2025-06,A,28
1,2025-06,B,28
2,2025-06,C,34
3,2025-06,D,6
4,2025-06,E,5
5,2025-06,F,7
6,2025-07,A,489
7,2025-07,B,289
8,2025-07,C,141
9,2025-07,D,39


In [209]:

sq = """ 
select user_id, tag_id, tag_name, created_at resignation_date, deleted_at 
from prj-prod-dataplatform.tendopay_raw.user_tag
where tag_id = 100
and deleted_at is null;
"""
userdf = client.query(sq).to_dataframe()


In [210]:
sq = """select user_id, target_maturity_flag, target , ee_resignation_date_correct 
from prj-prod-dataplatform.tendo_mart.tendo_collection_target_master"""
df_target = client.query(sq).to_dataframe()
df_target.shape

(45948, 4)

In [211]:
for col in df1.columns:
    col_type = str(df1[col].dtype).lower()
    
    # Handle date/time types
    if 'dbdate' in col_type or 'date' in col_type:
        try:
            df1[col] = pd.to_datetime(df1[col])
        except:
            pass
    elif 'dbtime' in col_type or 'time' in col_type:
        try:
            df1[col] = pd.to_datetime(df1[col], format='%H:%M:%S').dt.time
        except:
            pass
    elif 'dbtimestamp' in col_type or 'timestamp' in col_type:
        try:
            df1[col] = pd.to_datetime(df1[col])
        except:
            pass

df2 = dd.query("""select * from df1 
         left join userdf on userdf.user_id = df1.employee_id
            left join df_target on cast(df_target.user_id as string) = cast (df1.employee_id as string)
                   """).to_df()       

In [212]:
df2.head()

,ee_onboarding_date,Onboarding_month,employee_id,oop_risk_segment,user_id,tag_id,tag_name,resignation_date,deleted_at,user_id_1,target_maturity_flag,target,ee_resignation_date_correct
0,2025-06-19,2025-06,1406933,A,1406933,100,FREEZE_PERMANENT,2025-12-30 03:38:09+05:30,NaT,1406933,1,1,2025-12-29 05:30:00+05:30
1,2025-07-03,2025-07,1470987,A,1470987,100,FREEZE_PERMANENT,2025-12-30 03:38:09+05:30,NaT,1470987,1,1,2025-12-29 05:30:00+05:30
2,2025-07-04,2025-07,1398169,A,1398169,100,FREEZE_PERMANENT,2025-12-30 03:38:09+05:30,NaT,1398169,0,0,2025-12-29 05:30:00+05:30
3,2025-07-08,2025-07,1471810,A,1471810,100,FREEZE_PERMANENT,2025-12-30 03:38:09+05:30,NaT,1471810,1,0,2025-12-29 05:30:00+05:30
4,2025-07-03,2025-07,1471189,B,1471189,100,FREEZE_PERMANENT,2025-12-30 03:38:09+05:30,NaT,1471189,0,0,2025-12-29 05:30:00+05:30


In [213]:
dd.query("""select * from df2 where date(resignation_date) != date(ee_resignation_date_correct);""").to_df()

,ee_onboarding_date,Onboarding_month,employee_id,oop_risk_segment,user_id,tag_id,tag_name,resignation_date,deleted_at,user_id_1,target_maturity_flag,target,ee_resignation_date_correct
0,2025-06-19,2025-06,1406933,A,1406933,100,FREEZE_PERMANENT,2025-12-30 03:38:09+05:30,NaT,1406933,1,1,2025-12-29 05:30:00+05:30
1,2025-07-03,2025-07,1470987,A,1470987,100,FREEZE_PERMANENT,2025-12-30 03:38:09+05:30,NaT,1470987,1,1,2025-12-29 05:30:00+05:30
2,2025-07-04,2025-07,1398169,A,1398169,100,FREEZE_PERMANENT,2025-12-30 03:38:09+05:30,NaT,1398169,0,0,2025-12-29 05:30:00+05:30
3,2025-07-08,2025-07,1471810,A,1471810,100,FREEZE_PERMANENT,2025-12-30 03:38:09+05:30,NaT,1471810,1,0,2025-12-29 05:30:00+05:30
4,2025-07-03,2025-07,1471189,B,1471189,100,FREEZE_PERMANENT,2025-12-30 03:38:09+05:30,NaT,1471189,0,0,2025-12-29 05:30:00+05:30
...,...,...,...,...,...,...,...,...,...,...,...,...,...
530,2025-09-01,2025-09,1530597,A,1530597,100,FREEZE_PERMANENT,2026-02-19 19:05:49+05:30,NaT,1530597,1,1,2026-01-31 05:30:00+05:30
531,2025-09-22,2025-09,1521546,A,1521546,100,FREEZE_PERMANENT,2025-11-05 18:16:11+05:30,NaT,1521546,1,0,2025-10-30 05:30:00+05:30
532,2025-07-03,2025-07,1472948,B,1472948,100,FREEZE_PERMANENT,2026-02-23 17:22:56+05:30,NaT,1472948,1,0,2025-12-29 05:30:00+05:30
533,2025-10-14,2025-10,1578038,B,1578038,100,FREEZE_PERMANENT,2026-01-07 16:28:11+05:30,NaT,1578038,1,0,2025-12-15 05:30:00+05:30


In [214]:
data = dd.query("""select Onboarding_month, oop_risk_segment,
         count(distinct employee_id) as cnt_onboarded_with_sc2,
         count(distinct case when ee_resignation_date_correct is not null
         and date(ee_resignation_date_correct) < date('2026-02-01') then ee_resignation_date_correct end) as cnt_left_job_within_Jan_2026,
         count(distinct case when target_maturity_flag = 1 then df2.employee_id end) as cnt_had_oop_outstanding,
         count(distinct case when target_maturity_flag = 1 and target = 0 then df2.employee_id end) as cnt_oop_flag_bad_fpd30,
         round(count(distinct case when target_maturity_flag = 1 and target = 0 then df2.employee_id end)/
         count(distinct case when ee_resignation_date_correct is not null
         and date(ee_resignation_date_correct) < date('2026-02-01') then ee_resignation_date_correct end) *100, 2) as unit_rate_of_all_who_left_job
         from df2
         group by Onboarding_month, oop_risk_segment
         order by Onboarding_month
         """).to_df()

In [215]:
data

,Onboarding_month,oop_risk_segment,cnt_onboarded_with_sc2,cnt_left_job_within_Jan_2026,cnt_had_oop_outstanding,cnt_oop_flag_bad_fpd30,unit_rate_of_all_who_left_job
0,2025-06,F,7,0,0,0,NaN
1,2025-06,D,6,1,1,1,100.00
2,2025-06,E,5,1,1,1,100.00
3,2025-06,A,28,4,5,2,50.00
4,2025-06,C,34,3,3,1,33.33
5,2025-06,B,28,5,5,3,60.00
6,2025-07,E,37,6,6,4,66.67
7,2025-07,B,289,26,29,18,69.23
8,2025-07,D,39,5,4,4,80.00
9,2025-07,F,140,9,7,4,44.44


In [216]:
data.to_excel(r"D:\OneDrive - Tonik Financial Pte Ltd\MyStuff\Data Engineering\Tendo_Model_Monitoring_Dashboards_Requirement\Tendo_Monitoring_data\20260330_New_Tendo_Funnel_sc2_0_monthlytilldec25.xlsx", sheet_name='New_Tendo_funnel_scorecard2_0')

In [226]:
data2 = dd.query("""select Onboarding_month,
         count(distinct employee_id) as cnt_onboarded_with_sc2,
         count(distinct case when ee_resignation_date_correct is not null
         and date(ee_resignation_date_correct) < date('2026-02-01') then ee_resignation_date_correct end) as cnt_left_job_within_Jan_2026,
         count(distinct case when target_maturity_flag = 1 then df2.employee_id end) as cnt_had_oop_outstanding,
         count(distinct case when target_maturity_flag = 1 and target = 0 then df2.employee_id end) as cnt_oop_flag_bad_fpd30,
         round(count(distinct case when target_maturity_flag = 1 and target = 0 then df2.employee_id end)/
         count(distinct case when ee_resignation_date_correct is not null
         and date(ee_resignation_date_correct) < date('2026-02-01') then df2.employee_id end) , 2) as unit_rate_of_all_who_left_job
         from df2
         group by Onboarding_month
         order by 1
         """).to_df()

In [227]:
data2

,Onboarding_month,cnt_onboarded_with_sc2,cnt_left_job_within_Jan_2026,cnt_had_oop_outstanding,cnt_oop_flag_bad_fpd30,unit_rate_of_all_who_left_job
0,2025-06,108,14,15,8,0.50
1,2025-07,1135,71,117,78,0.51
2,2025-08,863,50,85,60,0.64
3,2025-09,921,59,106,77,0.60
4,2025-10,575,34,42,26,0.53
5,2025-11,951,32,63,43,0.65
6,2025-12,1079,13,25,12,0.55


In [228]:
data2.to_excel(r"D:\OneDrive - Tonik Financial Pte Ltd\MyStuff\Data Engineering\Tendo_Model_Monitoring_Dashboards_Requirement\Tendo_Monitoring_data\20260330_New_Tendo_Funnel_scorecard2_0_monthlydec25.xlsx", sheet_name='New_Tendo_funnel_sc_2_0_rs')

# Peso values of customers Onboarded

## Select the employees who onboarded from June to August 2025

In [229]:
sq = """ 
select distinct
date(user_timelines.first_account_activated_at) as ee_onboarding_date,
format_date('%Y-%m', date(user_timelines.first_account_activated_at)) Onboarding_month,
t1.employee_id,
t1.oop_risk_segment,
  from prj-prod-dataplatform.tendo_mart.tendo_scorecard_master_table_api t1
  inner join tendopay_raw.user_timelines on cast(user_timelines.user_id as string) = cast(t1.employee_id as string)
   where user_timelines.first_account_activated_at >= '2025-06-01'
     and user_timelines.first_account_activated_at < '2026-01-01'
  qualify row_number() over(partition by t1.employee_id order by t1.run_date) = 1;
"""
df1 = client.query(sq).to_dataframe()
print(f"The data for the customers who onboarded between June to August 2025 is: {df1.shape}")

The data for the customers who onboarded between June to August 2025 is: (5632, 4)


In [230]:
df1.groupby('Onboarding_month').agg({'employee_id':'nunique'}).reset_index()

,Onboarding_month,employee_id
0,2025-06,108
1,2025-07,1135
2,2025-08,863
3,2025-09,921
4,2025-10,575
5,2025-11,951
6,2025-12,1079


In [231]:
def convert_datetime_columns(df):
    for col in df.columns:
        col_type = str(df[col].dtype).lower()
        
        # Handle date/time types
        if 'dbdate' in col_type or 'date' in col_type:
            try:
                df[col] = pd.to_datetime(df[col])
            except:
                pass
        elif 'dbtime' in col_type or 'time' in col_type:
            try:
                df[col] = pd.to_datetime(df[col], format='%H:%M:%S').dt.time
            except:
                pass
        elif 'dbtimestamp' in col_type or 'timestamp' in col_type:
            try:
                df[col] = pd.to_datetime(df[col])
            except:
                pass

In [232]:
convert_datetime_columns(df1)

In [233]:
dd.query("""select employee_id, count(employee_id) as emp_count from df1 group by employee_id having emp_count > 1 order by 2 desc;""").to_df()

,employee_id,emp_count


In [234]:
sq = """ 
with frozen_tags AS (
  SELECT
    user_id,
    max(CASE WHEN tag_id IN (39, 100, 101, 102, 103) THEN 1 ELSE 0 END)
      AS frozen_tag
  FROM `tendopay_raw.user_tag`
  GROUP BY user_id
),
employee_info as
(SELECT
    u.id user_id,
    u.created_at sign_up_date,
    ut.first_account_activated_at AS ee_onboarding_date,
    ut.first_account_activated_at approval_date,
    u.employer_id employer_id,
    e.name employer_name,
    ii.employment_date employment_date,
    LENGTH(employment_date) LENGTH_employment_date,
    datetime_diff(
      CAST(ii.created_at AS date),
      CASE  ------original: datetime_diff(cast(ut.first_account_activated_at as date),
        WHEN LENGTH(employment_date) = 6
          THEN
            date(
              CAST(substr(employment_date, 1, 4) AS int64),
              CAST(substr(employment_date, 6, 1) AS int64),
              1)
        WHEN LENGTH(employment_date) = 7
          THEN
            date(
              CAST(substr(employment_date, 1, 4) AS int64),
              CAST(substr(employment_date, 6, 2) AS int64),
              1)
        WHEN LENGTH(employment_date) = 10
          THEN
            date(
              CAST(substr(employment_date, 1, 4) AS int64),
              CAST(substr(employment_date, 6, 2) AS int64),
              CAST(substr(employment_date, 9, 2) AS int64))
        END,
      day)
      / 365
      AS employer_time,  ----sometimes employment_date is missing, sometimes approval_date (first_account_activated_at) is missing
    u.gender gender,
    u.civil_status civil_status,
    ii.employee_status employment_status,
    CASE
      WHEN ii.income IN ('0-10000') THEN 5000
      WHEN ii.income IN ('10000-20000') THEN 15000
      WHEN ii.income IN ('20000-30000') THEN 25000
      WHEN ii.income IN ('30000-40000') THEN 35000
      WHEN ii.income IN ('40000-50000') THEN 45000
      WHEN ii.income IN ('50000+') THEN 50000
      ELSE CAST(ii.income AS numeric)
      END AS declared_income_num,
    ii.verified_net_income verified_net_income,
    CASE
      WHEN tag.frozen_tag = 1 THEN 'Frozen'
      ELSE 'Not Frozen'
      END Frozen_Status,

      cci.value                                                                               Credit_limit,
       
  FROM `tendopay_raw.users` u
  LEFT JOIN `tendopay_raw.income_info` ii
    ON u.id = ii.user_id
  LEFT JOIN `tendopay_raw.user_timelines` ut
    ON u.id = ut.user_id
  LEFT JOIN `tendopay_raw.employers` e
    ON u.employer_id = CAST(e.id AS string)
  LEFT JOIN frozen_tags tag
    ON u.id = tag.user_id

  LEFT JOIN `tendopay_raw.customer_credit_information` cci on u.id = cci.user_id
  WHERE u.product_type IN ('employer', 'payroll')

  and u.account_activated = 2
  and cci.`key` = 'credit-limit'
)
select user_id, 
sign_up_date,
ee_onboarding_date,
employer_id,
employment_date,
LENGTH_employment_date,
gender,
civil_status,
employment_status,
declared_income_num,
verified_net_income,
Frozen_Status,
Credit_limit,
 from employee_info
;
"""

df_employee = client.query(sq).to_dataframe(progress_bar_type='tqdm')
print(f"The data for the employee info is: {df_employee.shape}")


Job ID 0b2ed420-3ad1-4f07-b8d1-299708db81b6 successfully executed: 100%|██████████|
Downloading: 100%|██████████|
The data for the employee info is: (196746, 13)


In [235]:
df_employee.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 196746 entries, 0 to 196745
Data columns (total 13 columns):
 #   Column                  Non-Null Count   Dtype              
---  ------                  --------------   -----              
 0   user_id                 196746 non-null  Int64              
 1   sign_up_date            196746 non-null  datetime64[us, UTC]
 2   ee_onboarding_date      196744 non-null  datetime64[us, UTC]
 3   employer_id             196675 non-null  object             
 4   employment_date         196725 non-null  object             
 5   LENGTH_employment_date  196725 non-null  Int64              
 6   gender                  196711 non-null  object             
 7   civil_status            195494 non-null  object             
 8   employment_status       185661 non-null  object             
 9   declared_income_num     196729 non-null  object             
 10  verified_net_income     196590 non-null  object             
 11  Frozen_Status           19

In [236]:
df1.columns

Index(['ee_onboarding_date', 'Onboarding_month', 'employee_id',
       'oop_risk_segment'],
      dtype='object')

In [237]:
df1['user_id'] = df1['employee_id'].astype('int64')

## Merging Onboarded employee from June 2025 to August 2025 with employee info data

In [238]:
data = df1.merge(df_employee, left_on='user_id', right_on='user_id', how='left')
print(f"The shape of the data after merging employee info is: {data.shape}")

The shape of the data after merging employee info is: (5632, 17)


In [239]:
data.head()

,ee_onboarding_date_x,Onboarding_month,employee_id,oop_risk_segment,user_id,sign_up_date,ee_onboarding_date_y,employer_id,employment_date,LENGTH_employment_date,gender,civil_status,employment_status,declared_income_num,verified_net_income,Frozen_Status,Credit_limit
0,2025-10-04,2025-10,1342803,F,1342803,2025-02-13 16:14:33+00:00,2025-10-04 17:11:46+00:00,10200,2023-08,7,Male,Married,regular,24000.000000000,18500,Not Frozen,12333
1,2025-11-29,2025-11,1421170,E,1421170,2025-05-02 08:18:44+00:00,2025-11-29 12:47:58+00:00,10200,2024-04,7,Female,Single,regular,24000.000000000,19500,Not Frozen,17333.00
2,2025-09-10,2025-09,1523098,E,1523098,2025-08-21 02:28:24+00:00,2025-09-10 15:36:33+00:00,10200,2024-07,7,Male,Married,regular,30000.000000000,17500,Frozen,18244.00
3,2025-11-30,2025-11,1601310,C,1601310,2025-10-27 00:09:15+00:00,2025-11-30 11:16:38+00:00,10200,2024-11,7,Female,Single,regular,36500.000000000,31500,Not Frozen,31500.00
4,2025-12-07,2025-12,1650112,D,1650112,2025-12-06 10:18:13+00:00,2025-12-07 18:44:44+00:00,10065,2024-01,7,Male,Single,regular,30000.000000000,27000,Not Frozen,39405.00


In [240]:
def convert_datetime_columns(df):
    for col in df.columns:
        col_type = str(df[col].dtype).lower()
        
        # Handle date/time types
        if 'dbdate' in col_type or 'date' in col_type:
            try:
                df[col] = pd.to_datetime(df[col])
            except:
                pass
        elif 'dbtime' in col_type or 'time' in col_type:
            try:
                df[col] = pd.to_datetime(df[col], format='%H:%M:%S').dt.time
            except:
                pass
        elif 'dbtimestamp' in col_type or 'timestamp' in col_type:
            try:
                df[col] = pd.to_datetime(df[col])
            except:
                pass

Check duplicates

In [241]:
convert_datetime_columns(data)
dd.query("""select employee_id, count(employee_id) as emp_count from data group by employee_id having emp_count > 1 order by 2 desc;""").to_df()

,employee_id,emp_count


## OOP maturity target merged with Oleh's backscore table

In [242]:
sq = """ 
with b1 as 
(select user_id, target_maturity_flag, target , ee_resignation_date_correct 
from prj-prod-dataplatform.tendo_mart.tendo_collection_target_master
where target_maturity_flag = 1
)
select * from b1 
left join `prj-prod-dataplatform.risk_mart.tendo_backscored_new_users_jan23_feb26_20260301_oop_with_osbal` b2 on cast(b1.user_id as numeric) = cast(b2.ee_customer_id as numeric);
"""
df_target = client.query(sq).to_dataframe()
print(f"The data for the target maturity info is: {df_target.shape}")

The data for the target maturity info is: (36066, 11)


In [243]:
df_target['user_id'] = df_target['user_id'].astype('int64')

Check for duplicates

In [244]:
dd.query("""select user_id, count(user_id) cnt from df_target group by 1 having count(user_id) > 1 order by 2 desc;""").to_df()

,user_id,cnt


In [245]:
data1 = data.merge(df_target, left_on='user_id', right_on='user_id', how='inner')

In [246]:
data1.head()

,ee_onboarding_date_x,Onboarding_month,employee_id,oop_risk_segment,user_id,sign_up_date,ee_onboarding_date_y,employer_id,employment_date,LENGTH_employment_date,gender,civil_status,employment_status,declared_income_num,verified_net_income,Frozen_Status,Credit_limit,target_maturity_flag,target,ee_resignation_date_correct,ee_customer_id,calc_date,target_dev,score_oop,osbal_as_of_resignation_date,osbal_as_of_oop_eligible_date,osbal_as_of_current_date
0,2025-11-10,2025-11,1501448,C,1501448,2025-07-30 09:04:09+00:00,2025-11-10 15:09:00+00:00,10239,2024-07,7,Female,Single,regular,17000.000000000,16000,Frozen,12173.00,1,0,2025-12-25 00:00:00+00:00,1501448,2025-12-01,NaN,0.234570,4878.771,4878.771,4878.771
1,2025-11-30,2025-11,1446508,B,1446508,2025-05-31 13:57:44+00:00,2025-11-30 15:14:35+00:00,10270,2025-04,7,Female,Single,probationary,30000.000000000,23500,Frozen,19964,1,0,2026-01-27 00:00:00+00:00,1446508,2025-12-01,NaN,0.534165,15329.092,15329.092,15329.092
2,2025-09-13,2025-09,1548964,A,1548964,2025-09-13 05:36:18+00:00,2025-09-13 12:57:26+00:00,10224,2025-07,7,Male,Single,probationary,20000.000000000,20000,Frozen,8530.00,1,0,2025-10-16 00:00:00+00:00,1548964,2025-10-01,0.0,0.555388,2250.000,2250.000,2250.000
3,2025-09-23,2025-09,1559918,A,1559918,2025-09-23 13:11:41+00:00,2025-09-23 15:32:03+00:00,10283,2025-08,7,Female,Single,regular,27000.000000000,8500,Frozen,5971,1,0,2025-10-30 00:00:00+00:00,1559918,2025-10-01,0.0,0.550790,5022.037,5022.037,3669.733
4,2025-08-29,2025-08,1467809,B,1467809,2025-06-27 15:35:45+00:00,2025-08-29 10:54:52+00:00,10239,2025-05,7,Female,Single,regular,18500.000000000,17000,Frozen,13660.00,1,0,2025-12-25 00:00:00+00:00,1467809,2025-09-01,0.0,0.289562,2496.939,2496.939,2496.939


In [247]:
dd.query("""select employee_id, count(employee_id) cnt from data1 group by 1 having count(employee_id) > 1 order by 2 desc;""").to_df()

,employee_id,cnt


In [248]:
data3 = dd.query(f""" select Onboarding_month, oop_risk_segment,
         -- count(distinct employee_id) as cnt_onboarded_with_sc2,
         sum(cast(Credit_limit as numeric)) Credit_limit,
         -- count(distinct case when ee_resignation_date_correct is not null then employee_id end) as cnt_resigned_employees,
         -- count(distinct case when target_maturity_flag = 1 then employee_id end) as cnt_had_oop_eligible_cnt,
         -- count(distinct case when target = 0 then employee_id end) as cnt_oop_flag_bad_fpd30,
         -- count(distinct case when target = 1 then employee_id end) as cnt_oop_flag_good_fpd30,
         sum(cast(osbal_as_of_resignation_date as numeric)) as sum_oop_outstanding_resignation_date,
         sum(case when target_maturity_flag = 1 then cast(osbal_as_of_oop_eligible_date as numeric) else 0 end) as osbal_as_of_oop_eligible_date,
         sum(case when target = 0 then cast(coalesce(osbal_as_of_current_date, 0) as numeric) end) as tot_os_principal_amt_from_bad_customers,
         from data1
            group by Onboarding_month, oop_risk_segment
                 order by Onboarding_month, oop_risk_segment
         """
         ).to_df()

In [249]:
data3.head()

,Onboarding_month,oop_risk_segment,Credit_limit,sum_oop_outstanding_resignation_date,osbal_as_of_oop_eligible_date,tot_os_principal_amt_from_bad_customers
0,2025-06,A,118522.0,49193.030,21415.847,6299.818
1,2025-06,B,148429.0,114035.052,109709.934,50587.124
2,2025-06,C,237490.0,78286.440,78286.440,26157.168
3,2025-06,D,12716.0,10659.616,10659.616,10659.616
4,2025-06,E,31282.0,18166.701,18166.701,18166.701


In [250]:
data3.to_excel(r"D:\OneDrive - Tonik Financial Pte Ltd\MyStuff\Data Engineering\Tendo_Model_Monitoring_Dashboards_Requirement\Tendo_Monitoring_data\20260330_New_Tendo_Funnel_scorecard2_0_peso_monthly_dec25.xlsx", sheet_name='New_Tendo_funnel_sc2_0_peso')

In [251]:
data2 = dd.query(f"""select Onboarding_month,
         sum(cast(Credit_limit as numeric)) tot_cl_amt_onboarded_with_2_0,
         --- count(distinct case when ee_resignation_date_correct is not null then employee_id end) as cnt_resigned_employees,
         --- count(distinct case when target_maturity_flag = 1 then employee_id end) as cnt_had_oop_eligible_cnt,
         --- count(distinct case when target = 0 then employee_id end) as cnt_oop_flag_bad_fpd30,
         --- count(distinct case when target = 1 then employee_id end) as cnt_oop_flag_good_fpd30,
         sum(case when ee_resignation_date_correct is not null and date(ee_resignation_date_correct) < '2026-02-01' then cast(osbal_as_of_resignation_date as numeric) else 0 end) as tot_os_principal_amt_as_of_resignation_date,
         sum(case when ee_resignation_date_correct is not null and date(ee_resignation_date_correct) < '2026-02-01' and target_maturity_flag = 1 then cast(osbal_as_of_oop_eligible_date as numeric) else 0 end) as tot_os_principal_amt_as_of_oop_eligible_dt,
         sum(case when ee_resignation_date_correct is not null and date(ee_resignation_date_correct) < '2026-02-01' and target_maturity_flag = 1 and target = 0 then cast(coalesce(osbal_as_of_current_date, 0) as numeric) end) as tot_os_principal_amt_from_bad_customers,
         from data1
                 group by 1
                 order by 1
                   """
         ).to_df()

In [252]:
data2.sort_values(by='Onboarding_month')

,Onboarding_month,tot_cl_amt_onboarded_with_2_0,tot_os_principal_amt_as_of_resignation_date,tot_os_principal_amt_as_of_oop_eligible_dt,tot_os_principal_amt_from_bad_customers
0,2025-06,548439.0,233810.413,201708.112,111870.427
1,2025-07,3174333.5,1128698.139,1032891.722,764730.910
2,2025-08,2178571.0,767525.397,831600.720,743934.593
3,2025-09,2382031.0,942562.473,950550.112,679417.815
4,2025-10,1404889.0,428865.799,369538.541,300085.755
5,2025-11,1658608.0,406939.795,441083.473,406528.202
6,2025-12,690136.0,128042.459,132374.078,102179.732


In [253]:
data2.to_excel(r"D:\OneDrive - Tonik Financial Pte Ltd\MyStuff\Data Engineering\Tendo_Model_Monitoring_Dashboards_Requirement\Tendo_Monitoring_data\20260330_New_Tendo_Funnel_scorecard2_0_peso_dec25.xlsx", sheet_name='New_Tendo_funnel_sc2_0_peso')

# For v1

In [260]:
sq = """
WITH first_success AS (
  SELECT
    tendopay_user_id AS user_id,
    MIN(DATE(created_at)) AS onboarding_date
  FROM tendopay_raw.payment_responses
  WHERE status IN ('PTOK','CTOK')
  GROUP BY tendopay_user_id
),

-- ScoreCard 1.0 user level (risk segment)
scorecard_1_0 AS (
  WITH data_preparation AS (
    SELECT
      u.id AS user_id,
      ii.employment_date,
      DATETIME_DIFF(
        CAST(ii.created_at AS DATE),
        CASE
          WHEN LENGTH(ii.employment_date)=6  THEN SAFE.PARSE_DATE('%Y-%m',    ii.employment_date)
          WHEN LENGTH(ii.employment_date)=7  THEN SAFE.PARSE_DATE('%Y-%m',    ii.employment_date)
          WHEN LENGTH(ii.employment_date)=10 THEN SAFE.PARSE_DATE('%Y-%m-%d', ii.employment_date)
        END,
        DAY
      ) / 365 AS employer_time,
      u.gender,
      u.civil_status,
      CASE
        WHEN ii.income IN ('0-10000')     THEN 5000
        WHEN ii.income IN ('10000-20000') THEN 15000
        WHEN ii.income IN ('20000-30000') THEN 25000
        WHEN ii.income IN ('30000-40000') THEN 35000
        WHEN ii.income IN ('40000-50000') THEN 45000
        WHEN ii.income IN ('50000+')      THEN 50000
        ELSE CAST(ii.income AS NUMERIC)
      END AS declared_income_num
    FROM prj-prod-dataplatform.tendopay_raw.users u
    LEFT JOIN prj-prod-dataplatform.tendopay_raw.income_info ii
      ON u.id = ii.user_id
      left join `tendopay_raw.user_timelines` ut
      on u.id = ut.user_id
    WHERE u.product_type IN ('employer') 
    and ut.first_account_activated_at is not null

  ),

  scoring AS (
    SELECT
      *,
      (
        CASE
          WHEN employer_time < 0.55 THEN 98
          WHEN employer_time < 2.35 THEN 123
          WHEN employer_time < 3.85 THEN 139
          WHEN employer_time >= 3.85 THEN 183
          ELSE 119
        END
        + CASE
            WHEN gender = 'Female' THEN 120
            WHEN gender IN ('Male','not specified') THEN 118
            ELSE 119
          END
        + CASE
            WHEN declared_income_num < 21200 THEN 119
            WHEN declared_income_num < 26525 THEN 110
            WHEN declared_income_num < 33050 THEN 118
            WHEN declared_income_num >= 33050 THEN 146
            ELSE 119
          END
        + CASE
            WHEN civil_status IN ('Divorced','Married','Widowed') THEN 127
            WHEN civil_status = 'Single' THEN 117
            ELSE 119
          END
      ) AS score
    FROM data_preparation
  )

  SELECT DISTINCT
    user_id,
    CASE
      WHEN score > 532 THEN 'A'
      WHEN score > 492 THEN 'B'
      WHEN score > 478 THEN 'C'
      WHEN score > 468 THEN 'D'
      WHEN score > 452 THEN 'E'
      ELSE 'F'
    END AS risk_segment
  FROM scoring
),

-- onboarding users with SC 1.0
master_cohort AS (
  SELECT
    f.user_id,
    f.onboarding_date,
    sc.risk_segment
  FROM first_success f
  INNER JOIN scorecard_1_0 sc
    ON f.user_id = sc.user_id
  WHERE f.onboarding_date BETWEEN DATE '2025-06-01' AND DATE '2025-12-31'
),

-- resignation logic
ordered_repayments AS (
  SELECT
    user_id,
    vendor_id,
    repaid_at,
    LEAD(vendor_id) OVER (PARTITION BY user_id ORDER BY repaid_at) AS next_vendor
  FROM tendopay_raw.customer_repayment_responses
  WHERE vendor_id IN ('TPSD','TPFP')
    AND repaid_at IS NOT NULL
),

resignation AS (
  SELECT
    user_id,
    MAX(DATE_ADD(DATE(repaid_at), INTERVAL 1 MONTH)) AS resignation_date
  FROM ordered_repayments
  WHERE vendor_id='TPSD'
    AND next_vendor='TPFP'
  GROUP BY user_id
),


oop_data AS (
  SELECT
    ee_customer_id AS user_id,
    osbal_as_of_resignation_date
  FROM `prj-prod-dataplatform.risk_mart.tendo_backscored_new_users_jan23_feb26_20260301_oop_with_osbal`
),

oop_target AS (
  SELECT
    user_id,
    `target`, target_maturity_flag,ee_resignation_date_correct
  FROM `prj-prod-dataplatform.tendo_mart.tendo_collection_target_master`
  WHERE target_maturity_flag = 1

)

-- final
SELECT
  FORMAT_DATE('%Y-%m', mc.onboarding_date) AS onboarding_month,
  --mc.risk_segment,

  COUNT(DISTINCT mc.user_id) AS cnt_onboarded_with_1_0,

  COUNT(DISTINCT CASE WHEN date(tgt.ee_resignation_date_correct) <= DATE '2026-01-31' THEN mc.user_id END) AS cnt_left_job_within_jan_2026,

  COUNT(DISTINCT CASE WHEN   tgt.target_maturity_flag = 1 THEN mc.user_id END) AS cnt_had_oop_outstanding,


  COUNT(DISTINCT CASE WHEN  tgt.target = 0 THEN mc.user_id END) AS cnt_oop_flag_bad,

  round(SAFE_DIVIDE(
    COUNT(DISTINCT CASE
    WHEN  tgt.target = 0
    THEN mc.user_id
  END),
    COUNT(DISTINCT CASE
    WHEN
   target_maturity_flag = 1
    THEN mc.user_id 
  END)
  ) ,2) AS unit_rate_of_all_who_left_job

FROM master_cohort mc
LEFT JOIN resignation r
  ON mc.user_id = r.user_id
LEFT JOIN oop_data oop
  ON mc.user_id = oop.user_id
LEFT JOIN oop_target tgt
  ON cast(mc.user_id as string) = tgt.user_id

GROUP BY  onboarding_month--,  mc.risk_segment
ORDER BY --mc.risk_segment,
onboarding_month;
"""

df2v1 = client.query(sq).to_dataframe()
df2v1
 

,onboarding_month,cnt_onboarded_with_1_0,cnt_left_job_within_jan_2026,cnt_had_oop_outstanding,cnt_oop_flag_bad,unit_rate_of_all_who_left_job
0,2025-06,5302,505,533,350,0.66
1,2025-07,4567,414,450,296,0.66
2,2025-08,4039,290,322,202,0.63
3,2025-09,8610,569,627,436,0.70
4,2025-10,12346,864,963,716,0.74
5,2025-11,7892,443,495,360,0.73
6,2025-12,8592,120,178,91,0.51


In [262]:
df2v1.to_excel(r"D:\OneDrive - Tonik Financial Pte Ltd\MyStuff\Data Engineering\Tendo_Model_Monitoring_Dashboards_Requirement\Tendo_Monitoring_data\20260330_v1pesorateunit.xlsx", sheet_name='v1unitpesorate')

In [256]:
sq = """
   WITH first_success AS (
    SELECT
        tendopay_user_id AS user_id,
        MIN(DATE(created_at)) AS onboarding_date
    FROM tendopay_raw.payment_responses
    WHERE status IN ('PTOK','CTOK')
    GROUP BY tendopay_user_id
),

onboarded_cohort AS (
    SELECT *
    FROM first_success
    WHERE onboarding_date BETWEEN DATE '2025-06-01'
                              AND DATE '2025-12-31'
),

-- Build Score + Risk Segment
scorecard_1_0 AS (

    WITH data_preparation AS (
        SELECT
            u.id AS user_id,
            DATETIME_DIFF(
                CAST(ii.created_at AS DATE),
                SAFE.PARSE_DATE('%Y-%m-%d', ii.employment_date),
                DAY
            ) / 365 AS employer_time,
            u.gender,
            u.civil_status,
            CASE
                WHEN ii.income IN ('0-10000') THEN 5000
                WHEN ii.income IN ('10000-20000') THEN 15000
                WHEN ii.income IN ('20000-30000') THEN 25000
                WHEN ii.income IN ('30000-40000') THEN 35000
                WHEN ii.income IN ('40000-50000') THEN 45000
                WHEN ii.income IN ('50000+') THEN 50000
                ELSE CAST(ii.income AS NUMERIC)
            END AS declared_income_num
        FROM prj-prod-dataplatform.tendopay_raw.users u
        LEFT JOIN prj-prod-dataplatform.tendopay_raw.income_info ii
            ON u.id = ii.user_id
            left join `tendopay_raw.user_timelines` ut
      on u.id = ut.user_id
        WHERE u.product_type IN ('employer')
        and ut.first_account_activated_at is not null
    ),

    scoring AS (
        SELECT
            user_id,
            (
                CASE
                    WHEN employer_time < 0.55 THEN 98
                    WHEN employer_time < 2.35 THEN 123
                    WHEN employer_time < 3.85 THEN 139
                    WHEN employer_time >= 3.85 THEN 183
                    ELSE 119
                END
              + CASE
                    WHEN gender='Female' THEN 120
                    ELSE 118
                END
              + CASE
                    WHEN declared_income_num < 21200 THEN 119
                    WHEN declared_income_num < 26525 THEN 110
                    WHEN declared_income_num < 33050 THEN 118
                    WHEN declared_income_num >= 33050 THEN 146
                    ELSE 119
                END
              + CASE
                    WHEN civil_status IN ('Divorced','Married','Widowed') THEN 127
                    WHEN civil_status='Single' THEN 117
                    ELSE 119
                END
            ) AS score
        FROM data_preparation
    )

    SELECT
        user_id,
        CASE
            WHEN score > 532 THEN 'A'
            WHEN score > 492 THEN 'B'
            WHEN score > 478 THEN 'C'
            WHEN score > 468 THEN 'D'
            WHEN score > 452 THEN 'E'
            ELSE 'F'
        END AS risk_segment
    FROM scoring
),

scored_onboarded AS (
    SELECT
        o.user_id,
        o.onboarding_date,
        sc.risk_segment
    FROM onboarded_cohort o
    JOIN scorecard_1_0 sc
        ON o.user_id = sc.user_id
),

-- -- resignation
-- ordered_repayments AS (
--     SELECT
--         user_id,
--         vendor_id,
--         repaid_at,
--         LEAD(vendor_id) OVER(
--             PARTITION BY user_id
--             ORDER BY repaid_at
--         ) AS next_vendor
--     FROM tendopay_raw.customer_repayment_responses
--     WHERE vendor_id IN ('TPSD','TPFP')
-- ),

-- resigned_users AS (
--     SELECT
--         user_id,
--         MAX(DATE_ADD(DATE(repaid_at), INTERVAL 1 MONTH)) AS resignation_date
--     FROM ordered_repayments
--     WHERE vendor_id='TPSD'
--       AND next_vendor='TPFP'
--     GROUP BY user_id
-- ),

credit_limit AS (
    SELECT
        user_id,
        SAFE_CAST(value AS FLOAT64) AS credit_limit
    FROM prj-prod-dataplatform.tendopay_raw.customer_credit_information
    WHERE key='credit-limit'
),

oop_data AS (
    SELECT
        ee_customer_id AS user_id,
        osbal_as_of_resignation_date,
        osbal_as_of_oop_eligible_date,
        target_dev
    FROM `prj-prod-dataplatform.risk_mart.tendo_backscored_new_users_jan23_feb26_20260301_oop_with_osbal`
),
oop_target AS (
  SELECT
    user_id,
    `target`, target_maturity_flag,ee_resignation_date_correct
  FROM `prj-prod-dataplatform.tendo_mart.tendo_collection_target_master`
  WHERE target_maturity_flag = 1

)

SELECT
format_date('%Y-%m',s.onboarding_date) as onboarding_month,

--s.risk_segment,


SUM(cl.credit_limit) AS tot_cl_amt_onboarded_with_1_0,



SUM( case when date(tgt.ee_resignation_date_correct) <= DATE '2026-01-31'
    THEN oop.osbal_as_of_resignation_date end)
    AS tot_os_principal_amt_as_of_resignation_date,


SUM(CASE
    WHEN   target_maturity_flag = 1
    THEN oop.osbal_as_of_oop_eligible_date end)
    AS tot_os_principal_amt_as_of_oop_eligible_dt,


SUM(
   CASE
    WHEN  tgt.target = 0
    THEN oop.osbal_as_of_oop_eligible_date
    END
) AS tot_os_principal_amt_from_bad_customers,


safe_divide(
SUM(
   CASE
    WHEN  tgt.target = 0
    THEN oop.osbal_as_of_oop_eligible_date
    END
),
SUM( case when date(tgt.ee_resignation_date_correct) <= DATE '2026-01-31'
    THEN oop.osbal_as_of_resignation_date end)

) *100 as unit_rate_of_all_who_left_job


FROM scored_onboarded s
JOIN oop_data r
    ON s.user_id = r.user_id  
join   oop_target tgt
on  cast(r.user_id as string) = tgt.user_id

LEFT JOIN credit_limit cl
    ON s.user_id = cl.user_id
LEFT JOIN oop_data oop
    ON s.user_id = oop.user_id

GROUP BY -- s.risk_segment, 
onboarding_month
ORDER BY -- s.risk_segment,
onboarding_month;
"""
dfv1 = client.query(sq).to_dataframe()
dfv1


,onboarding_month,tot_cl_amt_onboarded_with_1_0,tot_os_principal_amt_as_of_resignation_date,tot_os_principal_amt_as_of_oop_eligible_dt,tot_os_principal_amt_from_bad_customers,unit_rate_of_all_who_left_job
0,2025-06,18099479.5,6.474458e+06,5.921355e+06,4.070457e+06,62.869464
1,2025-07,15548592.0,4.972061e+06,5.054511e+06,3.199943e+06,64.358479
2,2025-08,11426397.0,3.531153e+06,3.552277e+06,2.572754e+06,72.858747
3,2025-09,17698526.0,8.486249e+06,8.936645e+06,6.730110e+06,79.306062
4,2025-10,24908159.5,1.386236e+07,1.515295e+07,1.179684e+07,85.099786
5,2025-11,13363094.5,7.588289e+06,8.508873e+06,6.555536e+06,86.390170
6,2025-12,5370659.5,1.718960e+06,3.017319e+06,1.739411e+06,101.189776


In [257]:
dfv1.to_excel(r"D:\OneDrive - Tonik Financial Pte Ltd\MyStuff\Data Engineering\Tendo_Model_Monitoring_Dashboards_Requirement\Tendo_Monitoring_data\20260330_v1pesorate.xlsx", sheet_name='v1pesorate')